In [3]:
#check ver
import requests
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from lxml import etree

print("Cài đặt thành công")
print("Requests:", requests.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Cài đặt thành công
Requests: 2.32.5
NumPy: 2.2.6
Pandas: 2.3.3


In [ ]:
URL = "https://vinpearl.com/en"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,vi;q=0.8",
}

In [5]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait

options = Options()
options.add_argument("--start-maximized")
options.add_argument(f"user-agent={HEADERS['User-Agent']}")

driver = webdriver.Edge(options=options)
driver.get(URL)

# Chờ tối đa 120 giây đến khi hết trang "Just a moment..."
WebDriverWait(driver, 120).until(
    lambda d: (
        "just a moment" not in d.title.lower()
        and "attention required" not in d.title.lower()
        and len(d.page_source) > 5000
    )
)

html = driver.page_source

print("Tiêu đề:", driver.title)
print("URL:", driver.current_url)
print("Độ dài HTML:", len(html))

Tiêu đề: Vinpearl | Official website
URL: https://vinpearl.com/en
Độ dài HTML: 156790


In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "lxml")
items = soup.select("div.info-wrapper")
data_info_topic = {}
for item in items:
    title_tag = item.select_one("div.exp-item-tit")
    link_tag = item.select_one("div.exp-item-cta a[href]")

    if title_tag and link_tag:
        name = title_tag.get_text(" ", strip=True)
        link = link_tag.get("href")

        data_info_topic[name] = link

In [7]:
data_info_topic

{'Vinpearl Hotels & Resorts': 'https://vinpearl.com/en/hotels',
 'VinWonders Theme Parks': 'https://vinwonders.com/en/',
 'Weddings & Events': 'https://vinpearl.com/vi/meeting-events',
 'Vinpearl Golf': 'https://vinpearl.com/en/vinpearl-golf'}

In [12]:
# Lấy topic đầu tiên trong dictionary
topic_name_1, link1 = list(data_info_topic.items())[0]

print("Tên topic 1:", topic_name_1)
print("Link 1:", link1)

# Truy cập vào link 1 bằng trình duyệt Selenium hiện tại
driver.get(link1)

# Chờ trang tải xong và có đủ HTML
WebDriverWait(driver, 120).until(
    lambda d: (
        "just a moment" not in d.title.lower()
        and "attention required" not in d.title.lower()
        and d.execute_script("return document.readyState") == "complete"
        and len(d.page_source) > 5000
    )
)

# Lấy toàn bộ HTML của link 1
html_link1 = driver.page_source

print("Tiêu đề trang:", driver.title)
print("URL thực tế:", driver.current_url)
print("Độ dài HTML:", len(html_link1))

Tên topic 1: Vinpearl Hotels & Resorts
Link 1: https://vinpearl.com/en/hotels
Tiêu đề trang: Vinpearl | The holiday paradise of Vietnam
URL thực tế: https://vinpearl.com/en/hotels
Độ dài HTML: 258073


In [13]:
from urllib.parse import urljoin

base_url = "https://vinpearl.com"
location_dict = {}

soup_link1 = BeautifulSoup(html_link1, "lxml")

for a_tag in soup_link1.select(
    '#_destination > div > ul > li > a[href]'
):
    name = a_tag.get_text(" ", strip=True)
    href = a_tag.get("href")

    if name and href:
        location_dict[name] = urljoin(base_url, href)

for name, url in location_dict.items():
    print("Tên:", name)
    print("URL:", url)
    print("-" * 50)

Tên: Phu Quoc
URL: https://vinpearl.com/en/hotels-phu-quoc
--------------------------------------------------
Tên: Nha Trang
URL: https://vinpearl.com/en/hotels-nha-trang
--------------------------------------------------
Tên: Nam Hoi An
URL: https://vinpearl.com/en/hotels-nam-hoi-an
--------------------------------------------------
Tên: Ha Tinh
URL: https://vinpearl.com/en/hotels-ha-tinh
--------------------------------------------------
Tên: Nghe An
URL: https://vinpearl.com/en/hotels-nghe-an
--------------------------------------------------
Tên: Bac Ninh
URL: https://vinpearl.com/en/hotels-bac-ninh
--------------------------------------------------
Tên: Ha Long
URL: https://vinpearl.com/en/hotels-ha-long
--------------------------------------------------


In [14]:
location_html_dict = {}

for location_name, location_url in location_dict.items():
    location_driver = None

    try:
        print(f"Đang truy cập: {location_name}")
        print(f"URL: {location_url}")

        location_driver = webdriver.Edge(options=options)
        location_driver.get(location_url)

        WebDriverWait(location_driver, 120).until(
            lambda d: (
                d.execute_script("return document.readyState") == "complete"
                and "just a moment" not in d.title.lower()
                and "attention required" not in d.title.lower()
            )
        )

        html_location = location_driver.page_source

        location_html_dict[location_name] = {
            "name": location_name,
            "url": location_driver.current_url,
            "html": html_location
        }

        print(f"Đã lấy HTML: {location_name}")
        print(f"URL thực tế: {location_driver.current_url}")
        print(f"Độ dài HTML: {len(html_location)}")
        print("-" * 50)

    except Exception as error:
        print(f"Lỗi khi lấy HTML: {location_name}")
        print(f"URL lỗi: {location_url}")
        print(f"Loại lỗi: {type(error).__name__}")
        print(f"Nội dung lỗi: {error}")
        print("-" * 50)

    finally:
        if location_driver is not None:
            location_driver.quit()

Đang truy cập: Phu Quoc
URL: https://vinpearl.com/en/hotels-phu-quoc
Đã lấy HTML: Phu Quoc
URL thực tế: https://vinpearl.com/en/hotels-phu-quoc
Độ dài HTML: 209572
--------------------------------------------------
Đang truy cập: Nha Trang
URL: https://vinpearl.com/en/hotels-nha-trang
Đã lấy HTML: Nha Trang
URL thực tế: https://vinpearl.com/en/hotels-nha-trang
Độ dài HTML: 253813
--------------------------------------------------
Đang truy cập: Nam Hoi An
URL: https://vinpearl.com/en/hotels-nam-hoi-an
Đã lấy HTML: Nam Hoi An
URL thực tế: https://vinpearl.com/en/hotels-nam-hoi-an
Độ dài HTML: 185180
--------------------------------------------------
Đang truy cập: Ha Tinh
URL: https://vinpearl.com/en/hotels-ha-tinh
Đã lấy HTML: Ha Tinh
URL thực tế: https://vinpearl.com/en/hotels-ha-tinh
Độ dài HTML: 196279
--------------------------------------------------
Đang truy cập: Nghe An
URL: https://vinpearl.com/en/hotels-nghe-an
Đã lấy HTML: Nghe An
URL thực tế: https://vinpearl.com/en/hotels-

In [15]:
hotel_data_by_location = {}

for location_name, location_data in location_html_dict.items():
    location_soup = BeautifulSoup(location_data["html"], "lxml")

    hotel_list = []

    for hotel_link_tag in location_soup.select(
        'a[href]:has(h3)'
    ):
        hotel_name_tag = hotel_link_tag.select_one("h3")
        hotel_address_tag = hotel_link_tag.parent.select_one(
            "p:nth-of-type(1) span"
        )

        hotel_name = (
            hotel_name_tag.get_text(" ", strip=True)
            if hotel_name_tag
            else None
        )

        hotel_address = (
            hotel_address_tag.get_text(" ", strip=True)
            if hotel_address_tag
            else None
        )

        hotel_url = urljoin(
            location_data["url"],
            hotel_link_tag.get("href")
        )

        if hotel_name and hotel_url:
            hotel_list.append({
                "hotel_name": hotel_name,
                "hotel_address": hotel_address,
                "hotel_url": hotel_url
            })

    hotel_data_by_location[location_name] = {
        "location_name": location_name,
        "location_url": location_data["url"],
        "hotels": hotel_list
    }

In [16]:
#check data
for location_name, location_data in hotel_data_by_location.items():
    print(f"ĐỊA ĐIỂM: {location_name}")
    print(f"URL địa điểm: {location_data['location_url']}")
    print(f"Số khách sạn: {len(location_data['hotels'])}")
    print("=" * 60)

    for hotel in location_data["hotels"]:
        print("Tên:", hotel["hotel_name"])
        print("Địa chỉ:", hotel["hotel_address"])
        print("Link:", hotel["hotel_url"])
        print("-" * 50)

ĐỊA ĐIỂM: Phu Quoc
URL địa điểm: https://vinpearl.com/en/hotels-phu-quoc
Số khách sạn: 3
Tên: Vinpearl Resort & Spa Phu Quoc
Địa chỉ: Bai Dai Area, Ganh Dau Commune, Phu Quoc City, Kien Giang Province, Vietnam
Link: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc
--------------------------------------------------
Tên: Vinpearl Wonderworld Phu Quoc
Địa chỉ: Bai Dai Area, Ganh Dau Commune, Phu Quoc City, Kien Giang Province, Vietnam
Link: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc
--------------------------------------------------
Tên: VinHolidays Fiesta Phu Quoc
Địa chỉ: Bai Dai Tourist Area, Ganh Dau Ward, Phu Quoc Special Zone, An Giang Province, Vietnam
Link: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc
--------------------------------------------------
ĐỊA ĐIỂM: Nha Trang
URL địa điểm: https://vinpearl.com/en/hotels-nha-trang
Số khách sạn: 6
Tên: Vinpearl Beachfront Nha Trang
Địa chỉ: 78 – 80 Tran Phu Street, Nha Trang Ward, Khanh Hoa Prov

In [17]:
hotel_room_link_dict = {}

for location_name, location_data in hotel_data_by_location.items():
    hotel_room_link_dict[location_name] = []

    for hotel in location_data["hotels"]:
        hotel_name = hotel["hotel_name"]
        hotel_url = hotel["hotel_url"]

        hotel_driver = None

        try:
            print(f"Đang truy cập khách sạn: {hotel_name}")
            print(f"URL: {hotel_url}")

            hotel_driver = webdriver.Edge(options=options)
            hotel_driver.get(hotel_url)

            WebDriverWait(hotel_driver, 120).until(
                lambda d: (
                    d.execute_script("return document.readyState") == "complete"
                    and "just a moment" not in d.title.lower()
                    and "attention required" not in d.title.lower()
                )
            )

            hotel_detail_html = hotel_driver.page_source
            hotel_detail_soup = BeautifulSoup(
                hotel_detail_html,
                "lxml"
            )

            room_link_tag = hotel_detail_soup.select_one(
                '#detail-room > div:nth-of-type(1) > div > div > a[href]'
            )

            if room_link_tag is None:
                print(f"Không tìm thấy link phòng: {hotel_name}")
                print("-" * 50)
                continue

            room_page_url = urljoin(
                hotel_driver.current_url,
                room_link_tag["href"]
            )

            hotel_room_link_dict[location_name].append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_url": hotel_driver.current_url,
                "room_page_url": room_page_url
            })

            print(f"Đã lấy link phòng: {hotel_name}")
            print(f"Link phòng: {room_page_url}")
            print("-" * 50)

        except Exception as error:
            print(f"Lỗi khi lấy link phòng: {hotel_name}")
            print(f"Loại lỗi: {type(error).__name__}")
            print(f"Nội dung lỗi: {error}")
            print("-" * 50)

        finally:
            if hotel_driver is not None:
                hotel_driver.quit()

Đang truy cập khách sạn: Vinpearl Resort & Spa Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc
Đã lấy link phòng: Vinpearl Resort & Spa Phu Quoc
Link phòng: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/rooms
--------------------------------------------------
Đang truy cập khách sạn: Vinpearl Wonderworld Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc
Đã lấy link phòng: Vinpearl Wonderworld Phu Quoc
Link phòng: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/rooms
--------------------------------------------------
Đang truy cập khách sạn: VinHolidays Fiesta Phu Quoc
URL: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc
Đã lấy link phòng: VinHolidays Fiesta Phu Quoc
Link phòng: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/rooms
--------------------------------------------------
Đang truy cập khách sạn: Vinpearl Beachfront Nha Trang
URL: https://vinpearl.com/en/hotels/vinpearl-beachfro

In [18]:
for location_name, hotels in hotel_room_link_dict.items():
    print(f"ĐỊA ĐIỂM: {location_name}")
    print("=" * 60)

    for hotel in hotels:
        print("Tên khách sạn:", hotel["hotel_name"])
        print("Link khách sạn:", hotel["hotel_url"])
        print("Link trang phòng:", hotel["room_page_url"])
        print("-" * 50)

ĐỊA ĐIỂM: Phu Quoc
Tên khách sạn: Vinpearl Resort & Spa Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc
Link trang phòng: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/rooms
--------------------------------------------------
Tên khách sạn: Vinpearl Wonderworld Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc
Link trang phòng: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/rooms
--------------------------------------------------
Tên khách sạn: VinHolidays Fiesta Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc
Link trang phòng: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/rooms
--------------------------------------------------
ĐỊA ĐIỂM: Nha Trang
Tên khách sạn: Vinpearl Beachfront Nha Trang
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-beachfront-nha-trang
Link trang phòng: https://vinpearl.com/en/hotels/vinpearl-beachfront-nha

In [19]:
hotel_room_html_dict = {}

for location_name, hotels in hotel_room_link_dict.items():
    hotel_room_html_dict[location_name] = []

    for hotel in hotels:
        hotel_name = hotel["hotel_name"]
        room_page_url = hotel["room_page_url"]

        room_driver = None

        try:
            print(f"Đang truy cập trang phòng: {hotel_name}")
            print(f"URL: {room_page_url}")

            room_driver = webdriver.Edge(options=options)
            room_driver.get(room_page_url)

            WebDriverWait(room_driver, 120).until(
                lambda d: (
                    d.execute_script("return document.readyState") == "complete"
                    and "just a moment" not in d.title.lower()
                    and "attention required" not in d.title.lower()
                )
            )

            room_page_html = room_driver.page_source

            hotel_room_html_dict[location_name].append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_url": hotel["hotel_url"],
                "room_page_url": room_driver.current_url,
                "room_page_html": room_page_html
            })

            print(f"Đã lấy HTML trang phòng: {hotel_name}")
            print(f"URL thực tế: {room_driver.current_url}")
            print(f"Độ dài HTML: {len(room_page_html)}")
            print("-" * 50)

        except Exception as error:
            print(f"Lỗi khi lấy HTML trang phòng: {hotel_name}")
            print(f"URL lỗi: {room_page_url}")
            print(f"Loại lỗi: {type(error).__name__}")
            print(f"Nội dung lỗi: {error}")
            print("-" * 50)

        finally:
            if room_driver is not None:
                room_driver.quit()

Đang truy cập trang phòng: Vinpearl Resort & Spa Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/rooms
Đã lấy HTML trang phòng: Vinpearl Resort & Spa Phu Quoc
URL thực tế: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/rooms
Độ dài HTML: 2176913
--------------------------------------------------
Đang truy cập trang phòng: Vinpearl Wonderworld Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/rooms
Đã lấy HTML trang phòng: Vinpearl Wonderworld Phu Quoc
URL thực tế: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/rooms
Độ dài HTML: 3008050
--------------------------------------------------
Đang truy cập trang phòng: VinHolidays Fiesta Phu Quoc
URL: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/rooms
Đã lấy HTML trang phòng: VinHolidays Fiesta Phu Quoc
URL thực tế: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/rooms
Độ dài HTML: 2151793
--------------------------------------------------


In [33]:
import re
import pandas as pd
from bs4 import BeautifulSoup


# ============================================================
# 1. CÁC HÀM HỖ TRỢ
# ============================================================

def clean_text(tag):
    """
    Lấy nội dung text từ thẻ HTML và chuẩn hóa khoảng trắng.
    """
    if tag is None:
        return None

    text = tag.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def normalize_key(value):
    """
    Chuẩn hóa chuỗi để đối chiếu tên địa điểm và khách sạn.
    """
    if value is None:
        return ""

    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)

    return value


def get_image_url(img_tag):
    """
    Lấy URL ảnh phòng.

    Thứ tự ưu tiên:
    1. data-src
    2. data-lazy-src
    3. data-original
    4. src
    """
    if img_tag is None:
        return None

    image_url = (
        img_tag.get("data-src")
        or img_tag.get("data-lazy-src")
        or img_tag.get("data-original")
        or img_tag.get("src")
    )

    if image_url:
        image_url = image_url.strip()

    return image_url or None


def find_room_container(title_tag):
    """
    Tìm thẻ cha bao trùm toàn bộ thông tin của đúng một phòng.

    Container cần chứa:
    - tên phòng;
    - thông tin phòng;
    - ảnh hoặc phần giá hoặc tiện ích.
    """
    for parent in title_tag.parents:
        if parent.name not in ["div", "li"]:
            continue

        has_title = parent.select_one(".room-content-title")
        has_room_info = parent.select_one(".room-content-wrap")

        has_room_image = parent.select_one(
            'img[data-src*="/room_types/"], '
            'img[src*="/room_types/"], '
            'img.w-100[data-src], '
            'img.w-100[src], '
            'img[data-fancybox][data-src], '
            'img[data-fancybox][src]'
        )

        has_price = parent.select_one(
            ".room-content-panel-text"
        )

        has_contact = parent.select_one(
            'a.btn[href^="tel:"], '
            'a.primary-button[href^="tel:"]'
        )

        has_amenities = parent.select_one(
            ".information-footer-wrap"
        )

        if (
            has_title
            and has_room_info
            and (
                has_room_image
                or has_price
                or has_contact
                or has_amenities
            )
        ):
            return parent

    # Dự phòng nếu cấu trúc một số khách sạn khác biệt
    for parent in title_tag.parents:
        if parent.name not in ["div", "li"]:
            continue

        if (
            parent.select_one(".room-content-title")
            and parent.select_one(".room-content-wrap")
        ):
            return parent

    return None


def extract_room_image_url(room_container, title_tag=None):
    """
    Tìm URL ảnh của đúng phòng.

    Ưu tiên ảnh có đường dẫn /room_types/
    và ưu tiên giá trị data-src.
    """
    image_selectors = [
        'img[data-src*="/room_types/"]',
        'img[src*="/room_types/"]',
        'img.w-100[data-src]',
        'img.w-100[src]',
        'img[data-fancybox][data-src]',
        'img[data-fancybox][src]'
    ]

    if room_container is not None:
        for selector in image_selectors:
            image_tag = room_container.select_one(selector)
            image_url = get_image_url(image_tag)

            if image_url:
                return image_url

    if title_tag is not None:
        for parent in title_tag.parents:
            if parent.name not in ["div", "li"]:
                continue

            for selector in image_selectors:
                image_tag = parent.select_one(selector)
                image_url = get_image_url(image_tag)

                if image_url:
                    return image_url

    return None


# ============================================================
# 2. HÀM LẤY GIÁ PHÒNG
# ============================================================

def extract_room_prices(room_container):
    """
    Lấy hai trường giá của đúng từng phòng:

    - standard_rate
    - price_only_from

    Trường hợp có giá:
        Standard rate ~ 346USD/night
        Price only from ~ 329USD/night

    Trường hợp không có giá và phải liên hệ:
        href="tel:1900232389"

    Khi phải liên hệ, cả hai cột đều nhận giá trị href.
    """

    standard_rate = None
    price_only_from = None
    contact_href = None

    # ========================================================
    # A. TÌM PANEL GIÁ
    # ========================================================
    price_panel = room_container.select_one(
        ".room-content-panel-text"
    )

    if price_panel is not None:

        # ----------------------------------------------------
        # Standard rate
        # Ưu tiên span có class riêng
        # ----------------------------------------------------
        standard_price_tag = price_panel.select_one(
            ".room-content-panel-text__price"
        )

        if standard_price_tag is not None:
            standard_rate = clean_text(
                standard_price_tag
            )

        # ----------------------------------------------------
        # Price only from
        #
        # Thường nằm trong div thứ hai:
        # <div>
        #   Price only from
        #   <span>~ 329USD</span>
        #   <span class="...per-night">/night</span>
        # </div>
        # ----------------------------------------------------
        direct_divs = price_panel.find_all(
            "div",
            recursive=False
        )

        for div_tag in direct_divs:
            div_text = clean_text(div_tag) or ""
            div_text_lower = div_text.lower()

            # Standard rate dự phòng
            if (
                standard_rate is None
                and "standard rate" in div_text_lower
            ):
                price_tags = div_tag.find_all("span")

                for span_tag in price_tags:
                    span_classes = span_tag.get(
                        "class",
                        []
                    )

                    if (
                        "room-content-panel-text__per-night"
                        not in span_classes
                    ):
                        span_text = clean_text(span_tag)

                        if span_text:
                            standard_rate = span_text
                            break

            # Price only from
            if "price only from" in div_text_lower:
                price_tags = div_tag.find_all("span")

                for span_tag in price_tags:
                    span_classes = span_tag.get(
                        "class",
                        []
                    )

                    # Bỏ qua span chỉ chứa /night
                    if (
                        "room-content-panel-text__per-night"
                        in span_classes
                    ):
                        continue

                    span_text = clean_text(span_tag)

                    if (
                        span_text
                        and span_text.lower() != "/night"
                    ):
                        price_only_from = span_text
                        break

        # ----------------------------------------------------
        # Dự phòng theo thứ tự div
        # ----------------------------------------------------
        if direct_divs:
            if standard_rate is None:
                first_div_spans = direct_divs[0].find_all(
                    "span"
                )

                for span_tag in first_div_spans:
                    span_text = clean_text(span_tag)

                    if (
                        span_text
                        and span_text.lower() != "/night"
                    ):
                        standard_rate = span_text
                        break

            if (
                price_only_from is None
                and len(direct_divs) >= 2
            ):
                second_div_spans = direct_divs[1].find_all(
                    "span"
                )

                for span_tag in second_div_spans:
                    span_classes = span_tag.get(
                        "class",
                        []
                    )

                    if (
                        "room-content-panel-text__per-night"
                        in span_classes
                    ):
                        continue

                    span_text = clean_text(span_tag)

                    if (
                        span_text
                        and span_text.lower() != "/night"
                    ):
                        price_only_from = span_text
                        break

    # ========================================================
    # B. NẾU KHÔNG CÓ GIÁ, TÌM NÚT HOTEL CONTACT
    # ========================================================
    if standard_rate is None and price_only_from is None:

        contact_link = room_container.select_one(
            'a.btn[href^="tel:"], '
            'a.primary-button[href^="tel:"], '
            'a[href^="tel:"]'
        )

        if contact_link is not None:
            contact_href = contact_link.get("href")

            if contact_href:
                contact_href = contact_href.strip()

                standard_rate = contact_href
                price_only_from = contact_href

    return {
        "standard_rate": standard_rate,
        "price_only_from": price_only_from,
        "contact_href": contact_href
    }


# ============================================================
# 3. HÀM LẤY THÔNG TIN CƠ BẢN CỦA PHÒNG
# ============================================================

def extract_room_basic_info(room_container):
    """
    Tách thành các cột riêng:

    - guest_count
    - room_area
    - bed_types
    - wifi
    """
    guest_count = None
    room_area = None
    bed_types = []
    wifi = False

    info_items = room_container.select(
        ".room-content-wrap .room-content-wrap-icon"
    )

    for item in info_items:
        img = item.select_one("img")

        icon_src = ""
        icon_alt = ""

        if img:
            icon_src = (
                img.get("src")
                or img.get("data-src")
                or ""
            ).strip().lower()

            icon_alt = (
                img.get("alt")
                or ""
            ).strip().lower()

        item_text = item.get_text(
            " ",
            strip=True
        )

        item_text = re.sub(
            r"\s+",
            " ",
            item_text
        ).strip()

        item_text_lower = item_text.lower()

        # ====================================================
        # SỐ NGƯỜI
        # ====================================================
        if (
            "teaser_hotel_people" in icon_src
            or "hotel_people" in icon_src
            or "people.svg" in icon_src
        ):
            guest_match = re.search(
                r"\d+",
                item_text
            )

            if guest_match:
                guest_count = int(
                    guest_match.group()
                )
            else:
                guest_count = item_text or None

            continue

        # ====================================================
        # DIỆN TÍCH
        # ====================================================
        if (
            "teaser_hotel_dientich" in icon_src
            or "hotel_dientich" in icon_src
            or "dientich.svg" in icon_src
            or "area" in icon_src
        ):
            area_match = re.search(
                r"([\d.,]+)\s*m\s*(?:2|²)",
                item_text,
                flags=re.IGNORECASE
            )

            if area_match:
                room_area = (
                    f"{area_match.group(1)} m²"
                )
            else:
                number_match = re.search(
                    r"[\d.,]+",
                    item_text
                )

                if number_match:
                    room_area = (
                        f"{number_match.group()} m²"
                    )
                else:
                    room_area = item_text or None

            continue

        # ====================================================
        # WIFI
        # ====================================================
        if (
            "wifi" in icon_src
            or "wi-fi" in icon_src
            or "internet" in icon_src
            or "wifi" in icon_alt
            or "internet" in icon_alt
            or "wifi" in item_text_lower
            or "wi-fi" in item_text_lower
            or "internet" in item_text_lower
        ):
            wifi = True
            continue

        # ====================================================
        # LOẠI GIƯỜNG
        # ====================================================
        bed_keywords = [
            "king",
            "twin",
            "queen",
            "double",
            "single",
            "bunk",
            "sofa",
            "bed"
        ]

        is_bed_info = any(
            keyword in icon_src
            or keyword in icon_alt
            or keyword in item_text_lower
            for keyword in bed_keywords
        )

        if is_bed_info:
            bed_name = item_text

            if not bed_name and img:
                bed_name = img.get("alt")

            if bed_name:
                bed_name = re.sub(
                    r"\s+",
                    " ",
                    bed_name
                ).strip()

            if (
                bed_name
                and bed_name not in bed_types
            ):
                bed_types.append(bed_name)

    return {
        "guest_count": guest_count,
        "room_area": room_area,
        "bed_types": (
            ", ".join(bed_types)
            if bed_types
            else None
        ),
        "wifi": wifi
    }


def extract_amenities(room_container):
    """
    Lấy danh sách tiện ích và trả về dạng list.
    """
    amenities = []

    amenity_items = room_container.select(
        ".information-footer-wrap "
        ".information-footer-wrap-icon.attr"
    )

    for item in amenity_items:
        amenity_name = clean_text(item)

        if (
            amenity_name
            and amenity_name not in amenities
        ):
            amenities.append(amenity_name)

    return amenities


# ============================================================
# 4. TẠO BẢNG TRA ĐỊA CHỈ KHÁCH SẠN
# ============================================================

hotel_address_lookup = {}


for location_name, location_data in (
    hotel_data_by_location.items()
):

    if isinstance(location_data, dict):
        hotel_items = location_data.get(
            "hotels",
            []
        )

    elif isinstance(location_data, list):
        hotel_items = location_data

    else:
        hotel_items = []

    for hotel_item in hotel_items:
        hotel_item_name = hotel_item.get(
            "hotel_name"
        )

        if not hotel_item_name:
            continue

        hotel_address = (
            hotel_item.get("hotel_address")
            or hotel_item.get("address")
        )

        location_key = normalize_key(
            location_name
        )

        hotel_key = normalize_key(
            hotel_item_name
        )

        hotel_address_lookup[
            (location_key, hotel_key)
        ] = hotel_address


# Bảng tra dự phòng theo tên khách sạn
hotel_address_by_name = {}

for (
    location_key,
    hotel_key
), hotel_address in hotel_address_lookup.items():

    if hotel_key not in hotel_address_by_name:
        hotel_address_by_name[
            hotel_key
        ] = hotel_address


print(
    "Số khách sạn trong bảng tra địa chỉ:",
    len(hotel_address_lookup)
)


# ============================================================
# 5. DUYỆT TOÀN BỘ HTML PHÒNG
# ============================================================

room_data = []


for location_name, hotel_list in (
    hotel_room_html_dict.items()
):

    for hotel in hotel_list:
        hotel_name = hotel.get(
            "hotel_name"
        )

        hotel_url = hotel.get(
            "hotel_url"
        )

        room_page_url = hotel.get(
            "room_page_url"
        )

        room_page_html = hotel.get(
            "room_page_html"
        )

        # ----------------------------------------------------
        # Địa chỉ khách sạn
        # ----------------------------------------------------
        hotel_address = (
            hotel.get("hotel_address")
            or hotel.get("address")
        )

        if not hotel_address:
            location_key = normalize_key(
                location_name
            )

            hotel_key = normalize_key(
                hotel_name
            )

            hotel_address = (
                hotel_address_lookup.get(
                    (location_key, hotel_key)
                )
            )

            if not hotel_address:
                hotel_address = (
                    hotel_address_by_name.get(
                        hotel_key
                    )
                )

        if not room_page_html:
            print(
                f"Không có HTML phòng: "
                f"{hotel_name}"
            )
            continue

        soup = BeautifulSoup(
            room_page_html,
            "lxml"
        )

        room_title_tags = soup.select(
            ".room-content-title"
        )

        print(
            f"{location_name} | {hotel_name}: "
            f"tìm thấy {len(room_title_tags)} phòng"
        )

        for room_index, title_tag in enumerate(
            room_title_tags,
            start=1
        ):
            room_container = find_room_container(
                title_tag
            )

            if room_container is None:
                print(
                    "Không xác định được container: "
                    f"{clean_text(title_tag)}"
                )
                continue

            # =================================================
            # TÊN PHÒNG
            # =================================================
            room_name = clean_text(
                title_tag
            )

            # =================================================
            # MÔ TẢ PHÒNG
            # =================================================
            description_tag = (
                room_container.select_one(
                    ".room-content-text"
                )
            )

            room_description = clean_text(
                description_tag
            )

            # =================================================
            # LINK ẢNH
            # =================================================
            room_image_url = (
                extract_room_image_url(
                    room_container=room_container,
                    title_tag=title_tag
                )
            )

            # =================================================
            # THÔNG TIN PHÒNG
            # =================================================
            basic_info = (
                extract_room_basic_info(
                    room_container
                )
            )

            # =================================================
            # GIÁ PHÒNG
            # =================================================
            price_info = extract_room_prices(
                room_container
            )

            # =================================================
            # TIỆN ÍCH
            # =================================================
            amenity_list = extract_amenities(
                room_container
            )

            if any(
                (
                    "wifi" in amenity.lower()
                    or "wi-fi" in amenity.lower()
                    or "internet" in amenity.lower()
                )
                for amenity in amenity_list
            ):
                basic_info["wifi"] = True

            amenities_text = ", ".join(
                amenity_list
            )

            # =================================================
            # THÊM DỮ LIỆU
            # =================================================
            room_data.append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_address": hotel_address,
                "hotel_url": hotel_url,
                "room_page_url": room_page_url,
                "room_index": room_index,
                "room_name": room_name,
                "room_image_url": room_image_url,
                "room_description": room_description,
                "guest_count": basic_info[
                    "guest_count"
                ],
                "room_area": basic_info[
                    "room_area"
                ],
                "bed_types": basic_info[
                    "bed_types"
                ],
                "wifi": basic_info[
                    "wifi"
                ],
                "standard_rate": price_info[
                    "standard_rate"
                ],
                "price_only_from": price_info[
                    "price_only_from"
                ],
                "amenities": amenities_text
            })


# ============================================================
# 6. CHUYỂN THÀNH DATAFRAME
# ============================================================

room_df = pd.DataFrame(
    room_data
)


column_order = [
    "location_name",
    "hotel_name",
    "hotel_address",
    "hotel_url",
    "room_page_url",
    "room_index",
    "room_name",
    "room_image_url",
    "room_description",
    "guest_count",
    "room_area",
    "bed_types",
    "wifi",
    "standard_rate",
    "price_only_from",
    "amenities"
]


existing_columns = [
    column
    for column in column_order
    if column in room_df.columns
]


room_df = room_df[
    existing_columns
]


# ============================================================
# 7. KIỂM TRA DỮ LIỆU THIẾU
# ============================================================

if not room_df.empty:

    missing_address_df = (
        room_df[
            room_df["hotel_address"].isna()
            | room_df["hotel_address"]
            .astype(str)
            .str.strip()
            .eq("")
        ][
            [
                "location_name",
                "hotel_name",
                "hotel_address"
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    missing_image_df = (
        room_df[
            room_df["room_image_url"].isna()
            | room_df["room_image_url"]
            .astype(str)
            .str.strip()
            .eq("")
        ][
            [
                "location_name",
                "hotel_name",
                "room_name",
                "room_image_url"
            ]
        ]
        .reset_index(drop=True)
    )

    missing_price_df = (
        room_df[
            (
                room_df["standard_rate"].isna()
                | room_df["standard_rate"]
                .astype(str)
                .str.strip()
                .eq("")
            )
            &
            (
                room_df["price_only_from"].isna()
                | room_df["price_only_from"]
                .astype(str)
                .str.strip()
                .eq("")
            )
        ][
            [
                "location_name",
                "hotel_name",
                "room_name",
                "standard_rate",
                "price_only_from"
            ]
        ]
        .reset_index(drop=True)
    )

    print("\nHoàn thành!")
    print(
        "Tổng số phòng:",
        len(room_df)
    )

    print(
        "Số phòng chưa có địa chỉ:",
        len(missing_address_df)
    )

    print(
        "Số phòng chưa có ảnh:",
        len(missing_image_df)
    )

    print(
        "Số phòng không tìm thấy giá "
        "và cũng không có link liên hệ:",
        len(missing_price_df)
    )

    if not missing_address_df.empty:
        print(
            "\nKhách sạn chưa ghép được địa chỉ:"
        )
        display(
            missing_address_df
        )

    if not missing_image_df.empty:
        print(
            "\nPhòng chưa lấy được ảnh:"
        )
        display(
            missing_image_df
        )

    if not missing_price_df.empty:
        print(
            "\nPhòng chưa lấy được giá hoặc liên hệ:"
        )
        display(
            missing_price_df
        )

else:
    print(
        "Không thu được dữ liệu phòng."
    )


# ============================================================
# 8. GHI ĐÈ FILE CSV
# ============================================================

csv_file_name = (
    r"hotel\vinpearl_room_data.csv"
)

room_df.to_csv(
    csv_file_name,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nĐã cập nhật CSV: "
    f"{csv_file_name}"
)

print(
    "Các cột trong CSV:"
)

print(
    room_df.columns.tolist()
)


# ============================================================
# 9. HIỂN THỊ KẾT QUẢ
# ============================================================

display(
    room_df
)

Số khách sạn trong bảng tra địa chỉ: 15
Phu Quoc | Vinpearl Resort & Spa Phu Quoc: tìm thấy 7 phòng
Phu Quoc | Vinpearl Wonderworld Phu Quoc: tìm thấy 8 phòng
Phu Quoc | VinHolidays Fiesta Phu Quoc: tìm thấy 7 phòng
Nha Trang | Vinpearl Beachfront Nha Trang: tìm thấy 9 phòng
Nha Trang | Vinpearl Resort & Spa Nha Trang Bay: tìm thấy 11 phòng
Nha Trang | Vinpearl Resort Nha Trang: tìm thấy 10 phòng
Nha Trang | Vinpearl Luxury Nha Trang: tìm thấy 5 phòng
Nha Trang | Hon Tam Resort: tìm thấy 9 phòng
Nha Trang | Vinpearl Empire Nha Trang, Affiliated by Meliá: tìm thấy 4 phòng
Nam Hoi An | Vinpearl Resort & Golf Nam Hoi An: tìm thấy 8 phòng
Ha Tinh | Vinpearl Cua Sot Resort, Affiliated by Meliá: tìm thấy 3 phòng
Ha Tinh | Vinpearl Ha Tinh, Affiliated by Meliá: tìm thấy 8 phòng
Nghe An | Vinpearl Cua Hoi Resort, Affiliated by Meliá: tìm thấy 12 phòng
Bac Ninh | Vinpearl Hotel Bac Ninh: tìm thấy 6 phòng
Ha Long | Vinpearl Resort & Spa Ha Long: tìm thấy 9 phòng

Hoàn thành!
Tổng số phòng: 116
S

,location_name,hotel_name,hotel_address,hotel_url,room_page_url,room_index,room_name,room_image_url,room_description,guest_count,room_area,bed_types,wifi,standard_rate,price_only_from,amenities
0,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,"Bai Dai Area, Ganh Dau Commune, Phu Quoc City,...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,1,Deluxe Garden View Twin Bed,https://booking-static.vinpearl.com/room_types...,The 46m2 Deluxe Garden View Twin Bed comes wit...,4,46 m²,Twin-bed,True,~ 107USD,~ 102USD,"Electronic scales, Telephone, Shower, Air-cond..."
1,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,"Bai Dai Area, Ganh Dau Commune, Phu Quoc City,...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,2,Deluxe Garden View King Bed,https://booking-static.vinpearl.com/room_types...,The 46m2 Deluxe Garden View King Bed comes wit...,4,46 m²,King-bed,True,~ 107USD,~ 102USD,"Electronic scales, Telephone, Shower, Air-cond..."
2,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,"Bai Dai Area, Ganh Dau Commune, Phu Quoc City,...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,3,Deluxe Ocean View Twin Bed,https://booking-static.vinpearl.com/room_types...,The 46m2 Deluxe Ocean View Twin Bed comes with...,4,46 m²,Twin-bed,True,~ 126USD,~ 120USD,"Electronic scales, Telephone, Shower, Air-cond..."
3,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,"Bai Dai Area, Ganh Dau Commune, Phu Quoc City,...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,4,Deluxe Ocean View King Bed,https://booking-static.vinpearl.com/room_types...,The 46m2 Deluxe Ocean View King Bed comes with...,4,46 m²,King-bed,True,~ 126USD,~ 120USD,"Electronic scales, Telephone, Shower, Air-cond..."
4,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,"Bai Dai Area, Ganh Dau Commune, Phu Quoc City,...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,5,Junior Suite Garden View King Bed,https://booking-static.vinpearl.com/room_types...,"With an area of 86sqm,Junior Suite Garden View...",4,86 m²,King-bed,True,~ 176USD,~ 167USD,"Electronic scales, Telephone, King-bed, Shower..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Ha Long,Vinpearl Resort & Spa Ha Long,"Reu Island, Bai Chay Ward, Ha Long City, Quang...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,5,Executive Suite King Bed,https://booking-static.vinpearl.com/room_types...,"The 76m², Executive Suite King Bed is generous...",4,76 m²,King-bed,True,~ 321USD,~ 305USD,"Shower, Air-conditioner, Slippers, Hairdryer, ..."
112,Ha Long,Vinpearl Resort & Spa Ha Long,"Reu Island, Bai Chay Ward, Ha Long City, Quang...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,6,Family Suite,https://booking-static.vinpearl.com/room_types...,None,4,72 m²,"Twin-bed, King-bed",True,~ 476USD,~ 452USD,"Telephone, Shower, Twin-bed, Air-conditioner, ..."
113,Ha Long,Vinpearl Resort & Spa Ha Long,"Reu Island, Bai Chay Ward, Ha Long City, Quang...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,7,Deluxe Ocean View King Bed,https://booking-static.vinpearl.com/room_types...,The 38m² Deluxe Ocean View King Bed is thought...,4,38 m²,King-bed,True,tel:1900232389,tel:1900232389,"Telephone, Air-conditioner, Slippers, Hairdrye..."
114,Ha Long,Vinpearl Resort & Spa Ha Long,"Reu Island, Bai Chay Ward, Ha Long City, Quang...",https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,8,Deluxe Ocean View Twin Bed,https://booking-static.vinpearl.com/room_types...,The 38m² Deluxe Ocean View Twin Bed is thought...,4,38 m²,Twin-bed,True,tel:1900232389,tel:1900232389,"Telephone, Shower, Air-conditioner, Slippers, ..."


In [34]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait


# ============================================================
# LẤY LINK DINING CỦA TỪNG KHÁCH SẠN
# ============================================================

hotel_dining_link_dict = {}


for location_name, location_data in hotel_data_by_location.items():

    hotel_dining_link_dict[location_name] = []

    for hotel in location_data["hotels"]:

        hotel_name = hotel["hotel_name"]
        hotel_url = hotel["hotel_url"]

        hotel_driver = None

        try:
            print(f"Đang truy cập khách sạn: {hotel_name}")
            print(f"URL: {hotel_url}")

            hotel_driver = webdriver.Edge(
                options=options
            )

            hotel_driver.get(
                hotel_url
            )

            WebDriverWait(
                hotel_driver,
                120
            ).until(
                lambda d: (
                    d.execute_script(
                        "return document.readyState"
                    ) == "complete"
                    and "just a moment"
                    not in d.title.lower()
                    and "attention required"
                    not in d.title.lower()
                )
            )

            hotel_detail_html = (
                hotel_driver.page_source
            )

            hotel_detail_soup = BeautifulSoup(
                hotel_detail_html,
                "lxml"
            )

            # =================================================
            # TÌM LINK DINING
            #
            # HTML mẫu:
            # <div class="p-micriosites__dining__head">
            #     <h3>Dining</h3>
            #     <a href=".../foods"
            #        class="cpn-content-more">
            #         View all
            #     </a>
            # </div>
            # =================================================

            dining_link_tag = hotel_detail_soup.select_one(
                '#tab-home > div:nth-of-type(4) > div > div:nth-of-type(1) > a[href]'
                )

            # Dự phòng nếu class container có thay đổi nhẹ
            if dining_link_tag is None:

                for heading_tag in hotel_detail_soup.select(
                    "h2, h3, h4"
                ):
                    heading_text = heading_tag.get_text(
                        " ",
                        strip=True
                    ).lower()

                    if heading_text == "dining":

                        parent = heading_tag.parent

                        if parent is not None:
                            dining_link_tag = parent.select_one(
                                "a[href]"
                            )

                        if dining_link_tag is not None:
                            break

            if dining_link_tag is None:
                print(
                    f"Không tìm thấy link Dining: "
                    f"{hotel_name}"
                )
                print("-" * 50)
                continue

            dining_page_url = urljoin(
                hotel_driver.current_url,
                dining_link_tag["href"]
            )

            hotel_dining_link_dict[
                location_name
            ].append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_url": hotel_driver.current_url,
                "dining_page_url": dining_page_url
            })

            print(
                f"Đã lấy link Dining: {hotel_name}"
            )
            print(
                f"Link Dining: {dining_page_url}"
            )
            print("-" * 50)

        except Exception as error:
            print(
                f"Lỗi khi lấy link Dining: "
                f"{hotel_name}"
            )
            print(
                f"Loại lỗi: "
                f"{type(error).__name__}"
            )
            print(
                f"Nội dung lỗi: {error}"
            )
            print("-" * 50)

        finally:
            if hotel_driver is not None:
                hotel_driver.quit()

Đang truy cập khách sạn: Vinpearl Resort & Spa Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc
Đã lấy link Dining: Vinpearl Resort & Spa Phu Quoc
Link Dining: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/foods
--------------------------------------------------
Đang truy cập khách sạn: Vinpearl Wonderworld Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc
Đã lấy link Dining: Vinpearl Wonderworld Phu Quoc
Link Dining: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/foods
--------------------------------------------------
Đang truy cập khách sạn: VinHolidays Fiesta Phu Quoc
URL: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc
Đã lấy link Dining: VinHolidays Fiesta Phu Quoc
Link Dining: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/foods
--------------------------------------------------
Đang truy cập khách sạn: Vinpearl Beachfront Nha Trang
URL: https://vinpearl.com/en/hotels/vinpearl-be

In [35]:
for location_name, hotel_list in hotel_dining_link_dict.items():

    print()
    print(f"ĐỊA ĐIỂM: {location_name}")
    print("=" * 60)

    for hotel in hotel_list:
        print(f"Tên khách sạn: {hotel['hotel_name']}")
        print(f"Link khách sạn: {hotel['hotel_url']}")
        print(f"Link Dining: {hotel['dining_page_url']}")
        print("-" * 50)


ĐỊA ĐIỂM: Phu Quoc
Tên khách sạn: Vinpearl Resort & Spa Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc
Link Dining: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/foods
--------------------------------------------------
Tên khách sạn: Vinpearl Wonderworld Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc
Link Dining: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/foods
--------------------------------------------------
Tên khách sạn: VinHolidays Fiesta Phu Quoc
Link khách sạn: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc
Link Dining: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/foods
--------------------------------------------------

ĐỊA ĐIỂM: Nha Trang
Tên khách sạn: Vinpearl Beachfront Nha Trang
Link khách sạn: https://vinpearl.com/en/hotels/vinpearl-beachfront-nha-trang
Link Dining: https://vinpearl.com/en/hotels/vinpearl-beachfront-nha-trang/foods
-----

In [36]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait


# ============================================================
# LẤY TOÀN BỘ HTML TRANG DINING CỦA TỪNG KHÁCH SẠN
# ============================================================

hotel_dining_html_dict = {}


for location_name, hotel_list in hotel_dining_link_dict.items():

    hotel_dining_html_dict[location_name] = []

    print()
    print("=" * 70)
    print(f"ĐỊA ĐIỂM: {location_name}")
    print("=" * 70)

    for hotel in hotel_list:

        hotel_name = hotel.get("hotel_name")
        hotel_url = hotel.get("hotel_url")
        dining_page_url = hotel.get("dining_page_url")

        dining_driver = None

        try:
            if not dining_page_url:
                print(
                    f"Không có link Dining: {hotel_name}"
                )
                print("-" * 60)
                continue

            print(
                f"Đang truy cập trang Dining: {hotel_name}"
            )
            print(
                f"URL: {dining_page_url}"
            )

            dining_driver = webdriver.Edge(
                options=options
            )

            dining_driver.get(
                dining_page_url
            )

            # Chờ trang tải xong và vượt qua Cloudflare
            WebDriverWait(
                dining_driver,
                120
            ).until(
                lambda d: (
                    d.execute_script(
                        "return document.readyState"
                    ) == "complete"
                    and "just a moment"
                    not in d.title.lower()
                    and "attention required"
                    not in d.title.lower()
                )
            )

            # Cuộn xuống cuối trang để kích hoạt lazy loading
            last_height = dining_driver.execute_script(
                "return document.body.scrollHeight"
            )

            for _ in range(10):

                dining_driver.execute_script(
                    "window.scrollTo("
                    "0, document.body.scrollHeight"
                    ");"
                )

                WebDriverWait(
                    dining_driver,
                    10
                ).until(
                    lambda d: d.execute_script(
                        "return document.readyState"
                    ) == "complete"
                )

                new_height = dining_driver.execute_script(
                    "return document.body.scrollHeight"
                )

                if new_height == last_height:
                    break

                last_height = new_height

            # Trở lại đầu trang
            dining_driver.execute_script(
                "window.scrollTo(0, 0);"
            )

            # Lấy toàn bộ HTML sau khi trang đã tải
            dining_page_html = (
                dining_driver.page_source
            )

            actual_dining_url = (
                dining_driver.current_url
            )

            # Lưu HTML vào dictionary riêng
            hotel_dining_html_dict[
                location_name
            ].append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_url": hotel_url,
                "dining_page_url": actual_dining_url,
                "dining_page_html": dining_page_html
            })

            print(
                f"Đã lấy HTML Dining: {hotel_name}"
            )
            print(
                f"Độ dài HTML: "
                f"{len(dining_page_html):,} ký tự"
            )
            print("-" * 60)

        except Exception as error:
            print(
                f"Lỗi khi lấy HTML Dining: "
                f"{hotel_name}"
            )
            print(
                f"Loại lỗi: "
                f"{type(error).__name__}"
            )
            print(
                f"Nội dung lỗi: {error}"
            )
            print("-" * 60)

        finally:
            if dining_driver is not None:
                dining_driver.quit()


# ============================================================
# THỐNG KÊ KẾT QUẢ
# ============================================================

total_dining_pages = sum(
    len(hotel_list)
    for hotel_list in hotel_dining_html_dict.values()
)

print()
print("=" * 70)
print("HOÀN THÀNH LẤY HTML DINING")
print(f"Tổng số trang Dining đã lấy: {total_dining_pages}")
print("=" * 70)


ĐỊA ĐIỂM: Phu Quoc
Đang truy cập trang Dining: Vinpearl Resort & Spa Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-resort-spa-phu-quoc/foods
Đã lấy HTML Dining: Vinpearl Resort & Spa Phu Quoc
Độ dài HTML: 184,767 ký tự
------------------------------------------------------------
Đang truy cập trang Dining: Vinpearl Wonderworld Phu Quoc
URL: https://vinpearl.com/en/hotels/vinpearl-wonderworld-phu-quoc/foods
Đã lấy HTML Dining: Vinpearl Wonderworld Phu Quoc
Độ dài HTML: 219,870 ký tự
------------------------------------------------------------
Đang truy cập trang Dining: VinHolidays Fiesta Phu Quoc
URL: https://vinpearl.com/en/hotels/vinholidays-fiesta-phu-quoc/foods
Đã lấy HTML Dining: VinHolidays Fiesta Phu Quoc
Độ dài HTML: 152,402 ký tự
------------------------------------------------------------

ĐỊA ĐIỂM: Nha Trang
Đang truy cập trang Dining: Vinpearl Beachfront Nha Trang
URL: https://vinpearl.com/en/hotels/vinpearl-beachfront-nha-trang/foods
Đã lấy HTML Dining: Vinpearl B

In [37]:
import re
import pandas as pd
from bs4 import BeautifulSoup


# ============================================================
# 1. HÀM HỖ TRỢ
# ============================================================

def clean_dining_text(tag):
    """
    Lấy text từ thẻ HTML và chuẩn hóa khoảng trắng.
    """
    if tag is None:
        return None

    text = tag.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def get_dining_image_url(img_tag):
    """
    Lấy link ảnh dịch vụ Dining.

    Ưu tiên:
    1. data-src
    2. data-lazy-src
    3. data-original
    4. src
    """
    if img_tag is None:
        return None

    image_url = (
        img_tag.get("data-src")
        or img_tag.get("data-lazy-src")
        or img_tag.get("data-original")
        or img_tag.get("src")
    )

    if image_url:
        image_url = image_url.strip()

    return image_url or None


def find_dining_service_container(title_tag):
    """
    Tìm thẻ cha bao trùm đúng một nhà hàng/dịch vụ Dining.

    Container nên chứa:
    - tên dịch vụ;
    - ảnh restaurant-image;
    - mô tả hoặc giờ hoạt động.

    Không dùng XPath tuyệt đối vì cấu trúc div có thể thay đổi.
    """
    for parent in title_tag.parents:

        if parent.name not in ["div", "li", "article", "section"]:
            continue

        title_count = len(
            parent.select("h3")
        )

        has_image = parent.select_one(
            "img.restaurant-image, "
            'img[data-fancybox^="gallery_"]'
        )

        has_description = parent.select_one(
            "p"
        )

        # Ưu tiên container chỉ chứa một tiêu đề dịch vụ
        if (
            title_count == 1
            and has_image is not None
            and has_description is not None
        ):
            return parent

    # Dự phòng: container có tên và ảnh
    for parent in title_tag.parents:

        if parent.name not in ["div", "li", "article", "section"]:
            continue

        title_count = len(
            parent.select("h3")
        )

        has_image = parent.select_one(
            "img.restaurant-image, "
            'img[data-fancybox^="gallery_"]'
        )

        if (
            title_count == 1
            and has_image is not None
        ):
            return parent

    return None


def extract_opening_hours(service_container):
    """
    Lấy giờ hoạt động của nhà hàng.

    Ưu tiên các thẻ chứa từ khóa:
    - Opening hours
    - Open
    - Hours
    - Giờ hoạt động

    Sau đó mới dùng selector dự phòng.
    """

    opening_hours = None

    # --------------------------------------------------------
    # Cách 1: tìm theo nội dung text
    # --------------------------------------------------------
    candidate_tags = service_container.select(
        "span, div, p"
    )

    hour_keywords = [
        "opening hour",
        "opening time",
        "operating hour",
        "business hour",
        "hours:",
        "open:",
        "giờ hoạt động",
        "thời gian hoạt động"
    ]

    time_pattern = re.compile(
        r"\b\d{1,2}"
        r"(?::\d{2})?"
        r"\s*(?:am|pm)?"
        r"\s*(?:-|–|—|to)"
        r"\s*\d{1,2}"
        r"(?::\d{2})?"
        r"\s*(?:am|pm)?\b",
        flags=re.IGNORECASE
    )

    for tag in candidate_tags:
        text = clean_dining_text(tag)

        if not text:
            continue

        text_lower = text.lower()

        if (
            any(keyword in text_lower for keyword in hour_keywords)
            or time_pattern.search(text)
        ):
            # Tránh lấy một container quá lớn chứa toàn bộ nội dung
            if len(text) <= 300:
                opening_hours = text
                break

    # --------------------------------------------------------
    # Cách 2: dự phòng theo cấu trúc thường gặp
    # --------------------------------------------------------
    if opening_hours is None:
        hour_tag = service_container.select_one(
            ".restaurant-time span, "
            ".restaurant-hours span, "
            ".opening-hours span, "
            ".time span"
        )

        opening_hours = clean_dining_text(
            hour_tag
        )

    return opening_hours


def extract_dining_description(service_container):
    """
    Lấy phần mô tả của từng dịch vụ Dining.
    """

    description = None

    # Ưu tiên paragraph có nội dung dài nhất
    paragraph_tags = service_container.select(
        "p"
    )

    paragraph_texts = []

    for paragraph_tag in paragraph_tags:
        paragraph_text = clean_dining_text(
            paragraph_tag
        )

        if paragraph_text:
            paragraph_texts.append(
                paragraph_text
            )

    if paragraph_texts:
        # Mô tả thường là đoạn văn dài nhất
        description = max(
            paragraph_texts,
            key=len
        )

    return description


def extract_dining_contact(service_container):
    """
    Lấy thông tin liên hệ của đúng nhà hàng.

    Có thể lấy:
    - href dạng tel:
    - href dạng mailto:
    - số điện thoại hiển thị
    - nội dung trong vùng contact
    """

    contact = None

    # --------------------------------------------------------
    # 1. Ưu tiên href tel:
    # --------------------------------------------------------
    phone_link = service_container.select_one(
        'a[href^="tel:"]'
    )

    if phone_link is not None:
        contact = phone_link.get("href")

        if contact:
            return contact.strip()

    # --------------------------------------------------------
    # 2. Email
    # --------------------------------------------------------
    email_link = service_container.select_one(
        'a[href^="mailto:"]'
    )

    if email_link is not None:
        contact = email_link.get("href")

        if contact:
            return contact.strip()

    # --------------------------------------------------------
    # 3. Tìm vùng có class liên quan đến contact/phone
    # --------------------------------------------------------
    contact_tag = service_container.select_one(
        '[class*="contact"], '
        '[class*="phone"], '
        '[class*="hotline"]'
    )

    contact_text = clean_dining_text(
        contact_tag
    )

    if contact_text:
        return contact_text

    # --------------------------------------------------------
    # 4. Tìm số điện thoại trong text của container
    # --------------------------------------------------------
    full_text = clean_dining_text(
        service_container
    ) or ""

    phone_match = re.search(
        r"(?:\+?\d[\d\s().-]{7,}\d)",
        full_text
    )

    if phone_match:
        contact = re.sub(
            r"\s+",
            " ",
            phone_match.group()
        ).strip()

    return contact


def extract_dining_service(service_container, title_tag):
    """
    Lấy toàn bộ thông tin của một dịch vụ Dining.
    """

    # Tên dịch vụ
    service_name = clean_dining_text(
        title_tag
    )

    # Ảnh dịch vụ
    image_tag = service_container.select_one(
        "img.restaurant-image[data-src], "
        "img.restaurant-image[src], "
        'img[data-fancybox^="gallery_"][data-src], '
        'img[data-fancybox^="gallery_"][src]'
    )

    service_image_url = get_dining_image_url(
        image_tag
    )

    # Giờ hoạt động
    opening_hours = extract_opening_hours(
        service_container
    )

    # Mô tả
    description = extract_dining_description(
        service_container
    )

    # Liên hệ
    contact = extract_dining_contact(
        service_container
    )

    return {
        "service_name": service_name,
        "service_image_url": service_image_url,
        "opening_hours": opening_hours,
        "description": description,
        "contact": contact
    }


# ============================================================
# 2. DUYỆT HTML DINING CỦA TỪNG KHÁCH SẠN
# ============================================================

hotel_dining_service_data = []


for location_name, hotel_list in hotel_dining_html_dict.items():

    print()
    print("=" * 70)
    print(f"ĐỊA ĐIỂM: {location_name}")
    print("=" * 70)

    for hotel in hotel_list:

        hotel_name = hotel.get(
            "hotel_name"
        )

        hotel_url = hotel.get(
            "hotel_url"
        )

        dining_page_url = hotel.get(
            "dining_page_url"
        )

        dining_page_html = hotel.get(
            "dining_page_html"
        )

        if not dining_page_html:
            print(
                f"Không có HTML Dining: {hotel_name}"
            )
            print("-" * 60)
            continue

        dining_soup = BeautifulSoup(
            dining_page_html,
            "lxml"
        )

        # ====================================================
        # TÌM TÊN TỪNG DỊCH VỤ/NHÀ HÀNG
        # ====================================================

        title_tags = []

        # Cách chính:
        # tìm các h3 nằm trong khu vực có ảnh restaurant-image
        for h3_tag in dining_soup.select("h3"):

            service_container = (
                find_dining_service_container(
                    h3_tag
                )
            )

            if service_container is not None:
                title_tags.append(
                    h3_tag
                )

        # Loại bỏ tiêu đề trùng
        unique_title_tags = []
        seen_titles = set()

        for title_tag in title_tags:

            title_text = clean_dining_text(
                title_tag
            )

            if not title_text:
                continue

            normalized_title = (
                title_text.strip().lower()
            )

            if normalized_title in seen_titles:
                continue

            seen_titles.add(
                normalized_title
            )

            unique_title_tags.append(
                title_tag
            )

        print(
            f"{hotel_name}: "
            f"tìm thấy {len(unique_title_tags)} "
            f"dịch vụ Dining"
        )

        for service_index, title_tag in enumerate(
            unique_title_tags,
            start=1
        ):

            service_container = (
                find_dining_service_container(
                    title_tag
                )
            )

            if service_container is None:
                print(
                    "Không xác định được container: "
                    f"{clean_dining_text(title_tag)}"
                )
                continue

            service_info = extract_dining_service(
                service_container=service_container,
                title_tag=title_tag
            )

            hotel_dining_service_data.append({
                "location_name": location_name,
                "hotel_name": hotel_name,
                "hotel_url": hotel_url,
                "dining_page_url": dining_page_url,
                "service_index": service_index,
                "service_name": service_info[
                    "service_name"
                ],
                "service_image_url": service_info[
                    "service_image_url"
                ],
                "opening_hours": service_info[
                    "opening_hours"
                ],
                "description": service_info[
                    "description"
                ],
                "contact": service_info[
                    "contact"
                ]
            })

            print(
                f"{service_index}. "
                f"{service_info['service_name']}"
            )

        print("-" * 60)


# ============================================================
# 3. CHUYỂN THÀNH DATAFRAME RIÊNG
# ============================================================

hotel_dining_service_df = pd.DataFrame(
    hotel_dining_service_data
)


column_order = [
    "location_name",
    "hotel_name",
    "hotel_url",
    "dining_page_url",
    "service_index",
    "service_name",
    "service_image_url",
    "opening_hours",
    "description",
    "contact"
]


existing_columns = [
    column
    for column in column_order
    if column in hotel_dining_service_df.columns
]


hotel_dining_service_df = (
    hotel_dining_service_df[
        existing_columns
    ]
)


# ============================================================
# 4. KIỂM TRA KẾT QUẢ
# ============================================================

print()
print("=" * 70)
print("HOÀN THÀNH TÁCH DỮ LIỆU DINING")
print(
    "Tổng số dịch vụ Dining:",
    len(hotel_dining_service_df)
)
print("=" * 70)


if not hotel_dining_service_df.empty:

    missing_image_df = hotel_dining_service_df[
        hotel_dining_service_df[
            "service_image_url"
        ].isna()
        |
        hotel_dining_service_df[
            "service_image_url"
        ].astype(str).str.strip().eq("")
    ]

    missing_name_df = hotel_dining_service_df[
        hotel_dining_service_df[
            "service_name"
        ].isna()
        |
        hotel_dining_service_df[
            "service_name"
        ].astype(str).str.strip().eq("")
    ]

    print(
        "Số dịch vụ chưa có ảnh:",
        len(missing_image_df)
    )

    print(
        "Số dịch vụ chưa có tên:",
        len(missing_name_df)
    )


display(
    hotel_dining_service_df
)


ĐỊA ĐIỂM: Phu Quoc
Vinpearl Resort & Spa Phu Quoc: tìm thấy 6 dịch vụ Dining
1. Seashell Restaurant
2. Nemo Restaurant
3. The Beach Bistro
4. Pearl Lounge Café
5. Sim bar
6. Sea view Café
------------------------------------------------------------
Vinpearl Wonderworld Phu Quoc: tìm thấy 11 dịch vụ Dining
1. The Haven Beach Club
2. The Delight 1 Restaurant
3. The Delight 2 Restaurant
4. Taras Restaurant
5. Velvet Bar
6. La Perle d'Indochine Restaurant
7. The Delight Lounge
8. Haven Lounge
9. Pool Bar
10. BBQ Party & Fire Dance Show
11. The Haven Sunset Show
------------------------------------------------------------
VinHolidays Fiesta Phu Quoc: tìm thấy 2 dịch vụ Dining
1. Ola Costa Restaurant
2. Scorpio Bar VinHolidays
------------------------------------------------------------

ĐỊA ĐIỂM: Nha Trang
Vinpearl Beachfront Nha Trang: tìm thấy 4 dịch vụ Dining
1. Coral Restaurant
2. Lagoon Restaurant
3. Infinity Pool Bar
4. Lobby Bar
------------------------------------------------------

,location_name,hotel_name,hotel_url,dining_page_url,service_index,service_name,service_image_url,opening_hours,description,contact
0,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,1,Seashell Restaurant,https://statics.vinpearl.com/vinpearl-resort-&...,Seashell Restaurant Breakfast: 6:00AM – 10:30A...,Seashell Restaurant offers a wide range of Asi...,tel:(+84) 297 3550 550
1,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,2,Nemo Restaurant,https://statics.vinpearl.com/vinpearl-resort-&...,Nemo Restaurant Breakfast: 6:00AM – 10:30AM Lu...,Enjoy superb traditional Vietnamese and Southe...,tel:(+84) 297 3550 550
2,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,3,The Beach Bistro,https://statics.vinpearl.com/the-beach--3_1724...,12:00AM - 10:00PM,Step into The Beach Bistro and indulge in mome...,tel:(+84) 297 3550 550
3,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,4,Pearl Lounge Café,https://statics.vinpearl.com/vinpearl-resort-&...,07:00AM – 11:00PM,Bathed in natural light and imbued with Orient...,tel:(+84) 297 3550 550
4,Phu Quoc,Vinpearl Resort & Spa Phu Quoc,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,5,Sim bar,https://statics.vinpearl.com/vinpearl-resort-&...,Sim bar 09:00AM – 11:00PM (+84) 297 3550 550 (...,The most relaxing place to watch sunset on the...,tel:(+84) 297 3550 550
...,...,...,...,...,...,...,...,...,...,...
63,Ha Long,Vinpearl Resort & Spa Ha Long,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,1,Bayview Restaurant,https://statics.vinpearl.com/vinpearl-resort-&...,"Bayview Restaurant 6:00AM - 10:00PM Elegant, m...","Elegant, modern with stunning views of the spe...",None
64,Ha Long,Vinpearl Resort & Spa Ha Long,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,2,Accogliente Restaurant,https://statics.vinpearl.com/DSC00560-HDR_1749...,Accogliente Restaurant 12:00PM - 10:00PM With ...,"With its elegant ambiance, Accogliente Restaur...",None
65,Ha Long,Vinpearl Resort & Spa Ha Long,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,3,Pavilion Restaurant,https://statics.vinpearl.com/pavilion-1_170663...,6:00PM - 10:00PM,The freshest seafood and the finest sizzling m...,None
66,Ha Long,Vinpearl Resort & Spa Ha Long,https://vinpearl.com/en/hotels/vinpearl-resort...,https://vinpearl.com/en/hotels/vinpearl-resort...,4,Pearl Lounge,https://statics.vinpearl.com/VPRSHL Pearl Loun...,Pearl Lounge 7:00AM - 11:00PM With its unclutt...,"With its uncluttered open design, the Pearl Lo...",None


In [38]:
import os

# Tạo thư mục hotel nếu chưa tồn tại
output_folder = "hotel"
os.makedirs(output_folder, exist_ok=True)

# Tên file CSV
file_name = "vinpearl_dining_service_data.csv"

# Đường dẫn đầy đủ
csv_path = os.path.join(
    output_folder,
    file_name
)

# Lưu CSV
hotel_dining_service_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã lưu CSV tại: {csv_path}")
print(f"Tổng số dòng: {len(hotel_dining_service_df)}")
print(f"Tổng số cột: {len(hotel_dining_service_df.columns)}")

Đã lưu CSV tại: hotel\vinpearl_dining_service_data.csv
Tổng số dòng: 68
Tổng số cột: 10


## ✅ Hoàn thành dữ liệu Hotel

### Mô tả dữ liệu

Thư mục `hotel/` gồm 2 file CSV:

1. **`vinpearl_dining_service_data.csv`**  
   Chứa thông tin các dịch vụ Dining của từng khách sạn.

2. **`vinpearl_room_data.csv`**  
   Chứa thông tin chi tiết các loại phòng của từng khách sạn.

---

In [8]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
import time


# Lấy link VinWonders từ dictionary
vinwonders_url = data_info_topic.get(
    "VinWonders Theme Parks",
    "https://vinwonders.com/en/"
)

# Cấu hình Edge
vinwonders_options = Options()
vinwonders_options.add_argument("--start-maximized")

if "HEADERS" in globals() and "User-Agent" in HEADERS:
    vinwonders_options.add_argument(
        f"user-agent={HEADERS['User-Agent']}"
    )

# Tạo phiên WebDriver mới
vinwonders_driver = webdriver.Edge(
    options=vinwonders_options
)

try:
    vinwonders_driver.get(vinwonders_url)

    # Chờ trang tải xong và vượt qua Cloudflare
    WebDriverWait(vinwonders_driver, 120).until(
        lambda driver: (
            driver.execute_script(
                "return document.readyState"
            ) == "complete"
            and "just a moment" not in driver.title.lower()
            and len(driver.page_source) > 5000
        )
    )

    # Cuộn xuống để tải các nội dung lazy loading
    previous_height = 0

    for _ in range(15):
        vinwonders_driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )

        time.sleep(1.5)

        current_height = vinwonders_driver.execute_script(
            "return document.body.scrollHeight"
        )

        if current_height == previous_height:
            break

        previous_height = current_height

    # Lưu dữ liệu trực tiếp vào biến
    vinwonders_html = vinwonders_driver.page_source
    vinwonders_current_url = vinwonders_driver.current_url
    vinwonders_page_title = vinwonders_driver.title

    print("Đã lấy HTML thành công.")
    print("Tiêu đề:", vinwonders_page_title)
    print("URL thực tế:", vinwonders_current_url)
    print("Độ dài HTML:", len(vinwonders_html))

finally:
    vinwonders_driver.quit()

Đã lấy HTML thành công.
Tiêu đề: VinWonders – Beyond Your Dreams, Full of Surprises
URL thực tế: https://vinwonders.com/en/
Độ dài HTML: 327328


In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from itertools import zip_longest


# ============================================================
# 1. KIỂM TRA HTML ĐÃ TỒN TẠI
# ============================================================

if "vinwonders_html" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_html. "
        "Hãy chạy Cell 1 trước."
    )


# ============================================================
# 2. PHÂN TÍCH HTML
# ============================================================

vinwonders_soup = BeautifulSoup(
    vinwonders_html,
    "lxml"
)


# ============================================================
# 3. TÌM WRAPPER CHỨA TÊN DANH MỤC
# ============================================================

vinwonders_name_wrapper = None

for wrapper in vinwonders_soup.select(
    '[id^="swiper-wrapper-"]'
):
    category_name_tags = wrapper.select(
        "div > div > div > h3"
    )

    if category_name_tags:
        vinwonders_name_wrapper = wrapper
        break


# ============================================================
# 4. TÌM WRAPPER CHỨA LINK DANH MỤC
# ============================================================

vinwonders_link_wrapper = None

for wrapper in vinwonders_soup.select(
    '[id^="swiper-wrapper-"]'
):
    category_link_tags = wrapper.select(
        "div > div > div > figure > a"
    )

    if category_link_tags:
        vinwonders_link_wrapper = wrapper
        break


# ============================================================
# 5. LẤY DANH SÁCH TÊN DANH MỤC
# ============================================================

vinwonders_category_names = []

if vinwonders_name_wrapper is not None:
    category_name_tags = vinwonders_name_wrapper.select(
        "div > div > div > h3"
    )

    for name_tag in category_name_tags:
        category_name = name_tag.get_text(
            " ",
            strip=True
        )

        vinwonders_category_names.append(
            category_name
        )


# ============================================================
# 6. LẤY DANH SÁCH LINK DANH MỤC
# ============================================================

vinwonders_category_links = []

if vinwonders_link_wrapper is not None:
    category_link_tags = vinwonders_link_wrapper.select(
        "div > div > div > figure > a"
    )

    for link_tag in category_link_tags:
        category_href = link_tag.get(
            "href",
            ""
        ).strip()

        if category_href:
            category_link = urljoin(
                vinwonders_current_url,
                category_href
            )
        else:
            category_link = ""

        vinwonders_category_links.append(
            category_link
        )


# ============================================================
# 7. GHÉP TÊN DANH MỤC VỚI LINK
# ============================================================

vinwonders_category_list = []

for category_name, category_link in zip_longest(
    vinwonders_category_names,
    vinwonders_category_links,
    fillvalue=""
):
    # Không có tên thì không lưu
    if not category_name:
        continue

    category_data = {
        "category_name": category_name,
        "category_link": category_link
    }

    vinwonders_category_list.append(
        category_data
    )


# ============================================================
# 8. TẠO DICTIONARY ĐỂ TRA CỨU
# ============================================================

vinwonders_category_dict = {
    category["category_name"]: category["category_link"]
    for category in vinwonders_category_list
}


# ============================================================
# 9. HIỂN THỊ KẾT QUẢ
# ============================================================

print(
    "Số tên danh mục:",
    len(vinwonders_category_names)
)

print(
    "Số link danh mục:",
    len(vinwonders_category_links)
)

print(
    "Số dữ liệu đã ghép:",
    len(vinwonders_category_list)
)

print("-" * 80)

for category_index, category in enumerate(
    vinwonders_category_list,
    start=1
):
    print(
        f"{category_index}. "
        f"Tên danh mục: "
        f"{category['category_name']}"
    )

    print(
        f"   Link danh mục: "
        f"{category['category_link']}"
    )

    print("-" * 80)

Số tên danh mục: 4
Số link danh mục: 4
Số dữ liệu đã ghép: 4
--------------------------------------------------------------------------------
1. Tên danh mục: WONDER DISCOVERY
   Link danh mục: https://vinwonders.com/en/news/phu-quoc-united-center-the-top-resort-entertainment-paradise-in-vietnam/
--------------------------------------------------------------------------------
2. Tên danh mục: WONDER CULTURAL EXPERIENCES
   Link danh mục: https://vinwonders.com/en/the-quintessence-of-vietnam/
--------------------------------------------------------------------------------
3. Tên danh mục: WONDER ENTERTAINMENT
   Link danh mục: 
--------------------------------------------------------------------------------
4. Tên danh mục: WONDER FESTIVALS
   Link danh mục: 
--------------------------------------------------------------------------------


In [12]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
import time


# ============================================================
# 1. KIỂM TRA DỮ LIỆU DANH MỤC
# ============================================================

if "vinwonders_category_list" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_category_list. "
        "Hãy chạy cell lấy danh mục trước."
    )


# ============================================================
# 2. CẤU HÌNH WEBDRIVER
# ============================================================

vinwonders_detail_options = Options()
vinwonders_detail_options.add_argument("--start-maximized")

if "HEADERS" in globals() and "User-Agent" in HEADERS:
    vinwonders_detail_options.add_argument(
        f"user-agent={HEADERS['User-Agent']}"
    )


# ============================================================
# 3. BIẾN LƯU TOÀN BỘ HTML CỦA CÁC TRANG DANH MỤC
# ============================================================

vinwonders_category_html_list = []

vinwonders_category_html_dict = {}


# ============================================================
# 4. KHỞI TẠO WEBDRIVER MỚI
# ============================================================

vinwonders_detail_driver = webdriver.Edge(
    options=vinwonders_detail_options
)


try:
    total_categories = len(vinwonders_category_list)

    for category_index, category in enumerate(
        vinwonders_category_list,
        start=1
    ):
        category_name = category.get(
            "category_name",
            ""
        ).strip()

        category_link = category.get(
            "category_link",
            ""
        ).strip()

        print(
            f"[{category_index}/{total_categories}] "
            f"Đang xử lý: {category_name}"
        )

        # Nếu danh mục không có link thì lưu HTML rỗng
        if not category_link:
            category_html_data = {
                "category_name": category_name,
                "category_link": "",
                "page_title": "",
                "current_url": "",
                "html": "",
                "status": "no_link",
                "error": ""
            }

            vinwonders_category_html_list.append(
                category_html_data
            )

            vinwonders_category_html_dict[
                category_name
            ] = category_html_data

            print("Không có link, bỏ qua truy cập.")
            print("-" * 80)

            continue

        try:
            # Truy cập trang danh mục
            vinwonders_detail_driver.get(
                category_link
            )

            # Chờ trang tải xong
            WebDriverWait(
                vinwonders_detail_driver,
                120
            ).until(
                lambda driver: (
                    driver.execute_script(
                        "return document.readyState"
                    ) == "complete"
                    and "just a moment"
                    not in driver.title.lower()
                    and len(driver.page_source) > 1000
                )
            )

            # Cuộn trang để tải nội dung lazy loading
            previous_height = 0

            for _ in range(12):
                vinwonders_detail_driver.execute_script(
                    "window.scrollTo(0, document.body.scrollHeight);"
                )

                time.sleep(1)

                current_height = (
                    vinwonders_detail_driver.execute_script(
                        "return document.body.scrollHeight"
                    )
                )

                if current_height == previous_height:
                    break

                previous_height = current_height

            # Lấy thông tin trang
            category_page_title = (
                vinwonders_detail_driver.title
            )

            category_current_url = (
                vinwonders_detail_driver.current_url
            )

            category_html = (
                vinwonders_detail_driver.page_source
            )

            # Lưu kết quả của từng trang
            category_html_data = {
                "category_name": category_name,
                "category_link": category_link,
                "page_title": category_page_title,
                "current_url": category_current_url,
                "html": category_html,
                "status": "success",
                "error": ""
            }

            vinwonders_category_html_list.append(
                category_html_data
            )

            vinwonders_category_html_dict[
                category_name
            ] = category_html_data

            print("Lấy HTML thành công.")
            print(
                "Độ dài HTML:",
                len(category_html)
            )

        except Exception as error:
            category_html_data = {
                "category_name": category_name,
                "category_link": category_link,
                "page_title": "",
                "current_url": "",
                "html": "",
                "status": "error",
                "error": str(error)
            }

            vinwonders_category_html_list.append(
                category_html_data
            )

            vinwonders_category_html_dict[
                category_name
            ] = category_html_data

            print(
                "Lỗi:",
                type(error).__name__,
                str(error)
            )

        print("-" * 80)

finally:
    vinwonders_detail_driver.quit()


# ============================================================
# 5. THỐNG KÊ KẾT QUẢ
# ============================================================

vinwonders_success_count = sum(
    item["status"] == "success"
    for item in vinwonders_category_html_list
)

vinwonders_no_link_count = sum(
    item["status"] == "no_link"
    for item in vinwonders_category_html_list
)

vinwonders_error_count = sum(
    item["status"] == "error"
    for item in vinwonders_category_html_list
)

print()
print("Hoàn thành lấy HTML các trang danh mục.")
print(
    "Tổng số danh mục:",
    len(vinwonders_category_html_list)
)
print(
    "Thành công:",
    vinwonders_success_count
)
print(
    "Không có link:",
    vinwonders_no_link_count
)
print(
    "Bị lỗi:",
    vinwonders_error_count
)

[1/4] Đang xử lý: WONDER DISCOVERY
Lấy HTML thành công.
Độ dài HTML: 277931
--------------------------------------------------------------------------------
[2/4] Đang xử lý: WONDER CULTURAL EXPERIENCES
Lấy HTML thành công.
Độ dài HTML: 221792
--------------------------------------------------------------------------------
[3/4] Đang xử lý: WONDER ENTERTAINMENT
Không có link, bỏ qua truy cập.
--------------------------------------------------------------------------------
[4/4] Đang xử lý: WONDER FESTIVALS
Không có link, bỏ qua truy cập.
--------------------------------------------------------------------------------

Hoàn thành lấy HTML các trang danh mục.
Tổng số danh mục: 4
Thành công: 2
Không có link: 2
Bị lỗi: 0


In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import json
import re
import unicodedata


# ============================================================
# 1. KIỂM TRA DỮ LIỆU HTML
# ============================================================

if "vinwonders_category_html_list" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_category_html_list. "
        "Hãy chạy cell cào HTML trước."
    )


# ============================================================
# 2. THƯ MỤC LƯU DATABASE VINWONDERS
# ============================================================

vinwonders_database_directory = Path("vinwonders/")

vinwonders_database_directory.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. HÀM TẠO TÊN FILE AN TOÀN
# ============================================================

def create_safe_filename(text):
    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = text.encode(
        "ascii",
        "ignore"
    ).decode("utf-8")

    text = text.lower().strip()

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")


# ============================================================
# 4. HÀM CHUẨN HÓA TEXT
# ============================================================

def clean_text(text):
    if not text:
        return ""

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


# ============================================================
# 5. HÀM LẤY URL ẢNH THẬT
# ============================================================

def get_image_url(image_tag, base_url):
    """
    Thứ tự ưu tiên:
    1. data-lazy-src
    2. data-src
    3. data-original
    4. data-lazy
    5. src

    Bỏ các ảnh placeholder dạng data:image.
    """

    if image_tag is None:
        return ""

    image_attributes = [
        "data-lazy-src",
        "data-src",
        "data-original",
        "data-lazy",
        "src"
    ]

    for attribute_name in image_attributes:
        image_url = image_tag.get(
            attribute_name,
            ""
        ).strip()

        if not image_url:
            continue

        if image_url.startswith("data:image"):
            continue

        return urljoin(
            base_url,
            image_url
        )

    # Trường hợp chỉ có srcset hoặc data-srcset
    srcset_value = (
        image_tag.get("data-lazy-srcset", "")
        or image_tag.get("data-srcset", "")
        or image_tag.get("srcset", "")
    ).strip()

    if srcset_value:
        first_image = (
            srcset_value
            .split(",")[0]
            .strip()
            .split(" ")[0]
        )

        if (
            first_image
            and not first_image.startswith("data:image")
        ):
            return urljoin(
                base_url,
                first_image
            )

    return ""


# ============================================================
# 6. HÀM KIỂM TRA ẢNH NỘI DUNG
# ============================================================

def is_content_image(image_url):
    """
    Loại bỏ ảnh giao diện không phù hợp với RAG:
    icon, logo, cờ, nút điều hướng, background giao diện...
    """

    if not image_url:
        return False

    image_url_lower = image_url.lower()

    excluded_keywords = [
        "favicon",
        "/icon",
        "/ic_",
        "logo",
        "flag-",
        "/flag/",
        "arrow",
        "filter-icon",
        "calendar",
        "ic_show",
        "lightbulb",
        "bg-bird",
        "loading",
        "spinner",
        "placeholder",
        "avatar",
        "social",
        "facebook",
        "instagram",
        "youtube",
        "tiktok"
    ]

    return not any(
        keyword in image_url_lower
        for keyword in excluded_keywords
    )


# ============================================================
# 7. HÀM LẤY ẢNH TRONG MỘT PHẦN TỬ
# ============================================================

def extract_images_from_element(
    element,
    base_url
):
    image_urls = []

    if element is None:
        return image_urls

    for image_tag in element.find_all("img"):
        image_url = get_image_url(
            image_tag,
            base_url
        )

        if (
            image_url
            and is_content_image(image_url)
            and image_url not in image_urls
        ):
            image_urls.append(
                image_url
            )

    # Một số ảnh được đặt trong thẻ source của picture
    for source_tag in element.find_all("source"):
        source_value = (
            source_tag.get("data-srcset", "")
            or source_tag.get("srcset", "")
        ).strip()

        if not source_value:
            continue

        source_url = (
            source_value
            .split(",")[0]
            .strip()
            .split(" ")[0]
        )

        if source_url:
            source_url = urljoin(
                base_url,
                source_url
            )

            if (
                is_content_image(source_url)
                and source_url not in image_urls
            ):
                image_urls.append(
                    source_url
                )

    return image_urls


# ============================================================
# 8. HÀM XÓA THÀNH PHẦN KHÔNG CẦN THIẾT
# ============================================================

def remove_unnecessary_elements(soup):
    removable_tags = [
        "script",
        "style",
        "noscript",
        "svg",
        "canvas",
        "iframe",
        "form",
        "input",
        "button",
        "select",
        "option",
        "nav",
        "footer"
    ]

    for tag_name in removable_tags:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    removable_selectors = [
        '[class*="cookie"]',
        '[id*="cookie"]',
        '[class*="popup"]',
        '[id*="popup"]',
        '[class*="modal"]',
        '[id*="modal"]',
        '[class*="breadcrumb"]',
        '[class*="social"]',
        '[class*="share"]',
        '[class*="pagination"]',
        '[class*="advertisement"]'
    ]

    for selector in removable_selectors:
        for element in soup.select(selector):
            element.decompose()

    return soup


# ============================================================
# 9. HÀM LẤY ẢNH ĐẠI DIỆN TRANG
# ============================================================

def extract_page_image(soup, base_url):
    og_image_tag = soup.select_one(
        'meta[property="og:image"]'
    )

    if og_image_tag:
        og_image_url = og_image_tag.get(
            "content",
            ""
        ).strip()

        if og_image_url:
            return urljoin(
                base_url,
                og_image_url
            )

    twitter_image_tag = soup.select_one(
        'meta[name="twitter:image"]'
    )

    if twitter_image_tag:
        twitter_image_url = twitter_image_tag.get(
            "content",
            ""
        ).strip()

        if twitter_image_url:
            return urljoin(
                base_url,
                twitter_image_url
            )

    return ""


# ============================================================
# 10. HÀM TÁCH TOPIC, TEXT VÀ ẢNH
# ============================================================

def extract_topics_with_images(
    html_content,
    source_url
):
    if not html_content:
        return {
            "page_image_url": "",
            "topics": []
        }

    soup = BeautifulSoup(
        html_content,
        "lxml"
    )

    page_image_url = extract_page_image(
        soup,
        source_url
    )

    soup = remove_unnecessary_elements(
        soup
    )

    # Với trang bài viết VinWonders, đây là vùng nội dung chính
    main_content = (
        soup.select_one("#main-single-content")
        or soup.select_one(".news-details")
        or soup.select_one(".g_content")
        or soup.find("main")
        or soup.find("article")
        or soup.body
        or soup
    )

    topic_headings = main_content.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4"
        ]
    )

    topics = []

    for topic_index, heading in enumerate(
        topic_headings,
        start=1
    ):
        topic_title = clean_text(
            heading.get_text(
                " ",
                strip=True
            )
        )

        if len(topic_title) < 3:
            continue

        topic_content_parts = []
        topic_image_urls = []

        # Tìm khối gần nhất chứa heading
        topic_container = heading.parent

        # Ảnh nằm trong chính container của heading
        topic_image_urls.extend(
            extract_images_from_element(
                topic_container,
                source_url
            )
        )

        # Duyệt các phần tử đứng sau heading
        # cho đến khi gặp heading cùng hoặc cao hơn
        current_heading_level = int(
            heading.name[1]
        )

        for sibling in heading.next_elements:
            if sibling is heading:
                continue

            if getattr(
                sibling,
                "name",
                None
            ) in [
                "h1",
                "h2",
                "h3",
                "h4"
            ]:
                next_heading_level = int(
                    sibling.name[1]
                )

                if (
                    next_heading_level
                    <= current_heading_level
                ):
                    break

            if getattr(
                sibling,
                "name",
                None
            ) in [
                "p",
                "li",
                "blockquote"
            ]:
                text_value = clean_text(
                    sibling.get_text(
                        " ",
                        strip=True
                    )
                )

                if (
                    len(text_value) >= 3
                    and text_value
                    not in topic_content_parts
                ):
                    topic_content_parts.append(
                        text_value
                    )

            if getattr(
                sibling,
                "name",
                None
            ) == "img":
                image_url = get_image_url(
                    sibling,
                    source_url
                )

                if (
                    image_url
                    and is_content_image(image_url)
                    and image_url
                    not in topic_image_urls
                ):
                    topic_image_urls.append(
                        image_url
                    )

        topic_content = "\n".join(
            topic_content_parts
        ).strip()

        # Không lưu heading thuộc menu nếu không có nội dung và ảnh
        if (
            not topic_content
            and not topic_image_urls
        ):
            continue

        topic_data = {
            "topic_id": len(topics) + 1,
            "topic_title": topic_title,
            "content": topic_content,
            "content_length": len(
                topic_content
            ),
            "image_urls": topic_image_urls
        }

        topics.append(
            topic_data
        )

    return {
        "page_image_url": page_image_url,
        "topics": topics
    }


# ============================================================
# 11. TÁCH VÀ LƯU TỪNG DANH MỤC VINWONDERS
# ============================================================

vinwonders_saved_database_list = []

for category_data in vinwonders_category_html_list:

    category_name = category_data.get(
        "category_name",
        ""
    ).strip()

    page_title = category_data.get(
        "page_title",
        ""
    ).strip()

    category_link = category_data.get(
        "category_link",
        ""
    ).strip()

    source_url = (
        category_data.get(
            "current_url",
            ""
        ).strip()
        or category_link
    )

    page_html = category_data.get(
        "html",
        ""
    )

    page_status = category_data.get(
        "status",
        ""
    )

    print(
        f"Đang xử lý: {category_name}"
    )

    if (
        page_status != "success"
        or not page_html
    ):
        print(
            "Không có HTML thành công, bỏ qua."
        )
        print("-" * 80)
        continue

    extracted_page_data = (
        extract_topics_with_images(
            html_content=page_html,
            source_url=source_url
        )
    )

    category_filename = create_safe_filename(
        category_name
    )

    if not category_filename:
        category_filename = (
            f"vinwonders_category_"
            f"{len(vinwonders_saved_database_list) + 1}"
        )

    category_file_path = (
        vinwonders_database_directory
        / f"{category_filename}.json"
    )

    category_database_data = {
        "data_source": "VinWonders",
        "data_type": "tourism_topics",
        "category_name": category_name,
        "page_title": page_title,
        "source_url": source_url,
        "page_image_url": extracted_page_data[
            "page_image_url"
        ],
        "total_topics": len(
            extracted_page_data["topics"]
        ),
        "topics": extracted_page_data[
            "topics"
        ]
    }

    with open(
        category_file_path,
        "w",
        encoding="utf-8"
    ) as json_file:
        json.dump(
            category_database_data,
            json_file,
            ensure_ascii=False,
            indent=2
        )

    total_topic_images = sum(
        len(topic["image_urls"])
        for topic in extracted_page_data[
            "topics"
        ]
    )

    saved_item = {
        "category_name": category_name,
        "file_path": str(
            category_file_path
        ),
        "total_topics": len(
            extracted_page_data["topics"]
        ),
        "total_topic_images": total_topic_images
    }

    vinwonders_saved_database_list.append(
        saved_item
    )

    print(
        "Đã lưu:",
        category_file_path
    )
    print(
        "Số topic:",
        saved_item["total_topics"]
    )
    print(
        "Số link ảnh:",
        saved_item["total_topic_images"]
    )
    print("-" * 80)


# ============================================================
# 12. THỐNG KÊ KẾT QUẢ
# ============================================================

print()
print("Hoàn thành lưu database VinWonders.")
print(
    "Số file đã tạo:",
    len(vinwonders_saved_database_list)
)

for saved_item in vinwonders_saved_database_list:
    print(
        f"- {saved_item['category_name']}"
    )
    print(
        f"  File: {saved_item['file_path']}"
    )
    print(
        f"  Topics: {saved_item['total_topics']}"
    )
    print(
        f"  Images: {saved_item['total_topic_images']}"
    )

Đang xử lý: WONDER DISCOVERY
Đã lưu: vinwonders\wonder_discovery.json
Số topic: 16
Số link ảnh: 512
--------------------------------------------------------------------------------
Đang xử lý: WONDER CULTURAL EXPERIENCES
Đã lưu: vinwonders\wonder_cultural_experiences.json
Số topic: 41
Số link ảnh: 67
--------------------------------------------------------------------------------
Đang xử lý: WONDER ENTERTAINMENT
Không có HTML thành công, bỏ qua.
--------------------------------------------------------------------------------
Đang xử lý: WONDER FESTIVALS
Không có HTML thành công, bỏ qua.
--------------------------------------------------------------------------------

Hoàn thành lưu database VinWonders.
Số file đã tạo: 2
- WONDER DISCOVERY
  File: vinwonders\wonder_discovery.json
  Topics: 16
  Images: 512
- WONDER CULTURAL EXPERIENCES
  File: vinwonders\wonder_cultural_experiences.json
  Topics: 41
  Images: 67


In [9]:
from lxml import html
from urllib.parse import urljoin


# ============================================================
# 1. KIỂM TRA HTML
# ============================================================

if "vinwonders_html" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_html. "
        "Hãy chạy cell cào HTML trước."
    )

if "vinwonders_current_url" not in globals():
    vinwonders_current_url = "https://vinwonders.com/en/"


# ============================================================
# 2. CHUYỂN HTML SANG CÂY LXML
# ============================================================

vinwonders_html_tree = html.fromstring(
    vinwonders_html
)


# ============================================================
# 3. TÌM ĐÚNG MENU DESTINATIONS
# ============================================================

# Chỉ tìm li menu có thẻ a trực tiếp mang nội dung "Destinations".
# Sau đó chỉ lấy các li địa điểm trực tiếp trong submenu đầu tiên.
vinwonders_destination_parent_xpath = (
    '//li['
        './a[normalize-space()="Destinations"]'
    ']'
    '/ul[contains(concat(" ", normalize-space(@class), " "), '
        '" sub-menu ")]'
    '/li['
        'contains(concat(" ", normalize-space(@class), " "), '
        '" parent-item-location ")'
    ']'
)

vinwonders_destination_parent_elements = (
    vinwonders_html_tree.xpath(
        vinwonders_destination_parent_xpath
    )
)


# ============================================================
# 4. LẤY TÊN VÀ LINK ĐỊA ĐIỂM
# ============================================================

vinwonders_destination_list = []
vinwonders_seen_names = set()

for parent_element in vinwonders_destination_parent_elements:

    # Thẻ a trực tiếp của nhóm địa điểm
    destination_name_elements = parent_element.xpath(
        "./a[1]"
    )

    if not destination_name_elements:
        continue

    destination_name_element = destination_name_elements[0]

    destination_name = " ".join(
        destination_name_element
        .text_content()
        .split()
    )

    if not destination_name:
        continue

    destination_url = ""

    # --------------------------------------------------------
    # Ưu tiên link trên chính tên địa điểm
    # Ví dụ: Nha Trang, Phu Quoc, Hoi An – Da Nang
    # --------------------------------------------------------

    parent_href = (
        destination_name_element.get(
            "href",
            ""
        )
        or ""
    ).strip()

    if (
        parent_href
        and parent_href != "#"
        and not parent_href.lower().startswith(
            "javascript:"
        )
    ):
        destination_url = urljoin(
            vinwonders_current_url,
            parent_href
        )

    # --------------------------------------------------------
    # Nếu thẻ tên chỉ có href="#", lấy link con đầu tiên
    # Ví dụ: Ha Noi, Ho Chi Minh City, Nghe An, Ha Tinh...
    # --------------------------------------------------------

    if not destination_url:
        child_link_elements = parent_element.xpath(
            './ul['
                'contains(concat(" ", normalize-space(@class), " "), '
                '" sub-menu ")'
            ']'
            '/li[contains('
                'concat(" ", normalize-space(@class), " "), '
                '" child-item-location "'
            ')]'
            '/a['
                '@href'
                ' and normalize-space(@href) != ""'
                ' and normalize-space(@href) != "#"'
                ' and not(starts-with('
                    'translate(@href, '
                    '"ABCDEFGHIJKLMNOPQRSTUVWXYZ", '
                    '"abcdefghijklmnopqrstuvwxyz"), '
                    '"javascript:"'
                '))'
            '][1]'
        )

        if child_link_elements:
            child_href = (
                child_link_elements[0].get(
                    "href",
                    ""
                )
                or ""
            ).strip()

            destination_url = urljoin(
                vinwonders_current_url,
                child_href
            )

    # --------------------------------------------------------
    # Loại trùng giữa menu desktop và mobile
    # --------------------------------------------------------

    destination_name_key = (
        destination_name
        .casefold()
        .strip()
    )

    if destination_name_key in vinwonders_seen_names:
        continue

    if not destination_url:
        continue

    vinwonders_seen_names.add(
        destination_name_key
    )

    vinwonders_destination_list.append(
        {
            "destination_name": destination_name,
            "destination_url": destination_url
        }
    )


# ============================================================
# 5. ĐÁNH ID
# ============================================================

for destination_id, destination in enumerate(
    vinwonders_destination_list,
    start=1
):
    destination["destination_id"] = destination_id


# ============================================================
# 6. TẠO DICTIONARY
# ============================================================

vinwonders_destination_dict = {
    destination["destination_name"]:
    destination["destination_url"]
    for destination in vinwonders_destination_list
}


# ============================================================
# 7. KIỂM TRA KẾT QUẢ
# ============================================================

print(
    "Số thẻ địa điểm tìm được:",
    len(vinwonders_destination_parent_elements)
)

print(
    "Tổng số địa điểm sau khi loại trùng:",
    len(vinwonders_destination_list)
)

print("-" * 80)

for destination in vinwonders_destination_list:
    print(
        f"{destination['destination_id']}. "
        f"Tên địa điểm: "
        f"{destination['destination_name']}"
    )

    print(
        f"   Link: "
        f"{destination['destination_url']}"
    )

    print("-" * 80)

Số thẻ địa điểm tìm được: 8
Tổng số địa điểm sau khi loại trùng: 8
--------------------------------------------------------------------------------
1. Tên địa điểm: Nha Trang
   Link: https://vinwonders.com/en/nha-trang-destination/
--------------------------------------------------------------------------------
2. Tên địa điểm: Phu Quoc
   Link: https://vinwonders.com/en/phu-quoc-destination/
--------------------------------------------------------------------------------
3. Tên địa điểm: Ha Noi
   Link: https://vinwonders.com/en/grand-world/
--------------------------------------------------------------------------------
4. Tên địa điểm: Ho Chi Minh City
   Link: https://vinwonders.com/en/grand-park/
--------------------------------------------------------------------------------
5. Tên địa điểm: Hoi An – Da Nang
   Link: https://vinwonders.com/en/nam-hoi-an_en/
--------------------------------------------------------------------------------
6. Tên địa điểm: Nghe An
   Link: https://

In [10]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException
import time


# ============================================================
# 1. KIỂM TRA BIẾN ĐẦU VÀO
# ============================================================

if "vinwonders_destination_list" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_destination_list. "
        "Hãy chạy cell lấy danh sách địa điểm trước."
    )

if not vinwonders_destination_list:
    raise ValueError("vinwonders_destination_list đang rỗng.")


# ============================================================
# 2. CÁC BIẾN LƯU KẾT QUẢ
# ============================================================

vinwonders_destination_html_list = []

vinwonders_destination_html_dict = {}

vinwonders_destination_raw_html_dict = {}

vinwonders_destination_success_list = []

vinwonders_destination_failed_list = []


# ============================================================
# 3. HÀM TẠO WEBDRIVER MỚI
# ============================================================

def create_vinwonders_webdriver():
    """
    Tạo một Edge WebDriver mới.
    Mỗi địa điểm sẽ sử dụng một WebDriver riêng.
    """

    edge_options = Options()

    edge_options.add_argument("--start-maximized")
    edge_options.add_argument("--disable-notifications")
    edge_options.add_argument("--disable-popup-blocking")
    edge_options.add_argument("--disable-extensions")
    edge_options.add_argument("--disable-gpu")

    if (
        "HEADERS" in globals()
        and isinstance(HEADERS, dict)
        and HEADERS.get("User-Agent")
    ):
        edge_options.add_argument(
            f"user-agent={HEADERS['User-Agent']}"
        )

    driver = webdriver.Edge(options=edge_options)

    # Không cho driver chờ quá lâu khi mở URL
    driver.set_page_load_timeout(45)

    return driver


# ============================================================
# 4. CRAWL TỪNG ĐỊA ĐIỂM
# ============================================================

total_destinations = len(vinwonders_destination_list)

for destination_index, destination in enumerate(
    vinwonders_destination_list,
    start=1
):
    destination_id = destination.get(
        "destination_id",
        destination_index
    )

    destination_name = destination.get(
        "destination_name",
        f"destination_{destination_index}"
    ).strip()

    destination_url = destination.get(
        "destination_url",
        ""
    ).strip()

    print(
        f"\n[{destination_index}/{total_destinations}] "
        f"Đang crawl: {destination_name}"
    )
    print("URL:", destination_url)

    destination_driver = None

    try:
        if not destination_url:
            raise ValueError("Địa điểm không có URL.")

        # Mỗi URL mở một WebDriver mới
        destination_driver = create_vinwonders_webdriver()

        try:
            destination_driver.get(destination_url)

        except TimeoutException:
            # Có thể trang chưa báo tải xong nhưng DOM đã có dữ liệu
            print(
                "Trang tải quá 45 giây, "
                "đang thử lấy HTML hiện có..."
            )

            destination_driver.execute_script(
                "window.stop();"
            )

        # Chỉ chờ DOM cơ bản xuất hiện, không cuộn trang
        WebDriverWait(
            destination_driver,
            20
        ).until(
            lambda driver: (
                driver.execute_script(
                    "return document.readyState"
                ) in ["interactive", "complete"]
                and len(driver.page_source) > 1000
            )
        )

        # Chờ ngắn để JavaScript render phần nội dung chính
        time.sleep(2)

        destination_page_title = destination_driver.title

        destination_current_url = (
            destination_driver.current_url
        )

        destination_full_html = (
            destination_driver.page_source
        )

        destination_html_length = len(
            destination_full_html
        )

        # Phát hiện trang Cloudflare cơ bản
        html_lower = destination_full_html.lower()
        title_lower = destination_page_title.lower()

        cloudflare_detected = (
            "just a moment" in title_lower
            or "cf-chl-" in html_lower
            or "checking your browser" in html_lower
        )

        if cloudflare_detected:
            destination_status = "cloudflare"
            destination_error = (
                "Trang có dấu hiệu đang hiển thị "
                "màn hình kiểm tra Cloudflare."
            )
        else:
            destination_status = "success"
            destination_error = ""

        destination_html_data = {
            "destination_id": destination_id,
            "destination_name": destination_name,
            "source_url": destination_url,
            "current_url": destination_current_url,
            "page_title": destination_page_title,
            "html": destination_full_html,
            "html_length": destination_html_length,
            "status": destination_status,
            "error": destination_error
        }

        vinwonders_destination_html_list.append(
            destination_html_data
        )

        vinwonders_destination_html_dict[
            destination_name
        ] = destination_html_data

        vinwonders_destination_raw_html_dict[
            destination_name
        ] = destination_full_html

        if destination_status == "success":
            vinwonders_destination_success_list.append(
                destination_html_data
            )

            print("Crawl thành công.")
        else:
            vinwonders_destination_failed_list.append(
                destination_html_data
            )

            print("Trang có dấu hiệu bị Cloudflare kiểm tra.")

        print("Tiêu đề:", destination_page_title)
        print("URL thực tế:", destination_current_url)
        print("Độ dài HTML:", destination_html_length)

    except Exception as error:
        destination_html_data = {
            "destination_id": destination_id,
            "destination_name": destination_name,
            "source_url": destination_url,
            "current_url": "",
            "page_title": "",
            "html": "",
            "html_length": 0,
            "status": "error",
            "error": str(error)
        }

        vinwonders_destination_html_list.append(
            destination_html_data
        )

        vinwonders_destination_html_dict[
            destination_name
        ] = destination_html_data

        vinwonders_destination_raw_html_dict[
            destination_name
        ] = ""

        vinwonders_destination_failed_list.append(
            destination_html_data
        )

        print("Crawl thất bại:", repr(error))

    finally:
        # Đóng WebDriver của địa điểm hiện tại
        if destination_driver is not None:
            try:
                destination_driver.quit()
                print("Đã đóng WebDriver.")
            except WebDriverException:
                pass

    # Nghỉ nhẹ trước khi mở trình duyệt mới
    time.sleep(2)


# ============================================================
# 5. TỔNG KẾT
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH CRAWL")
print("=" * 70)

print(
    "Tổng số địa điểm:",
    len(vinwonders_destination_list)
)

print(
    "Thành công:",
    len(vinwonders_destination_success_list)
)

print(
    "Thất bại hoặc bị chặn:",
    len(vinwonders_destination_failed_list)
)


[1/8] Đang crawl: Nha Trang
URL: https://vinwonders.com/en/nha-trang-destination/
Crawl thành công.
Tiêu đề: Điểm đến Nha Trang (Tỉnh thành) – EN – VinWonders
URL thực tế: https://vinwonders.com/en/nha-trang-destination/
Độ dài HTML: 332051
Đã đóng WebDriver.

[2/8] Đang crawl: Phu Quoc
URL: https://vinwonders.com/en/phu-quoc-destination/
Crawl thành công.
Tiêu đề: Điểm đến Phú Quốc_EN – VinWonders
URL thực tế: https://vinwonders.com/en/phu-quoc-destination/
Độ dài HTML: 266058
Đã đóng WebDriver.

[3/8] Đang crawl: Ha Noi
URL: https://vinwonders.com/en/grand-world/
Crawl thành công.
Tiêu đề: Grand World Ocean City East of Hanoi | Official Website
URL thực tế: https://vinwonders.com/en/grand-world/
Độ dài HTML: 278222
Đã đóng WebDriver.

[4/8] Đang crawl: Ho Chi Minh City
URL: https://vinwonders.com/en/grand-park/
Crawl thành công.
Tiêu đề: Grand Park Ho Chi Minh City | Official Website
URL thực tế: https://vinwonders.com/en/grand-park/
Độ dài HTML: 292088
Đã đóng WebDriver.

[5/8] Đan

In [11]:
from pathlib import Path
import re
import unicodedata


# ============================================================
# 1. KIỂM TRA DỮ LIỆU HTML ĐÃ CRAWL
# ============================================================

if "vinwonders_destination_html_list" not in globals():
    raise ValueError(
        "Chưa có biến vinwonders_destination_html_list. "
        "Hãy chạy cell crawl HTML trước."
    )

if not vinwonders_destination_html_list:
    raise ValueError(
        "vinwonders_destination_html_list đang rỗng."
    )


# ============================================================
# 2. TẠO THƯ MỤC html
# ============================================================

html_output_directory = Path("html")

html_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Thư mục lưu HTML:",
    html_output_directory.resolve()
)


# ============================================================
# 3. HÀM CHUYỂN TÊN ĐỊA ĐIỂM THÀNH TÊN FILE HỢP LỆ
# ============================================================

def create_safe_html_filename(destination_name):
    """
    Chuyển tên địa điểm thành tên file an toàn.

    Ví dụ:
    'Nha Trang'     -> 'nha-trang.html'
    'Phú Quốc'      -> 'phu-quoc.html'
    'Hội An/Đà Nẵng' -> 'hoi-an-da-nang.html'
    """

    destination_name = str(destination_name).strip()

    # Chuẩn hóa Unicode và loại bỏ dấu tiếng Việt
    normalized_name = unicodedata.normalize(
        "NFD",
        destination_name
    )

    ascii_name = "".join(
        character
        for character in normalized_name
        if unicodedata.category(character) != "Mn"
    )

    # Xử lý riêng chữ đ/Đ
    ascii_name = ascii_name.replace("đ", "d")
    ascii_name = ascii_name.replace("Đ", "D")

    # Chuyển về chữ thường
    safe_name = ascii_name.lower()

    # Thay ký tự không phải chữ hoặc số bằng dấu gạch ngang
    safe_name = re.sub(
        r"[^a-z0-9]+",
        "-",
        safe_name
    )

    # Xóa dấu gạch ngang ở đầu và cuối
    safe_name = safe_name.strip("-")

    if not safe_name:
        safe_name = "unknown-destination"

    return f"{safe_name}.html"


# ============================================================
# 4. XUẤT MỖI HTML THÀNH MỘT FILE RIÊNG
# ============================================================

vinwonders_exported_html_file_list = []

vinwonders_skipped_html_file_list = []

used_filename_set = set()


for destination_index, destination_data in enumerate(
    vinwonders_destination_html_list,
    start=1
):
    destination_name = destination_data.get(
        "destination_name",
        f"destination_{destination_index}"
    )

    destination_html = destination_data.get(
        "html",
        ""
    )

    destination_status = destination_data.get(
        "status",
        ""
    )

    # Không xuất file nếu không có HTML
    if not destination_html:
        vinwonders_skipped_html_file_list.append({
            "destination_name": destination_name,
            "status": destination_status,
            "reason": "Không có nội dung HTML."
        })

        print(
            f"[BỎ QUA] {destination_name}: "
            "không có nội dung HTML."
        )

        continue

    html_filename = create_safe_html_filename(
        destination_name
    )

    # Tránh ghi đè khi có hai địa điểm trùng tên
    original_filename = html_filename
    duplicate_number = 2

    while html_filename in used_filename_set:
        filename_stem = Path(original_filename).stem

        html_filename = (
            f"{filename_stem}-{duplicate_number}.html"
        )

        duplicate_number += 1

    used_filename_set.add(html_filename)

    html_file_path = (
        html_output_directory / html_filename
    )

    # utf-8-sig giúp một số trình soạn thảo Windows
    # nhận diện tiếng Việt chính xác hơn
    html_file_path.write_text(
        destination_html,
        encoding="utf-8-sig"
    )

    exported_file_data = {
        "destination_name": destination_name,
        "filename": html_filename,
        "file_path": str(html_file_path),
        "absolute_path": str(
            html_file_path.resolve()
        ),
        "html_length": len(destination_html),
        "status": destination_status
    }

    vinwonders_exported_html_file_list.append(
        exported_file_data
    )

    print(
        f"[ĐÃ LƯU] {destination_name} "
        f"-> {html_file_path}"
    )


# ============================================================
# 5. TỔNG KẾT
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH XUẤT FILE HTML")
print("=" * 70)

print(
    "Thư mục:",
    html_output_directory.resolve()
)

print(
    "Số file đã xuất:",
    len(vinwonders_exported_html_file_list)
)

print(
    "Số địa điểm bị bỏ qua:",
    len(vinwonders_skipped_html_file_list)
)

Thư mục lưu HTML: D:\vinuni\T013\data_crawl\html
[ĐÃ LƯU] Nha Trang -> html\nha-trang.html
[ĐÃ LƯU] Phu Quoc -> html\phu-quoc.html
[ĐÃ LƯU] Ha Noi -> html\ha-noi.html
[ĐÃ LƯU] Ho Chi Minh City -> html\ho-chi-minh-city.html
[ĐÃ LƯU] Hoi An – Da Nang -> html\hoi-an-da-nang.html
[ĐÃ LƯU] Nghe An -> html\nghe-an.html
[ĐÃ LƯU] Ha Tinh -> html\ha-tinh.html
[ĐÃ LƯU] Hai Phong -> html\hai-phong.html

HOÀN THÀNH XUẤT FILE HTML
Thư mục: D:\vinuni\T013\data_crawl\html
Số file đã xuất: 8
Số địa điểm bị bỏ qua: 0


In [7]:
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
import json
import re


# ============================================================
# 1. ĐỌC FILE HTML NHA TRANG
# ============================================================

nha_trang_html_path = Path(
    "html/nha-trang.html"
)

if not nha_trang_html_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: "
        f"{nha_trang_html_path.resolve()}"
    )

nha_trang_html = nha_trang_html_path.read_text(
    encoding="utf-8-sig",
    errors="ignore"
)

nha_trang_soup = BeautifulSoup(
    nha_trang_html,
    "lxml"
)


# ============================================================
# 2. CÁC HÀM HỖ TRỢ
# ============================================================

def clean_text(value):
    """
    Chuẩn hóa nội dung văn bản:
    - Loại khoảng trắng thừa
    - Nối các dòng thành một dòng
    """

    if value is None:
        return ""

    if hasattr(value, "get_text"):
        value = value.get_text(
            " ",
            strip=True
        )

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def get_image_url(image_tag):
    """
    Ưu tiên URL ảnh thật trong thuộc tính lazy load.
    """

    if image_tag is None:
        return ""

    return (
        image_tag.get("data-lazy-src")
        or image_tag.get("data-src")
        or image_tag.get("data-original")
        or image_tag.get("src")
        or ""
    ).strip()


def get_absolute_url(url, base_url):
    """
    Chuyển URL tương đối thành URL tuyệt đối.
    """

    if not url:
        return ""

    return urljoin(
        base_url,
        url
    )


# ============================================================
# 3. THÔNG TIN CHUNG CỦA TRANG
# ============================================================

canonical_tag = nha_trang_soup.select_one(
    'link[rel="canonical"]'
)

nha_trang_source_url = (
    canonical_tag.get(
        "href",
        ""
    ).strip()
    if canonical_tag
    else "https://vinwonders.com/en/nha-trang-destination/"
)

page_title_tag = nha_trang_soup.select_one(
    "title"
)

nha_trang_page_title = clean_text(
    page_title_tag
)


# ============================================================
# 4. PHẦN:
# Welcome to THE ISLAND OF ENDLESS EXPERIENCES!
# ============================================================

welcome_section = nha_trang_soup.select_one(
    ".address-having"
)

welcome_section_title = ""

welcome_experience_list = []

if welcome_section:

    welcome_section_title = clean_text(
        welcome_section.select_one(
            ".address-having_title"
        )
    )

    welcome_cards = welcome_section.select(
        ".swiper-slide .slide-card"
    )

    for card_index, card in enumerate(
        welcome_cards,
        start=1
    ):
        image_tag = card.select_one(
            ".slide-card_img"
        )

        title_tag = card.select_one(
            ".slide-card_content_title"
        )

        description_tag = card.select_one(
            ".slide-card_content_description"
        )

        welcome_experience_list.append({
            "item_id": card_index,

            "title": clean_text(
                title_tag
            ),

            "description": clean_text(
                description_tag
            ),

            "image_url": get_absolute_url(
                get_image_url(
                    image_tag
                ),
                nha_trang_source_url
            ),

            "image_alt": (
                image_tag.get(
                    "alt",
                    ""
                ).strip()
                if image_tag
                else ""
            )
        })


# ============================================================
# 5. PHẦN:
# Unique Experiences to Conquer at Vinpearl Nha Trang
# ============================================================

unique_experience_section = nha_trang_soup.select_one(
    ".experience"
)

unique_experience_section_title = ""

unique_experience_list = []

if unique_experience_section:

    unique_experience_section_title = clean_text(
        unique_experience_section.select_one(
            ".experience_title"
        )
    )

    unique_cards = unique_experience_section.select(
        ".experience-slide .slide-card"
    )

    for card_index, card in enumerate(
        unique_cards,
        start=1
    ):
        link_tag = card.select_one(
            "a[href]"
        )

        image_tag = card.select_one(
            ".slide-card_img"
        )

        title_tag = card.select_one(
            ".slide-card_content_title"
        )

        description_tag = card.select_one(
            ".slide-card_content_description"
        )

        location_tag = card.select_one(
            ".slide-card_content_add"
        )

        topic_url = (
            get_absolute_url(
                link_tag.get(
                    "href",
                    ""
                ),
                nha_trang_source_url
            )
            if link_tag
            else ""
        )

        unique_experience_list.append({
            "topic_id": card_index,

            "topic_group": (
                "unique_experiences"
            ),

            "title": clean_text(
                title_tag
            ),

            "description": clean_text(
                description_tag
            ),

            "location": clean_text(
                location_tag
            ),

            "image_url": get_absolute_url(
                get_image_url(
                    image_tag
                ),
                nha_trang_source_url
            ),

            "image_alt": (
                image_tag.get(
                    "alt",
                    ""
                ).strip()
                if image_tag
                else ""
            ),

            "topic_url": topic_url
        })


# ============================================================
# 6. PHẦN:
# EXPERIENCE JOURNEYS
# ============================================================

experience_journey_section = nha_trang_soup.select_one(
    ".trip.trip-mobile#news-and-experiences"
)

experience_journey_section_title = ""

experience_journey_list = []

if experience_journey_section:

    experience_journey_section_title = clean_text(
        experience_journey_section.select_one(
            ".trip_title"
        )
    )

    journey_cards = experience_journey_section.select(
        ".trip_card-list .trip_card-item"
    )

    for journey_index, card in enumerate(
        journey_cards,
        start=1
    ):
        # Ảnh chính của card
        image_tag = card.select_one(
            "img.trip_card-item-img"
        )

        # Tiêu đề
        title_tag = card.select_one(
            ".trip-card_card-desc_title"
        )

        # Mô tả
        description_tag = card.select_one(
            ".trip-card_card-desc_description"
        )

        # Link View more
        link_tag = card.select_one(
            ".trip-card_card-desc_button a[href]"
        )

        journey_url = (
            get_absolute_url(
                link_tag.get(
                    "href",
                    ""
                ),
                nha_trang_source_url
            )
            if link_tag
            else ""
        )

        journey_title = clean_text(
            title_tag
        )

        journey_description = clean_text(
            description_tag
        )

        journey_image_url = get_absolute_url(
            get_image_url(
                image_tag
            ),
            nha_trang_source_url
        )

        # Chỉ thêm card nếu có dữ liệu thực
        if (
            not journey_title
            and not journey_description
            and not journey_url
        ):
            continue

        experience_journey_list.append({
            "topic_id": journey_index,

            "topic_group": (
                "experience_journeys"
            ),

            "title": journey_title,

            "description": journey_description,

            "location": "",

            "image_url": journey_image_url,

            "image_alt": (
                image_tag.get(
                    "alt",
                    ""
                ).strip()
                if image_tag
                else ""
            ),

            "topic_url": journey_url
        })


# ============================================================
# 7. DANH SÁCH TẤT CẢ TOPIC CÓ LINK TRUY CẬP
# ============================================================

nha_trang_all_topic_list = (
    unique_experience_list
    + experience_journey_list
)


# ============================================================
# 8. JSON TỔNG CỦA TRANG NHA TRANG
# ============================================================

nha_trang_destination_data = {
    "document_type": "destination",

    "destination_name": "Nha Trang",

    "language": "en",

    "source_url": nha_trang_source_url,

    "page_title": nha_trang_page_title,

    "statistics": {
        "welcome_experience_count": len(
            welcome_experience_list
        ),

        "unique_experience_count": len(
            unique_experience_list
        ),

        "experience_journey_count": len(
            experience_journey_list
        ),

        "total_linked_topic_count": len(
            nha_trang_all_topic_list
        )
    },

    "sections": {
        "welcome_experiences": {
            "section_title": (
                welcome_section_title
            ),

            "items": (
                welcome_experience_list
            )
        },

        "unique_experiences": {
            "section_title": (
                unique_experience_section_title
            ),

            "items": (
                unique_experience_list
            )
        },

        "experience_journeys": {
            "section_title": (
                experience_journey_section_title
            ),

            "items": (
                experience_journey_list
            )
        }
    }
}


# ============================================================
# 9. LƯU JSON
# ============================================================

json_output_directory = Path(
    "json"
)

json_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

nha_trang_destination_json_path = (
    json_output_directory
    / "nha-trang-destination.json"
)

nha_trang_destination_json_path.write_text(
    json.dumps(
        nha_trang_destination_data,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 10. KIỂM TRA
# ============================================================

print("=" * 70)
print("KẾT QUẢ TRÍCH XUẤT TRANG NHA TRANG")
print("=" * 70)

print(
    "Tiêu đề trang:",
    nha_trang_page_title
)

print("\n1. WELCOME EXPERIENCES")

print(
    "Tiêu đề phần:",
    welcome_section_title
)

print(
    "Số nội dung:",
    len(welcome_experience_list)
)

print("\n2. UNIQUE EXPERIENCES")

print(
    "Tiêu đề phần:",
    unique_experience_section_title
)

print(
    "Số topic:",
    len(unique_experience_list)
)

print("\n3. EXPERIENCE JOURNEYS")

print(
    "Tiêu đề phần:",
    experience_journey_section_title
)

print(
    "Số topic:",
    len(experience_journey_list)
)

print(
    "\nTổng topic có thể tiếp tục crawl:",
    len(nha_trang_all_topic_list)
)

print(
    "\nĐã lưu JSON:",
    nha_trang_destination_json_path.resolve()
)


# ============================================================
# 11. IN DANH SÁCH EXPERIENCE JOURNEYS
# ============================================================

print("\n" + "=" * 70)
print("DANH SÁCH EXPERIENCE JOURNEYS")
print("=" * 70)

for journey in experience_journey_list:
    print("-" * 70)

    print(
        "ID:",
        journey["topic_id"]
    )

    print(
        "Tên:",
        journey["title"]
    )

    print(
        "Mô tả:",
        journey["description"]
    )

    print(
        "Ảnh:",
        journey["image_url"]
    )

    print(
        "Link:",
        journey["topic_url"]
    )

KẾT QUẢ TRÍCH XUẤT TRANG NHA TRANG
Tiêu đề trang: Điểm đến Nha Trang (Tỉnh thành) – EN – VinWonders

1. WELCOME EXPERIENCES
Tiêu đề phần: Welcome to THE ISLAND OF ENDLESS EXPERIENCES!
Số nội dung: 4

2. UNIQUE EXPERIENCES
Tiêu đề phần: Unique Experiences to Conquer at Vinpearl Nha Trang
Số topic: 13

3. EXPERIENCE JOURNEYS
Tiêu đề phần: EXPERIENCE JOURNEYS
Số topic: 3

Tổng topic có thể tiếp tục crawl: 16

Đã lưu JSON: D:\vinuni\T013\data_crawl\json\nha-trang-destination.json

DANH SÁCH EXPERIENCE JOURNEYS
----------------------------------------------------------------------
ID: 1
Tên: Explore Hòn Tre in 2 Days 1 Night – The Call of the Sea Paradise
Mô tả: When in Nha Trang, you must visit Hon Tre! We recommend a 2-day-1-night journey to fully explore this paradise in Nha Trang. Enjoy a luxurious stay at Vinpearl Resort Nha Trang, relax with authentic Korean-style saunas at Aquafield, discover the thrills of VinWonders, and experience the excitement at Vinpearl Harbour!
Ảnh: https://s

In [8]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException
from pathlib import Path
from urllib.parse import urljoin
import unicodedata
import re


# ============================================================
# 1. ĐỌC FILE HTML TRANG NHA TRANG
# ============================================================

nha_trang_html_path = Path(
    "html/nha-trang.html"
)

if not nha_trang_html_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: "
        f"{nha_trang_html_path.resolve()}"
    )

nha_trang_html = nha_trang_html_path.read_text(
    encoding="utf-8-sig",
    errors="ignore"
)

nha_trang_soup = BeautifulSoup(
    nha_trang_html,
    "lxml"
)


# ============================================================
# 2. XÁC ĐỊNH URL GỐC CỦA TRANG
# ============================================================

canonical_tag = nha_trang_soup.select_one(
    'link[rel="canonical"]'
)

nha_trang_source_url = (
    canonical_tag.get("href", "").strip()
    if canonical_tag
    else "https://vinwonders.com/en/nha-trang-destination/"
)


# ============================================================
# 3. CÁC HÀM HỖ TRỢ
# ============================================================

def clean_text(value):
    """
    Chuẩn hóa khoảng trắng trong nội dung.
    """

    if value is None:
        return ""

    if hasattr(value, "get_text"):
        value = value.get_text(
            " ",
            strip=True
        )

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def make_absolute_url(url, base_url):
    """
    Chuyển URL tương đối thành URL tuyệt đối.
    """

    url = str(url or "").strip()

    if not url:
        return ""

    return urljoin(
        base_url,
        url
    )


def get_image_url(image_tag, base_url):
    """
    Lấy URL ảnh, ưu tiên thuộc tính ảnh lazy-load.
    """

    if image_tag is None:
        return ""

    image_url = (
        image_tag.get("data-lazy-src")
        or image_tag.get("data-src")
        or image_tag.get("data-original")
        or image_tag.get("src")
        or ""
    )

    return make_absolute_url(
        image_url,
        base_url
    )


def create_topic_slug(text):
    """
    Chuyển tiêu đề thành tên file an toàn.

    Ví dụ:
    'Discover Nha Trang' -> 'discover-nha-trang'
    """

    text = str(text or "").strip()

    normalized_text = unicodedata.normalize(
        "NFD",
        text
    )

    ascii_text = "".join(
        character
        for character in normalized_text
        if unicodedata.category(character) != "Mn"
    )

    ascii_text = (
        ascii_text
        .replace("đ", "d")
        .replace("Đ", "D")
        .lower()
    )

    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        ascii_text
    ).strip("-")

    return slug or "unknown-topic"


# ============================================================
# 4. LẤY NHÓM UNIQUE EXPERIENCES
# ============================================================

unique_experience_list = []

unique_experience_section = nha_trang_soup.select_one(
    ".experience"
)

if unique_experience_section is not None:

    # Lấy tất cả card trong vùng experience
    unique_experience_cards = (
        unique_experience_section.select(
            ".slide-card"
        )
    )

    for card_index, card in enumerate(
        unique_experience_cards,
        start=1
    ):
        link_tag = card.select_one(
            "a[href]"
        )

        image_tag = card.select_one(
            "img"
        )

        title_tag = (
            card.select_one(
                ".slide-card_content_title"
            )
            or card.select_one(
                ".slide-card-content-title"
            )
        )

        description_tag = (
            card.select_one(
                ".slide-card_content_description"
            )
            or card.select_one(
                ".slide-card-content-description"
            )
        )

        location_tag = (
            card.select_one(
                ".slide-card_content_add"
            )
            or card.select_one(
                ".slide-card-content-add"
            )
        )

        topic_url = (
            make_absolute_url(
                link_tag.get("href", ""),
                nha_trang_source_url
            )
            if link_tag
            else ""
        )

        topic_title = clean_text(
            title_tag
        )

        # Chỉ lưu card có tiêu đề hoặc URL
        if not topic_title and not topic_url:
            continue

        unique_experience_list.append({
            "topic_id": (
                f"unique-experience-{card_index}"
            ),
            "topic_group": "unique_experiences",
            "topic_group_title": (
                "Unique Experiences to Conquer "
                "at Vinpearl Nha Trang"
            ),
            "title": topic_title,
            "description": clean_text(
                description_tag
            ),
            "location": clean_text(
                location_tag
            ),
            "image_url": get_image_url(
                image_tag,
                nha_trang_source_url
            ),
            "image_alt": (
                image_tag.get(
                    "alt",
                    ""
                ).strip()
                if image_tag
                else ""
            ),
            "topic_url": topic_url
        })


# ============================================================
# 5. LẤY NHÓM EXPERIENCE JOURNEYS
# ============================================================

experience_journey_list = []

experience_journey_section = nha_trang_soup.select_one(
    "#news-and-experiences"
)

if experience_journey_section is not None:

    # Theo XPath:
    # //*[@id="news-and-experiences"]/div/div/div
    #
    # Tuy nhiên cấu trúc có thể có nhiều div lồng nhau.
    # Vì vậy tìm các khối chứa link truy cập.

    journey_link_tags = experience_journey_section.select(
        "a[href]"
    )

    processed_journey_cards = set()

    for journey_link_tag in journey_link_tags:

        # Tìm card gần nhất chứa cả ảnh và nội dung
        journey_card = journey_link_tag.find_parent(
            lambda tag: (
                tag.name == "div"
                and tag.find("img") is not None
                and len(
                    tag.find_all(
                        "p",
                        recursive=True
                    )
                ) >= 2
            )
        )

        if journey_card is None:
            continue

        # Tránh xử lý một card nhiều lần
        journey_card_identity = id(
            journey_card
        )

        if journey_card_identity in processed_journey_cards:
            continue

        processed_journey_cards.add(
            journey_card_identity
        )

        paragraph_tags = journey_card.find_all(
            "p",
            recursive=True
        )

        image_tag = journey_card.find(
            "img"
        )

        # Theo XPath người dùng cung cấp:
        # p[1] = tiêu đề
        # p[2] = mô tả
        # p[3]/a = link truy cập

        title_tag = (
            paragraph_tags[0]
            if len(paragraph_tags) >= 1
            else None
        )

        description_tag = (
            paragraph_tags[1]
            if len(paragraph_tags) >= 2
            else None
        )

        access_link_tag = None

        if len(paragraph_tags) >= 3:
            access_link_tag = paragraph_tags[2].select_one(
                "a[href]"
            )

        # Fallback nếu link không nằm đúng paragraph thứ ba
        if access_link_tag is None:
            access_link_tag = journey_link_tag

        journey_url = (
            make_absolute_url(
                access_link_tag.get(
                    "href",
                    ""
                ),
                nha_trang_source_url
            )
            if access_link_tag
            else ""
        )

        journey_title = clean_text(
            title_tag
        )

        journey_description = clean_text(
            description_tag
        )

        # Bỏ card không có nội dung hữu ích
        if not journey_title and not journey_url:
            continue

        experience_journey_list.append({
            "topic_id": (
                f"experience-journey-"
                f"{len(experience_journey_list) + 1}"
            ),
            "topic_group": "experience_journeys",
            "topic_group_title": "EXPERIENCE JOURNEYS",
            "title": journey_title,
            "description": journey_description,
            "location": "",
            "image_url": get_image_url(
                image_tag,
                nha_trang_source_url
            ),
            "image_alt": (
                image_tag.get(
                    "alt",
                    ""
                ).strip()
                if image_tag
                else ""
            ),
            "topic_url": journey_url
        })


# ============================================================
# 6. GỘP HAI NHÓM TOPIC
# ============================================================

nha_trang_all_topic_list = (
    unique_experience_list
    + experience_journey_list
)


# ============================================================
# 7. LOẠI URL TRÙNG VÀ URL RỖNG
# ============================================================

nha_trang_unique_topic_list = []

seen_topic_urls = set()

for topic in nha_trang_all_topic_list:
    topic_url = topic.get(
        "topic_url",
        ""
    ).strip()

    if not topic_url:
        continue

    # Bỏ anchor để tránh coi cùng trang là nhiều URL
    normalized_topic_url = topic_url.split(
        "#",
        1
    )[0].rstrip("/")

    if normalized_topic_url in seen_topic_urls:
        continue

    seen_topic_urls.add(
        normalized_topic_url
    )

    nha_trang_unique_topic_list.append(
        topic
    )


# ============================================================
# 8. KIỂM TRA KẾT QUẢ TRÍCH XUẤT
# ============================================================

print("=" * 70)
print("KẾT QUẢ TRÍCH XUẤT TOPIC")
print("=" * 70)

print(
    "Unique Experiences:",
    len(unique_experience_list)
)

print(
    "Experience Journeys:",
    len(experience_journey_list)
)

print(
    "Tổng trước khi loại trùng:",
    len(nha_trang_all_topic_list)
)

print(
    "Tổng topic hợp lệ sau khi loại trùng:",
    len(nha_trang_unique_topic_list)
)

for topic_index, topic in enumerate(
    nha_trang_unique_topic_list,
    start=1
):
    print("-" * 70)

    print(
        topic_index,
        "|",
        topic["topic_group"]
    )

    print(
        "Tiêu đề:",
        topic["title"]
    )

    print(
        "URL:",
        topic["topic_url"]
    )


# ============================================================
# 9. TẠO CÁC THƯ MỤC LƯU HTML
# ============================================================

nha_trang_topic_html_directory = Path(
    "html/nha-trang-topics"
)

unique_experience_html_directory = (
    nha_trang_topic_html_directory
    / "unique-experiences"
)

experience_journey_html_directory = (
    nha_trang_topic_html_directory
    / "experience-journeys"
)

unique_experience_html_directory.mkdir(
    parents=True,
    exist_ok=True
)

experience_journey_html_directory.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 10. CÁC BIẾN LƯU KẾT QUẢ CRAWL
# ============================================================

nha_trang_topic_crawl_results = []

nha_trang_topic_success_list = []

nha_trang_topic_failed_list = []

nha_trang_topic_raw_html_dict = {}


# ============================================================
# 11. HÀM TẠO WEBDRIVER MỚI
# ============================================================

def create_topic_webdriver():
    """
    Mỗi topic sử dụng một Edge WebDriver riêng.
    """

    edge_options = Options()

    edge_options.add_argument(
        "--start-maximized"
    )

    edge_options.add_argument(
        "--disable-notifications"
    )

    edge_options.add_argument(
        "--disable-popup-blocking"
    )

    edge_options.add_argument(
        "--disable-extensions"
    )

    edge_options.add_argument(
        "--disable-gpu"
    )

    if (
        "HEADERS" in globals()
        and isinstance(
            HEADERS,
            dict
        )
        and HEADERS.get(
            "User-Agent"
        )
    ):
        edge_options.add_argument(
            f"user-agent="
            f"{HEADERS['User-Agent']}"
        )

    driver = webdriver.Edge(
        options=edge_options
    )

    # Giới hạn tối đa cho việc mở URL.
    # Không phải chờ cố định.
    driver.set_page_load_timeout(
        45
    )

    return driver


# ============================================================
# 12. CRAWL TỪNG TOPIC
# ============================================================

total_topics = len(
    nha_trang_unique_topic_list
)

for topic_index, topic in enumerate(
    nha_trang_unique_topic_list,
    start=1
):
    topic_title = topic.get(
        "title",
        f"topic-{topic_index}"
    ).strip()

    topic_url = topic.get(
        "topic_url",
        ""
    ).strip()

    topic_group = topic.get(
        "topic_group",
        "unknown"
    )

    print(
        f"\n[{topic_index}/{total_topics}] "
        f"Đang crawl: {topic_title}"
    )

    print(
        "Nhóm:",
        topic_group
    )

    print(
        "URL:",
        topic_url
    )

    topic_driver = None

    try:
        if not topic_url:
            raise ValueError(
                "Topic không có URL."
            )

        # Mỗi URL dùng một WebDriver riêng
        topic_driver = create_topic_webdriver()

        try:
            topic_driver.get(
                topic_url
            )

        except TimeoutException:
            print(
                "Trang tải quá 45 giây, "
                "đang lấy HTML hiện có..."
            )

            try:
                topic_driver.execute_script(
                    "window.stop();"
                )

            except Exception:
                pass

        # Chỉ chờ DOM cơ bản.
        # Khi đạt điều kiện sẽ thoát ngay,
        # không bắt buộc chờ đủ 20 giây.
        try:
            WebDriverWait(
                topic_driver,
                20,
                poll_frequency=0.5
            ).until(
                lambda driver: (
                    driver.execute_script(
                        "return document.readyState"
                    )
                    in [
                        "interactive",
                        "complete"
                    ]
                    and len(
                        driver.page_source
                    ) > 1000
                )
            )

        except TimeoutException:
            print(
                "DOM chưa đạt điều kiện "
                "sau thời gian tối đa. "
                "Đang lấy HTML hiện có..."
            )

        # ====================================================
        # LẤY HTML NGAY
        # ====================================================

        topic_page_title = (
            topic_driver.title
        )

        topic_current_url = (
            topic_driver.current_url
        )

        topic_full_html = (
            topic_driver.page_source
        )

        topic_html_length = len(
            topic_full_html
        )

        # ====================================================
        # KIỂM TRA CLOUDFLARE
        # ====================================================

        html_lower = topic_full_html.lower()

        title_lower = (
            topic_page_title.lower()
        )

        cloudflare_detected = (
            "just a moment" in title_lower
            or "checking your browser" in html_lower
            or "cf-chl-" in html_lower
            or "verify you are human" in html_lower
        )

        if cloudflare_detected:
            topic_status = "cloudflare"

            topic_error = (
                "Trang có dấu hiệu đang hiển thị "
                "màn hình kiểm tra Cloudflare."
            )

        elif topic_html_length <= 1000:
            topic_status = "incomplete"

            topic_error = (
                "HTML có độ dài quá ngắn, "
                "có thể trang chưa tải đủ."
            )

        else:
            topic_status = "success"
            topic_error = ""

        # ====================================================
        # CHỌN THƯ MỤC THEO NHÓM
        # ====================================================

        if topic_group == "experience_journeys":
            topic_output_directory = (
                experience_journey_html_directory
            )

        else:
            topic_output_directory = (
                unique_experience_html_directory
            )

        # ====================================================
        # TẠO VÀ LƯU FILE HTML
        # ====================================================

        topic_slug = create_topic_slug(
            topic_title
        )

        topic_html_filename = (
            f"{topic_index:02d}-"
            f"{topic_slug}.html"
        )

        topic_html_path = (
            topic_output_directory
            / topic_html_filename
        )

        topic_html_path.write_text(
            topic_full_html,
            encoding="utf-8-sig"
        )

        # ====================================================
        # LƯU KẾT QUẢ
        # ====================================================

        topic_html_data = {
            **topic,
            "topic_index": topic_index,
            "page_title": topic_page_title,
            "source_url": topic_url,
            "current_url": topic_current_url,
            "html_filename": (
                topic_html_filename
            ),
            "html_path": str(
                topic_html_path
            ),
            "html_length": (
                topic_html_length
            ),
            "crawl_status": (
                topic_status
            ),
            "crawl_error": (
                topic_error
            )
        }

        nha_trang_topic_crawl_results.append(
            topic_html_data
        )

        # Dùng URL làm key để tránh trùng tiêu đề
        nha_trang_topic_raw_html_dict[
            topic_url
        ] = topic_full_html

        if topic_status == "success":
            nha_trang_topic_success_list.append(
                topic_html_data
            )

            print(
                "Crawl thành công."
            )

        else:
            nha_trang_topic_failed_list.append(
                topic_html_data
            )

            print(
                "Trang chưa đạt trạng thái thành công:",
                topic_status
            )

        print(
            "Tiêu đề trang:",
            topic_page_title
        )

        print(
            "URL thực tế:",
            topic_current_url
        )

        print(
            "Độ dài HTML:",
            topic_html_length
        )

        print(
            "Đã lưu:",
            topic_html_path
        )

    except Exception as error:
        topic_html_data = {
            **topic,
            "topic_index": topic_index,
            "page_title": "",
            "source_url": topic_url,
            "current_url": "",
            "html_filename": "",
            "html_path": "",
            "html_length": 0,
            "crawl_status": "error",
            "crawl_error": str(
                error
            )
        }

        nha_trang_topic_crawl_results.append(
            topic_html_data
        )

        nha_trang_topic_failed_list.append(
            topic_html_data
        )

        nha_trang_topic_raw_html_dict[
            topic_url
        ] = ""

        print(
            "Crawl thất bại:",
            repr(error)
        )

    finally:
        # Đóng WebDriver sau mỗi topic
        if topic_driver is not None:
            try:
                topic_driver.quit()

                print(
                    "Đã đóng WebDriver."
                )

            except WebDriverException:
                pass


# ============================================================
# 13. TỔNG KẾT
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH CRAWL TẤT CẢ TOPIC")
print("=" * 70)

print(
    "Unique Experiences:",
    len(unique_experience_list)
)

print(
    "Experience Journeys:",
    len(experience_journey_list)
)

print(
    "Tổng URL sau khi loại trùng:",
    len(nha_trang_unique_topic_list)
)

print(
    "Crawl thành công:",
    len(nha_trang_topic_success_list)
)

print(
    "Thất bại, thiếu hoặc bị chặn:",
    len(nha_trang_topic_failed_list)
)

print(
    "Thư mục Unique Experiences:",
    unique_experience_html_directory.resolve()
)

print(
    "Thư mục Experience Journeys:",
    experience_journey_html_directory.resolve()
)

KẾT QUẢ TRÍCH XUẤT TOPIC
Unique Experiences: 13
Experience Journeys: 3
Tổng trước khi loại trùng: 16
Tổng topic hợp lệ sau khi loại trùng: 15
----------------------------------------------------------------------
1 | unique_experiences
Tiêu đề: Witness the Full Tata Show
URL: https://vinwonders.com/vi/tata-show/
----------------------------------------------------------------------
2 | unique_experiences
Tiêu đề: Capture Moments at The World Garden
URL: https://vinwonders.com/vi/wonderpedia/news/tham-quan-kings-garden-nha-trang-vuon-thu-doc-dao-nhat-hon-tre/
----------------------------------------------------------------------
3 | unique_experiences
Tiêu đề: Immerse yourself in The Underwater World
URL: https://vinwonders.com/vi/wonderpedia/news/thuy-cung-vinpearl-nha-trang-gia-ve-huong-dan-tham-quan/
----------------------------------------------------------------------
4 | unique_experiences
Tiêu đề: Conquer 15 rides at Festive Hill
URL: https://vinwonders.com/vi/wonderpedia/news/tr

In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
import re
import unicodedata


# ============================================================
# 1. ĐỌC FILE HTML NHA TRANG
# ============================================================

nha_trang_html_path = Path("html/nha-trang.html")

if not nha_trang_html_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {nha_trang_html_path.resolve()}"
    )

nha_trang_html = nha_trang_html_path.read_text(
    encoding="utf-8-sig",
    errors="ignore"
)

nha_trang_soup = BeautifulSoup(
    nha_trang_html,
    "lxml"
)


# ============================================================
# 2. URL GỐC
# ============================================================

canonical_tag = nha_trang_soup.select_one(
    'link[rel="canonical"]'
)

nha_trang_source_url = (
    canonical_tag.get("href", "").strip()
    if canonical_tag
    else "https://vinwonders.com/en/nha-trang-destination/"
)


# ============================================================
# 3. HÀM HỖ TRỢ
# ============================================================

def clean_text(tag_or_text):
    if tag_or_text is None:
        return ""

    if hasattr(tag_or_text, "get_text"):
        tag_or_text = tag_or_text.get_text(
            " ",
            strip=True
        )

    return re.sub(
        r"\s+",
        " ",
        str(tag_or_text)
    ).strip()


def make_absolute_url(url, base_url):
    url = str(url or "").strip()

    if not url:
        return ""

    return urljoin(
        base_url,
        url
    )


def get_image_url(image_tag, base_url):
    if image_tag is None:
        return ""

    image_url = (
        image_tag.get("data-lazy-src")
        or image_tag.get("data-src")
        or image_tag.get("data-original")
        or image_tag.get("src")
        or ""
    )

    return make_absolute_url(
        image_url,
        base_url
    )


def create_safe_slug(text):
    text = str(text or "").strip()

    normalized_text = unicodedata.normalize(
        "NFD",
        text
    )

    ascii_text = "".join(
        character
        for character in normalized_text
        if unicodedata.category(character) != "Mn"
    )

    ascii_text = (
        ascii_text
        .replace("đ", "d")
        .replace("Đ", "D")
        .lower()
    )

    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        ascii_text
    ).strip("-")

    return slug or "unknown-topic"


# ============================================================
# 4. LẤY UNIQUE EXPERIENCES
# ============================================================

unique_experience_list = []

unique_section = nha_trang_soup.select_one(
    ".experience"
)

if unique_section:
    unique_cards = unique_section.select(
        ".slide-card"
    )

    for card_index, card in enumerate(
        unique_cards,
        start=1
    ):
        link_tag = card.select_one(
            "a[href]"
        )

        image_tag = card.select_one(
            "img"
        )

        title_tag = (
            card.select_one(
                ".slide-card_content_title"
            )
            or card.select_one(
                ".slide-card-content-title"
            )
        )

        description_tag = (
            card.select_one(
                ".slide-card_content_description"
            )
            or card.select_one(
                ".slide-card-content-description"
            )
        )

        topic_url = (
            make_absolute_url(
                link_tag.get("href", ""),
                nha_trang_source_url
            )
            if link_tag
            else ""
        )

        topic_title = clean_text(
            title_tag
        )

        if not topic_title and not topic_url:
            continue

        unique_experience_list.append({
            "topic_id": f"unique-{card_index:02d}",
            "topic_group": "unique_experiences",
            "topic_group_title": (
                "Unique Experiences to Conquer "
                "at Vinpearl Nha Trang"
            ),
            "title": topic_title,
            "description": clean_text(
                description_tag
            ),
            "image_url": get_image_url(
                image_tag,
                nha_trang_source_url
            ),
            "image_alt": (
                image_tag.get("alt", "").strip()
                if image_tag
                else ""
            ),
            "topic_url": topic_url
        })


# ============================================================
# 5. LẤY EXPERIENCE JOURNEYS
# ============================================================

experience_journey_list = []

journey_section = nha_trang_soup.select_one(
    "#news-and-experiences"
)

if journey_section:
    # Theo XPath người dùng cung cấp:
    # //*[@id="news-and-experiences"]/div/div/div
    #
    # Chọn các div trực tiếp chứa ảnh và khối nội dung.
    journey_cards = journey_section.select(
        ":scope > div > div > div"
    )

    # Fallback nếu parser không hỗ trợ cấu trúc trên
    if not journey_cards:
        journey_cards = journey_section.find_all(
            "div",
            recursive=True
        )

    processed_cards = set()

    for card in journey_cards:
        image_tag = card.find(
            "img",
            recursive=False
        )

        # Có những trang ảnh nằm sâu hơn một cấp
        if image_tag is None:
            image_tag = card.find(
                "img"
            )

        direct_divs = card.find_all(
            "div",
            recursive=False
        )

        content_div = None

        for direct_div in direct_divs:
            paragraphs = direct_div.find_all(
                "p",
                recursive=False
            )

            if len(paragraphs) >= 2:
                content_div = direct_div
                break

        if content_div is None:
            paragraphs = card.find_all(
                "p"
            )
        else:
            paragraphs = content_div.find_all(
                "p",
                recursive=False
            )

        if len(paragraphs) < 2:
            continue

        title_tag = paragraphs[0]
        description_tag = paragraphs[1]

        access_link_tag = None

        # Theo XPath: p[3]/a
        if len(paragraphs) >= 3:
            access_link_tag = paragraphs[2].select_one(
                "a[href]"
            )

        # Fallback: tìm link trong content div hoặc card
        if access_link_tag is None and content_div:
            access_link_tag = content_div.select_one(
                "a[href]"
            )

        if access_link_tag is None:
            access_link_tag = card.select_one(
                "a[href]"
            )

        journey_title = clean_text(
            title_tag
        )

        journey_description = clean_text(
            description_tag
        )

        journey_url = (
            make_absolute_url(
                access_link_tag.get("href", ""),
                nha_trang_source_url
            )
            if access_link_tag
            else ""
        )

        # Chống lấy trùng cùng một card
        card_key = (
            journey_title,
            journey_url
        )

        if card_key in processed_cards:
            continue

        processed_cards.add(
            card_key
        )

        if not journey_title and not journey_url:
            continue

        experience_journey_list.append({
            "topic_id": (
                f"journey-"
                f"{len(experience_journey_list) + 1:02d}"
            ),
            "topic_group": "experience_journeys",
            "topic_group_title": "EXPERIENCE JOURNEYS",
            "title": journey_title,
            "description": journey_description,
            "image_url": get_image_url(
                image_tag,
                nha_trang_source_url
            ),
            "image_alt": (
                image_tag.get("alt", "").strip()
                if image_tag
                else ""
            ),
            "topic_url": journey_url
        })


# ============================================================
# 6. GỘP VÀ LOẠI URL TRÙNG
# ============================================================

nha_trang_all_topic_list = (
    unique_experience_list
    + experience_journey_list
)

nha_trang_unique_topic_list = []

seen_topic_keys = set()

for topic in nha_trang_all_topic_list:
    topic_url = topic.get(
        "topic_url",
        ""
    ).strip()

    topic_title = topic.get(
        "title",
        ""
    ).strip()

    normalized_url = (
        topic_url
        .split("#", 1)[0]
        .rstrip("/")
    )

    topic_key = (
        topic.get("topic_group"),
        normalized_url or topic_title
    )

    if topic_key in seen_topic_keys:
        continue

    seen_topic_keys.add(
        topic_key
    )

    nha_trang_unique_topic_list.append(
        topic
    )


# ============================================================
# 7. KIỂM TRA
# ============================================================

print("=" * 70)
print("KẾT QUẢ TRÍCH XUẤT")
print("=" * 70)

print(
    "Unique Experiences:",
    len(unique_experience_list)
)

print(
    "Experience Journeys:",
    len(experience_journey_list)
)

print(
    "Tổng topic:",
    len(nha_trang_unique_topic_list)
)

print("\nEXPERIENCE JOURNEYS:")

for item in experience_journey_list:
    print("-" * 70)
    print("Tiêu đề:", item["title"])
    print("Mô tả:", item["description"])
    print("Ảnh:", item["image_url"])
    print("Link:", item["topic_url"])

KẾT QUẢ TRÍCH XUẤT
Unique Experiences: 13
Experience Journeys: 3
Tổng topic: 15

EXPERIENCE JOURNEYS:
----------------------------------------------------------------------
Tiêu đề: Explore Hòn Tre in 2 Days 1 Night – The Call of the Sea Paradise
Mô tả: When in Nha Trang, you must visit Hon Tre! We recommend a 2-day-1-night journey to fully explore this paradise in Nha Trang. Enjoy a luxurious stay at Vinpearl Resort Nha Trang, relax with authentic Korean-style saunas at Aquafield, discover the thrills of VinWonders, and experience the excitement at Vinpearl Harbour!
Ảnh: https://static.vinwonders.com/production_style/style/images/orange-arrow.svg
Link: https://vinwonders.com/en/wonderpedia/news/explore-hon-tre-in-2-days-1-night-the-call-of-the-sea-paradise/
----------------------------------------------------------------------
Tiêu đề: The Perfect 3-Day, 2-Night Itinerary for Hòn Tre Island – Experience Vinpearl Nha Trang to the Fullest
Mô tả: Looking for a luxurious getaway that blen

In [6]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException
from pathlib import Path


# ============================================================
# 1. KIỂM TRA DỮ LIỆU
# ============================================================

if "nha_trang_unique_topic_list" not in globals():
    raise ValueError(
        "Chưa có biến nha_trang_unique_topic_list. "
        "Hãy chạy Cell 1 trước."
    )

if not nha_trang_unique_topic_list:
    raise ValueError(
        "nha_trang_unique_topic_list đang rỗng."
    )


# ============================================================
# 2. TẠO THƯ MỤC
# ============================================================

topic_root_directory = Path(
    "html/nha-trang-topics"
)

unique_html_directory = (
    topic_root_directory
    / "unique-experiences"
)

journey_html_directory = (
    topic_root_directory
    / "experience-journeys"
)

unique_html_directory.mkdir(
    parents=True,
    exist_ok=True
)

journey_html_directory.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. BIẾN LƯU KẾT QUẢ
# ============================================================

nha_trang_topic_crawl_results = []

nha_trang_topic_success_list = []

nha_trang_topic_failed_list = []

nha_trang_topic_raw_html_dict = {}


# ============================================================
# 4. TẠO WEBDRIVER
# ============================================================

def create_topic_webdriver():
    edge_options = Options()

    edge_options.add_argument(
        "--start-maximized"
    )

    edge_options.add_argument(
        "--disable-notifications"
    )

    edge_options.add_argument(
        "--disable-popup-blocking"
    )

    edge_options.add_argument(
        "--disable-extensions"
    )

    edge_options.add_argument(
        "--disable-gpu"
    )

    if (
        "HEADERS" in globals()
        and isinstance(HEADERS, dict)
        and HEADERS.get("User-Agent")
    ):
        edge_options.add_argument(
            f"user-agent={HEADERS['User-Agent']}"
        )

    driver = webdriver.Edge(
        options=edge_options
    )

    driver.set_page_load_timeout(
        45
    )

    return driver


# ============================================================
# 5. CRAWL TẤT CẢ TOPIC
# ============================================================

total_topics = len(
    nha_trang_unique_topic_list
)

for topic_index, topic in enumerate(
    nha_trang_unique_topic_list,
    start=1
):
    topic_title = topic.get(
        "title",
        f"topic-{topic_index}"
    ).strip()

    topic_url = topic.get(
        "topic_url",
        ""
    ).strip()

    topic_group = topic.get(
        "topic_group",
        "unknown"
    )

    print(
        f"\n[{topic_index}/{total_topics}] "
        f"Đang crawl: {topic_title}"
    )

    print(
        "Nhóm:",
        topic_group
    )

    print(
        "URL:",
        topic_url
    )

    topic_driver = None

    try:
        if not topic_url:
            raise ValueError(
                "Topic không có URL truy cập."
            )

        topic_driver = create_topic_webdriver()

        try:
            topic_driver.get(
                topic_url
            )

        except TimeoutException:
            print(
                "Trang tải quá 45 giây, "
                "đang lấy HTML hiện có..."
            )

            try:
                topic_driver.execute_script(
                    "window.stop();"
                )
            except Exception:
                pass

        try:
            WebDriverWait(
                topic_driver,
                20,
                poll_frequency=0.5
            ).until(
                lambda driver: (
                    driver.execute_script(
                        "return document.readyState"
                    )
                    in ["interactive", "complete"]
                    and len(driver.page_source) > 1000
                )
            )

        except TimeoutException:
            print(
                "DOM chưa đạt điều kiện, "
                "đang lấy HTML hiện có..."
            )

        topic_page_title = (
            topic_driver.title
        )

        topic_current_url = (
            topic_driver.current_url
        )

        topic_full_html = (
            topic_driver.page_source
        )

        topic_html_length = len(
            topic_full_html
        )

        html_lower = topic_full_html.lower()
        title_lower = topic_page_title.lower()

        cloudflare_detected = (
            "just a moment" in title_lower
            or "checking your browser" in html_lower
            or "cf-chl-" in html_lower
            or "verify you are human" in html_lower
        )

        if cloudflare_detected:
            topic_status = "cloudflare"
            topic_error = (
                "Trang có dấu hiệu bị Cloudflare."
            )

        elif topic_html_length <= 1000:
            topic_status = "incomplete"
            topic_error = (
                "HTML quá ngắn, có thể chưa tải đủ."
            )

        else:
            topic_status = "success"
            topic_error = ""

        if topic_group == "experience_journeys":
            topic_output_directory = (
                journey_html_directory
            )
        else:
            topic_output_directory = (
                unique_html_directory
            )

        topic_slug = create_safe_slug(
            topic_title
        )

        topic_html_filename = (
            f"{topic_index:02d}-"
            f"{topic_slug}.html"
        )

        topic_html_path = (
            topic_output_directory
            / topic_html_filename
        )

        topic_html_path.write_text(
            topic_full_html,
            encoding="utf-8-sig"
        )

        topic_html_data = {
            **topic,
            "topic_index": topic_index,
            "page_title": topic_page_title,
            "source_url": topic_url,
            "current_url": topic_current_url,
            "html_filename": topic_html_filename,
            "html_path": str(topic_html_path),
            "html_length": topic_html_length,
            "crawl_status": topic_status,
            "crawl_error": topic_error
        }

        nha_trang_topic_crawl_results.append(
            topic_html_data
        )

        nha_trang_topic_raw_html_dict[
            topic_url
        ] = topic_full_html

        if topic_status == "success":
            nha_trang_topic_success_list.append(
                topic_html_data
            )

            print(
                "Crawl thành công."
            )

        else:
            nha_trang_topic_failed_list.append(
                topic_html_data
            )

            print(
                "Trạng thái:",
                topic_status
            )

        print(
            "Đã lưu:",
            topic_html_path
        )

        print(
            "HTML length:",
            topic_html_length
        )

    except Exception as error:
        topic_html_data = {
            **topic,
            "topic_index": topic_index,
            "page_title": "",
            "source_url": topic_url,
            "current_url": "",
            "html_filename": "",
            "html_path": "",
            "html_length": 0,
            "crawl_status": "error",
            "crawl_error": str(error)
        }

        nha_trang_topic_crawl_results.append(
            topic_html_data
        )

        nha_trang_topic_failed_list.append(
            topic_html_data
        )

        print(
            "Crawl thất bại:",
            repr(error)
        )

    finally:
        if topic_driver is not None:
            try:
                topic_driver.quit()
                print("Đã đóng WebDriver.")

            except WebDriverException:
                pass


# ============================================================
# 6. TỔNG KẾT THEO NHÓM
# ============================================================

successful_unique_count = sum(
    result["crawl_status"] == "success"
    and result["topic_group"] == "unique_experiences"
    for result in nha_trang_topic_crawl_results
)

successful_journey_count = sum(
    result["crawl_status"] == "success"
    and result["topic_group"] == "experience_journeys"
    for result in nha_trang_topic_crawl_results
)

print("\n" + "=" * 70)
print("HOÀN THÀNH CRAWL TOPIC")
print("=" * 70)

print(
    "Unique Experiences đã crawl:",
    successful_unique_count
)

print(
    "Experience Journeys đã crawl:",
    successful_journey_count
)

print(
    "Tổng thành công:",
    len(nha_trang_topic_success_list)
)

print(
    "Tổng thất bại:",
    len(nha_trang_topic_failed_list)
)

print(
    "Thư mục Experience Journeys:",
    journey_html_directory.resolve()
)


[1/15] Đang crawl: Witness the Full Tata Show
Nhóm: unique_experiences
URL: https://vinwonders.com/vi/tata-show/
Crawl thành công.
Đã lưu: html\nha-trang-topics\unique-experiences\01-witness-the-full-tata-show.html
HTML length: 195861
Đã đóng WebDriver.

[2/15] Đang crawl: Capture Moments at The World Garden
Nhóm: unique_experiences
URL: https://vinwonders.com/vi/wonderpedia/news/tham-quan-kings-garden-nha-trang-vuon-thu-doc-dao-nhat-hon-tre/
Crawl thành công.
Đã lưu: html\nha-trang-topics\unique-experiences\02-capture-moments-at-the-world-garden.html
HTML length: 246361
Đã đóng WebDriver.

[3/15] Đang crawl: Immerse yourself in The Underwater World
Nhóm: unique_experiences
URL: https://vinwonders.com/vi/wonderpedia/news/thuy-cung-vinpearl-nha-trang-gia-ve-huong-dan-tham-quan/
Crawl thành công.
Đã lưu: html\nha-trang-topics\unique-experiences\03-immerse-yourself-in-the-underwater-world.html
HTML length: 245938
Đã đóng WebDriver.

[4/15] Đang crawl: Conquer 15 rides at Festive Hill
Nhó

In [9]:
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
import json
import re


# ============================================================
# 1. KIỂM TRA KẾT QUẢ CRAWL
# ============================================================

if "nha_trang_topic_crawl_results" not in globals():
    raise ValueError(
        "Chưa có nha_trang_topic_crawl_results. "
        "Hãy chạy Cell 2 trước."
    )


# ============================================================
# 2. THƯ MỤC JSON
# ============================================================

json_root_directory = Path(
    "json/nha-trang"
)

unique_json_directory = (
    json_root_directory
    / "unique-experiences"
)

journey_json_directory = (
    json_root_directory
    / "experience-journeys"
)

unique_json_directory.mkdir(
    parents=True,
    exist_ok=True
)

journey_json_directory.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. HÀM TRÍCH XUẤT CHUNG
# ============================================================

def normalize_content_text(text):
    return re.sub(
        r"\s+",
        " ",
        str(text or "")
    ).strip()


def get_meta_content(soup, selector):
    tag = soup.select_one(
        selector
    )

    if not tag:
        return ""

    return normalize_content_text(
        tag.get("content", "")
    )


def get_content_image_url(image_tag, base_url):
    if image_tag is None:
        return ""

    image_url = (
        image_tag.get("data-lazy-src")
        or image_tag.get("data-src")
        or image_tag.get("src")
        or ""
    )

    return urljoin(
        base_url,
        image_url
    )


def extract_page_to_json(
    html,
    source_url
):
    soup = BeautifulSoup(
        html,
        "lxml"
    )

    page_title = normalize_content_text(
        soup.title.get_text(
            " ",
            strip=True
        )
        if soup.title
        else ""
    )

    canonical_tag = soup.select_one(
        'link[rel="canonical"]'
    )

    canonical_url = (
        canonical_tag.get("href", "").strip()
        if canonical_tag
        else source_url
    )

    meta_description = get_meta_content(
        soup,
        'meta[name="description"]'
    )

    og_title = get_meta_content(
        soup,
        'meta[property="og:title"]'
    )

    og_description = get_meta_content(
        soup,
        'meta[property="og:description"]'
    )

    og_image = get_meta_content(
        soup,
        'meta[property="og:image"]'
    )

    # Xóa phần không cần cho RAG
    for unwanted_tag in soup.select(
        """
        script,
        style,
        noscript,
        svg,
        iframe,
        form,
        nav,
        footer,
        button
        """
    ):
        unwanted_tag.decompose()

    main_content = (
        soup.select_one("article")
        or soup.select_one("main")
        or soup.select_one(".entry-content")
        or soup.select_one(".post-content")
        or soup.select_one(".article-content")
        or soup.select_one(".page-content")
        or soup.body
        or soup
    )

    content_blocks = []

    for element in main_content.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li",
            "blockquote",
            "table",
            "img"
        ]
    ):
        if element.name == "img":
            image_url = get_content_image_url(
                element,
                canonical_url
            )

            if image_url:
                content_blocks.append({
                    "type": "image",
                    "url": image_url,
                    "alt": normalize_content_text(
                        element.get("alt", "")
                    )
                })

            continue

        if element.name == "table":
            table_rows = []

            for row in element.select("tr"):
                cells = [
                    normalize_content_text(
                        cell.get_text(
                            " ",
                            strip=True
                        )
                    )
                    for cell in row.select(
                        "th, td"
                    )
                ]

                if any(cells):
                    table_rows.append(
                        cells
                    )

            if table_rows:
                content_blocks.append({
                    "type": "table",
                    "rows": table_rows
                })

            continue

        text = normalize_content_text(
            element.get_text(
                " ",
                strip=True
            )
        )

        if not text:
            continue

        type_map = {
            "h1": "heading_1",
            "h2": "heading_2",
            "h3": "heading_3",
            "h4": "heading_4",
            "p": "paragraph",
            "li": "list_item",
            "blockquote": "quote"
        }

        content_blocks.append({
            "type": type_map[element.name],
            "text": text
        })

    page_links = []

    seen_links = set()

    for link_tag in main_content.select(
        "a[href]"
    ):
        link_url = urljoin(
            canonical_url,
            link_tag.get("href", "")
        )

        link_text = normalize_content_text(
            link_tag.get_text(
                " ",
                strip=True
            )
        )

        link_key = (
            link_text,
            link_url
        )

        if link_key in seen_links:
            continue

        seen_links.add(
            link_key
        )

        page_links.append({
            "text": link_text,
            "url": link_url
        })

    full_text_parts = []

    for block in content_blocks:
        if "text" in block:
            full_text_parts.append(
                block["text"]
            )

        elif block["type"] == "table":
            for row in block["rows"]:
                full_text_parts.append(
                    " | ".join(row)
                )

    full_text = "\n".join(
        full_text_parts
    )

    return {
        "page_title": page_title,
        "canonical_url": canonical_url,
        "metadata": {
            "meta_description": meta_description,
            "og_title": og_title,
            "og_description": og_description,
            "og_image": og_image
        },
        "content_blocks": content_blocks,
        "links": page_links,
        "full_text": full_text,
        "character_count": len(full_text)
    }


# ============================================================
# 4. TẠO DOCUMENT JSON CHO TỪNG TOPIC
# ============================================================

nha_trang_topic_json_documents = []

for topic_result in nha_trang_topic_crawl_results:
    html_path_text = topic_result.get(
        "html_path",
        ""
    )

    extracted_page_data = {
        "page_title": topic_result.get(
            "page_title",
            ""
        ),
        "canonical_url": topic_result.get(
            "current_url",
            ""
        ),
        "metadata": {},
        "content_blocks": [],
        "links": [],
        "full_text": "",
        "character_count": 0
    }

    if html_path_text:
        html_path = Path(
            html_path_text
        )

        if html_path.exists():
            topic_html = html_path.read_text(
                encoding="utf-8-sig",
                errors="ignore"
            )

            extracted_page_data = extract_page_to_json(
                html=topic_html,
                source_url=topic_result.get(
                    "current_url"
                ) or topic_result.get(
                    "topic_url",
                    ""
                )
            )

    topic_document = {
        "document_type": "nha_trang_topic",
        "destination": "Nha Trang",
        "language": "en",

        "topic_group": topic_result.get(
            "topic_group",
            ""
        ),

        "topic_group_title": topic_result.get(
            "topic_group_title",
            ""
        ),

        "topic_id": topic_result.get(
            "topic_id",
            ""
        ),

        "card_data": {
            "title": topic_result.get(
                "title",
                ""
            ),
            "description": topic_result.get(
                "description",
                ""
            ),
            "image_url": topic_result.get(
                "image_url",
                ""
            ),
            "image_alt": topic_result.get(
                "image_alt",
                ""
            ),
            "topic_url": topic_result.get(
                "topic_url",
                ""
            )
        },

        "crawl_data": {
            "status": topic_result.get(
                "crawl_status",
                ""
            ),
            "error": topic_result.get(
                "crawl_error",
                ""
            ),
            "source_url": topic_result.get(
                "source_url",
                ""
            ),
            "current_url": topic_result.get(
                "current_url",
                ""
            ),
            "html_filename": topic_result.get(
                "html_filename",
                ""
            ),
            "html_path": topic_result.get(
                "html_path",
                ""
            ),
            "html_length": topic_result.get(
                "html_length",
                0
            )
        },

        "page_data": extracted_page_data
    }

    nha_trang_topic_json_documents.append(
        topic_document
    )

    topic_slug = create_safe_slug(
        topic_result.get(
            "title",
            "unknown-topic"
        )
    )

    topic_group = topic_result.get(
        "topic_group",
        ""
    )

    if topic_group == "experience_journeys":
        output_directory = (
            journey_json_directory
        )
    else:
        output_directory = (
            unique_json_directory
        )

    topic_json_path = (
        output_directory
        / f"{topic_slug}.json"
    )

    topic_json_path.write_text(
        json.dumps(
            topic_document,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print(
        "Đã lưu JSON:",
        topic_json_path
    )


# ============================================================
# 5. CHIA DANH SÁCH THEO NHÓM
# ============================================================

unique_experience_json_documents = [
    document
    for document in nha_trang_topic_json_documents
    if document["topic_group"]
    == "unique_experiences"
]

experience_journey_json_documents = [
    document
    for document in nha_trang_topic_json_documents
    if document["topic_group"]
    == "experience_journeys"
]


# ============================================================
# 6. FILE JSON TỔNG
# ============================================================

nha_trang_complete_json = {
    "document_type": "destination_topic_collection",
    "destination": "Nha Trang",
    "language": "en",
    "source_url": nha_trang_source_url,

    "statistics": {
        "unique_experience_count": len(
            unique_experience_json_documents
        ),
        "experience_journey_count": len(
            experience_journey_json_documents
        ),
        "total_topic_count": len(
            nha_trang_topic_json_documents
        )
    },

    "unique_experiences": (
        unique_experience_json_documents
    ),

    "experience_journeys": (
        experience_journey_json_documents
    ),

    "all_topics": (
        nha_trang_topic_json_documents
    )
}

nha_trang_complete_json_path = (
    json_root_directory
    / "nha-trang-all-topics.json"
)

nha_trang_complete_json_path.write_text(
    json.dumps(
        nha_trang_complete_json,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 7. TỔNG KẾT
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH XUẤT JSON")
print("=" * 70)

print(
    "Unique Experiences JSON:",
    len(unique_experience_json_documents)
)

print(
    "Experience Journeys JSON:",
    len(experience_journey_json_documents)
)

print(
    "Tổng topic JSON:",
    len(nha_trang_topic_json_documents)
)

print(
    "File JSON tổng:",
    nha_trang_complete_json_path.resolve()
)

Đã lưu JSON: json\nha-trang\unique-experiences\witness-the-full-tata-show.json
Đã lưu JSON: json\nha-trang\unique-experiences\capture-moments-at-the-world-garden.json
Đã lưu JSON: json\nha-trang\unique-experiences\immerse-yourself-in-the-underwater-world.json
Đã lưu JSON: json\nha-trang\unique-experiences\conquer-15-rides-at-festive-hill.json
Đã lưu JSON: json\nha-trang\unique-experiences\experience-southeast-asia-s-1-water-attraction.json
Đã lưu JSON: json\nha-trang\unique-experiences\zipline-with-courage.json
Đã lưu JSON: json\nha-trang\unique-experiences\count-every-destination-at-the-flying-theater.json
Đã lưu JSON: json\nha-trang\unique-experiences\relax-in-all-7-therapy-rooms-at-aquafield-nha-trang.json
Đã lưu JSON: json\nha-trang\unique-experiences\watch-the-stunt-show-without-flinching.json
Đã lưu JSON: json\nha-trang\unique-experiences\hunt-for-unbeatable-deals-at-vinpearl-harbour.json
Đã lưu JSON: json\nha-trang\unique-experiences\chill-at-the-lagoon-beach-club.json
Đã lưu JS

In [1]:
from bs4 import BeautifulSoup, Tag, Comment
from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag
from datetime import datetime, timezone
import unicodedata
import hashlib
import json
import re


# ============================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN THỰC TẾ
# ============================================================

project_directory = Path(
    r"D:\vinuni\T013\data_crawl"
)

unique_experience_html_directory = (
    project_directory
    / "html"
    / "nha-trang-topics"
    / "unique-experiences"
)

experience_journey_html_directory = (
    project_directory
    / "html"
    / "nha-trang-topics"
    / "experience-journeys"
)

destination_json_path = (
    project_directory
    / "json"
    / "nha-trang-destination.json"
)

json_output_directory = (
    project_directory
    / "json"
    / "nha-trang-complete"
)

topic_json_directory = (
    json_output_directory
    / "topics"
)

unique_json_directory = (
    topic_json_directory
    / "unique-experiences"
)

journey_json_directory = (
    topic_json_directory
    / "experience-journeys"
)


# ============================================================
# 2. KIỂM TRA ĐƯỜNG DẪN
# ============================================================

if not unique_experience_html_directory.exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục Unique Experiences:\n"
        f"{unique_experience_html_directory}"
    )

if not experience_journey_html_directory.exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục Experience Journeys:\n"
        f"{experience_journey_html_directory}"
    )

if not destination_json_path.exists():
    raise FileNotFoundError(
        "Không tìm thấy JSON tổng quan Nha Trang:\n"
        f"{destination_json_path}"
    )

unique_json_directory.mkdir(
    parents=True,
    exist_ok=True
)

journey_json_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Unique Experiences:", unique_experience_html_directory)
print("Experience Journeys:", experience_journey_html_directory)
print("JSON đầu ra:", json_output_directory)


# ============================================================
# 3. ĐỌC JSON TỔNG QUAN NHA TRANG
# ============================================================

nha_trang_destination_data = json.loads(
    destination_json_path.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 4. HÀM CHUẨN HÓA VĂN BẢN
# ============================================================

def normalize_text(value):
    """
    Chuẩn hóa khoảng trắng và ký tự đặc biệt.
    """

    if value is None:
        return ""

    if isinstance(value, Tag):
        value = value.get_text(
            " ",
            strip=True
        )

    value = str(value)

    value = value.replace(
        "\xa0",
        " "
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_multiline_text(value):
    """
    Chuẩn hóa văn bản nhiều dòng nhưng vẫn giữ ranh giới đoạn.
    """

    if value is None:
        return ""

    normalized_lines = []

    for line in str(value).splitlines():
        line = normalize_text(line)

        if line:
            normalized_lines.append(line)

    return "\n".join(
        normalized_lines
    )


# ============================================================
# 5. HÀM TẠO SLUG VÀ DOCUMENT ID
# ============================================================

def create_slug(value):
    value = normalize_text(value)

    normalized_value = unicodedata.normalize(
        "NFD",
        value
    )

    ascii_value = "".join(
        character
        for character in normalized_value
        if unicodedata.category(character) != "Mn"
    )

    ascii_value = (
        ascii_value
        .replace("đ", "d")
        .replace("Đ", "D")
        .lower()
    )

    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        ascii_value
    ).strip("-")

    return slug or "unknown-topic"


def create_document_id(
    topic_group,
    source_url,
    title
):
    raw_value = "|".join([
        normalize_text(topic_group),
        normalize_text(source_url),
        normalize_text(title)
    ])

    digest = hashlib.sha1(
        raw_value.encode("utf-8")
    ).hexdigest()[:12]

    return (
        f"nha-trang-"
        f"{create_slug(topic_group)}-"
        f"{digest}"
    )


# ============================================================
# 6. HÀM XỬ LÝ URL
# ============================================================

def normalize_url(
    url,
    base_url=""
):
    url = normalize_text(url)

    if not url:
        return ""

    absolute_url = urljoin(
        base_url,
        url
    )

    clean_url, _ = urldefrag(
        absolute_url
    )

    return clean_url


def url_match_key(url):
    """
    Tạo khóa URL để ghép HTML với dữ liệu card.
    """

    url = normalize_text(url)

    if not url:
        return ""

    parsed_url = urlparse(url)

    host = (
        parsed_url.netloc
        .lower()
        .replace("www.", "")
    )

    path = (
        parsed_url.path
        .lower()
        .rstrip("/")
    )

    return f"{host}{path}"


def is_http_url(url):
    return url.startswith(
        ("http://", "https://")
    )


# ============================================================
# 7. ĐỌC META HTML
# ============================================================

def get_meta_content(
    soup,
    selector
):
    tag = soup.select_one(selector)

    if tag is None:
        return ""

    return normalize_text(
        tag.get("content", "")
    )


def get_page_title(soup):
    if soup.title is None:
        return ""

    return normalize_text(
        soup.title
    )


def get_canonical_url(
    soup,
    fallback_url=""
):
    canonical_tag = soup.select_one(
        'link[rel="canonical"]'
    )

    if canonical_tag:
        canonical_url = normalize_url(
            canonical_tag.get("href", ""),
            fallback_url
        )

        if canonical_url:
            return canonical_url

    return fallback_url


# ============================================================
# 8. LẤY URL ẢNH THẬT
# ============================================================

def get_best_image_url(
    image_tag,
    base_url=""
):
    if image_tag is None:
        return ""

    image_url = (
        image_tag.get("data-lazy-src")
        or image_tag.get("data-src")
        or image_tag.get("data-original")
        or image_tag.get("data-image")
        or image_tag.get("src")
        or ""
    )

    image_url = normalize_url(
        image_url,
        base_url
    )

    if (
        image_url.startswith("data:")
        or "svg+xml" in image_url
    ):
        return ""

    return image_url


def get_srcset_image_url(
    image_tag,
    base_url=""
):
    if image_tag is None:
        return ""

    srcset = (
        image_tag.get("data-lazy-srcset")
        or image_tag.get("srcset")
        or ""
    )

    if not srcset:
        return ""

    image_candidates = []

    for srcset_item in srcset.split(","):
        parts = srcset_item.strip().split()

        if not parts:
            continue

        candidate_url = parts[0]
        candidate_width = 0

        if len(parts) > 1:
            width_match = re.search(
                r"(\d+)w",
                parts[1]
            )

            if width_match:
                candidate_width = int(
                    width_match.group(1)
                )

        image_candidates.append({
            "url": candidate_url,
            "width": candidate_width
        })

    if not image_candidates:
        return ""

    image_candidates.sort(
        key=lambda item: item["width"],
        reverse=True
    )

    return normalize_url(
        image_candidates[0]["url"],
        base_url
    )


# ============================================================
# 9. TẠO MAP CARD DATA TỪ NHA-TRANG-DESTINATION.JSON
# ============================================================

card_data_by_url = {}

destination_sections = (
    nha_trang_destination_data
    .get("sections", {})
)

for section_key in [
    "unique_experiences",
    "experience_journeys"
]:
    section_data = destination_sections.get(
        section_key,
        {}
    )

    section_title = section_data.get(
        "section_title",
        ""
    )

    for item in section_data.get(
        "items",
        []
    ):
        topic_url = item.get(
            "topic_url",
            ""
        )

        key = url_match_key(
            topic_url
        )

        if not key:
            continue

        card_data_by_url[key] = {
            **item,
            "section_key": section_key,
            "section_title": section_title
        }


# ============================================================
# 10. PHÁT HIỆN LOẠI TRANG
# ============================================================

def detect_page_type(
    soup,
    html_path,
    topic_group
):
    if topic_group == "experience_journeys":
        return "experience_journey_article"

    if soup.select_one(
        "#main-single-content.news-details"
    ):
        return "vinwonders_article"

    if soup.select_one(
        "#main-single-content"
    ):
        return "vinwonders_article"

    if soup.select_one(
        ".page-main-content"
    ):
        return "landing_page"

    if (
        soup.select_one("#p-hotel-custom")
        or soup.select_one(".node-hotel")
    ):
        return "hotel_page"

    if soup.select_one(
        "main.p-vg__golf-course"
    ):
        return "golf_page"

    page_title = get_page_title(
        soup
    ).lower()

    if "tata show" in page_title:
        return "show_landing_page"

    if soup.find("article"):
        return "generic_article"

    return "generic_landing_page"


# ============================================================
# 11. CHỌN VÙNG NỘI DUNG CHÍNH
# ============================================================

def choose_main_content(
    soup,
    page_type
):
    selector_map = {
        "experience_journey_article": [
            "#main-single-content.news-details",
            "#main-single-content",
            ".news-details",
            ".qb_listing_detailds_page",
            "article"
        ],

        "vinwonders_article": [
            "#main-single-content.news-details",
            "#main-single-content",
            ".news-details",
            ".qb_listing_detailds_page",
            "article"
        ],

        "landing_page": [
            ".page-main-content",
            "main",
            ".page-content",
            ".region-content",
            ".content"
        ],

        "hotel_page": [
            "#p-hotel-custom",
            ".node-hotel.hotel_homepage",
            ".node-hotel",
            "#detail-general",
            ".region-content"
        ],

        "golf_page": [
            "main.p-vg__golf-course",
            ".p-vg__golf-course",
            ".c-vg__golf-island",
            "main"
        ],

        "show_landing_page": [
            "main",
            ".page-main-content",
            ".page-content",
            ".content"
        ],

        "generic_article": [
            "article",
            "main",
            ".entry-content",
            ".post-content",
            ".article-content",
            ".page-content"
        ],

        "generic_landing_page": [
            "main",
            ".page-main-content",
            ".page-content",
            ".region-content",
            ".content",
            "body"
        ]
    }

    for selector in selector_map.get(
        page_type,
        []
    ):
        candidates = soup.select(
            selector
        )

        valid_candidates = []

        for candidate in candidates:
            candidate_text_length = len(
                normalize_text(candidate)
            )

            if candidate_text_length >= 100:
                valid_candidates.append(
                    candidate
                )

        if valid_candidates:
            return max(
                valid_candidates,
                key=lambda candidate: len(
                    normalize_text(candidate)
                )
            )

    return soup.body or soup


# ============================================================
# 12. XÓA MENU, FOOTER, POPUP VÀ NỘI DUNG RÁC
# ============================================================

UNWANTED_SELECTORS = [
    "script",
    "style",
    "noscript",
    "template",
    "svg",
    "canvas",
    "iframe",
    "form",
    "button",

    "header",
    "footer",
    "nav",
    "aside",

    ".header",
    ".footer",
    ".site-header",
    ".site-footer",
    ".main-header",
    ".main-footer",

    ".menu",
    ".navbar",
    ".navigation",

    ".breadcrumb",
    ".breadcrumbs",

    ".popup",
    ".modal",
    ".modal-backdrop",
    ".cookie",
    ".cookie-banner",

    ".social-share",
    ".share",
    ".sharing",

    ".search",
    ".search-box",
    ".search-ticket",
    ".search-history",

    ".booking",
    ".booking-form",
    ".booking-widget",
    ".book-now",

    ".related-post",
    ".related-posts",
    ".post-related",
    ".posts-for-you",

    ".featured-offers",
    ".most-popular",

    ".kk-star-ratings",
    "#toc_container",

    ".advertisement",
    ".ads",
    ".sidebar",
    ".widget",

    ".slick-arrow",
    ".swiper-button-next",
    ".swiper-button-prev",
    ".swiper-pagination"
]


def clean_main_content(main_content):
    """
    Sao chép vùng nội dung rồi xóa phần không cần thiết.
    """

    copied_soup = BeautifulSoup(
        str(main_content),
        "lxml"
    )

    copied_main = (
        copied_soup.body
        or copied_soup
    )

    for selector in UNWANTED_SELECTORS:
        for element in copied_main.select(
            selector
        ):
            element.decompose()

    for comment in copied_main.find_all(
        string=lambda text: isinstance(
            text,
            Comment
        )
    ):
        comment.extract()

    for element in copied_main.select(
        '[aria-hidden="true"], [hidden]'
    ):
        element.decompose()

    return copied_main


# ============================================================
# 13. XÁC ĐỊNH LOẠI BLOCK
# ============================================================

def infer_block_type(element):
    tag_name = element.name.lower()

    if tag_name in [
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6"
    ]:
        return f"heading_{tag_name[1:]}"

    if tag_name == "p":
        return "paragraph"

    if tag_name == "li":
        return "list_item"

    if tag_name == "blockquote":
        return "quote"

    if tag_name in [
        "figcaption",
        "caption"
    ]:
        return "caption"

    return ""


# ============================================================
# 14. LỌC VĂN BẢN RÁC
# ============================================================

NOISE_TEXTS = {
    "view more",
    "read more",
    "see more",
    "book now",
    "buy now",
    "xem thêm",
    "đặt ngay",
    "search",
    "search history",
    "most popular",
    "featured offers",
    "related posts",
    "posts for you"
}


def is_noise_text(text):
    normalized_text = normalize_text(
        text
    ).lower()

    if not normalized_text:
        return True

    if normalized_text in NOISE_TEXTS:
        return True

    if not re.search(
        r"[A-Za-zÀ-ỹ0-9]",
        normalized_text
    ):
        return True

    return False


# ============================================================
# 15. TRÍCH XUẤT BẢNG
# ============================================================

def extract_table(table_tag):
    table_rows = []

    for row_tag in table_tag.select(
        "tr"
    ):
        row_values = []

        for cell_tag in row_tag.select(
            "th, td"
        ):
            row_values.append(
                normalize_text(
                    cell_tag
                )
            )

        if any(row_values):
            table_rows.append(
                row_values
            )

    return table_rows


# ============================================================
# 16. TRÍCH XUẤT CONTENT BLOCK THEO THỨ TỰ
# ============================================================

def extract_content_blocks(
    main_content,
    base_url
):
    content_blocks = []

    seen_text_blocks = set()
    seen_images = set()
    seen_tables = set()

    target_tags = [
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "p",
        "li",
        "blockquote",
        "figcaption",
        "table",
        "img"
    ]

    for element in main_content.find_all(
        target_tags
    ):
        tag_name = element.name.lower()

        # ----------------------------------------------------
        # IMAGE
        # ----------------------------------------------------

        if tag_name == "img":
            image_url = (
                get_best_image_url(
                    element,
                    base_url
                )
                or get_srcset_image_url(
                    element,
                    base_url
                )
            )

            if (
                not image_url
                or image_url in seen_images
            ):
                continue

            image_url_lower = image_url.lower()

            unwanted_image_keywords = [
                "logo",
                "icon",
                "arrow",
                "favicon",
                "loading",
                "placeholder"
            ]

            if any(
                keyword in image_url_lower
                for keyword in unwanted_image_keywords
            ):
                continue

            seen_images.add(
                image_url
            )

            content_blocks.append({
                "type": "image",
                "url": image_url,
                "alt": normalize_text(
                    element.get("alt", "")
                ),
                "title": normalize_text(
                    element.get("title", "")
                )
            })

            continue

        # ----------------------------------------------------
        # TABLE
        # ----------------------------------------------------

        if tag_name == "table":
            table_rows = extract_table(
                element
            )

            if not table_rows:
                continue

            table_key = json.dumps(
                table_rows,
                ensure_ascii=False
            )

            if table_key in seen_tables:
                continue

            seen_tables.add(
                table_key
            )

            content_blocks.append({
                "type": "table",
                "rows": table_rows
            })

            continue

        # ----------------------------------------------------
        # TEXT
        # ----------------------------------------------------

        text = normalize_text(
            element
        )

        if is_noise_text(text):
            continue

        block_type = infer_block_type(
            element
        )

        if not block_type:
            continue

        if (
            block_type in [
                "paragraph",
                "list_item",
                "caption"
            ]
            and len(text) < 3
        ):
            continue

        text_key = (
            block_type,
            text.lower()
        )

        if text_key in seen_text_blocks:
            continue

        seen_text_blocks.add(
            text_key
        )

        block_data = {
            "type": block_type,
            "text": text
        }

        inline_links = []

        for link_tag in element.select(
            "a[href]"
        ):
            link_url = normalize_url(
                link_tag.get("href", ""),
                base_url
            )

            link_text = normalize_text(
                link_tag
            )

            if (
                link_url
                and is_http_url(link_url)
            ):
                inline_links.append({
                    "text": link_text,
                    "url": link_url
                })

        if inline_links:
            block_data["links"] = (
                inline_links
            )

        content_blocks.append(
            block_data
        )

    return content_blocks


# ============================================================
# 17. TRÍCH XUẤT TOÀN BỘ LINK
# ============================================================

def extract_links(
    main_content,
    base_url
):
    extracted_links = []

    seen_link_keys = set()

    for link_tag in main_content.select(
        "a[href]"
    ):
        link_url = normalize_url(
            link_tag.get("href", ""),
            base_url
        )

        link_text = normalize_text(
            link_tag
        )

        if (
            not link_url
            or not is_http_url(link_url)
        ):
            continue

        link_key = (
            link_url,
            link_text.lower()
        )

        if link_key in seen_link_keys:
            continue

        seen_link_keys.add(
            link_key
        )

        parsed_url = urlparse(
            link_url
        )

        extracted_links.append({
            "text": link_text,
            "url": link_url,
            "is_internal": (
                "vinwonders.com"
                in parsed_url.netloc
                or "vinpearl.com"
                in parsed_url.netloc
            )
        })

    return extracted_links


# ============================================================
# 18. TRÍCH XUẤT DANH SÁCH ẢNH
# ============================================================

def extract_images(
    main_content,
    base_url
):
    extracted_images = []

    seen_image_urls = set()

    for image_tag in main_content.select(
        "img"
    ):
        image_url = (
            get_best_image_url(
                image_tag,
                base_url
            )
            or get_srcset_image_url(
                image_tag,
                base_url
            )
        )

        if (
            not image_url
            or image_url in seen_image_urls
        ):
            continue

        image_url_lower = image_url.lower()

        unwanted_image_keywords = [
            "logo",
            "icon",
            "arrow",
            "favicon",
            "loading",
            "placeholder"
        ]

        if any(
            keyword in image_url_lower
            for keyword in unwanted_image_keywords
        ):
            continue

        seen_image_urls.add(
            image_url
        )

        extracted_images.append({
            "url": image_url,
            "alt": normalize_text(
                image_tag.get("alt", "")
            ),
            "title": normalize_text(
                image_tag.get("title", "")
            )
        })

    return extracted_images


# ============================================================
# 19. GOM CONTENT BLOCK THÀNH SECTION
# ============================================================

def build_sections(content_blocks):
    sections = []

    current_section = {
        "section_id": "introduction",
        "heading": "",
        "heading_level": 0,
        "blocks": []
    }

    section_counter = 0

    for block in content_blocks:
        block_type = block.get(
            "type",
            ""
        )

        is_heading = block_type.startswith(
            "heading_"
        )

        if is_heading:
            if (
                current_section["blocks"]
                or current_section["heading"]
            ):
                sections.append(
                    current_section
                )

            section_counter += 1

            level_match = re.search(
                r"heading_(\d+)",
                block_type
            )

            heading_level = (
                int(level_match.group(1))
                if level_match
                else 2
            )

            current_section = {
                "section_id": (
                    f"section-{section_counter:02d}-"
                    f"{create_slug(block.get('text', ''))}"
                ),
                "heading": block.get(
                    "text",
                    ""
                ),
                "heading_level": heading_level,
                "blocks": []
            }

        else:
            current_section["blocks"].append(
                block
            )

    if (
        current_section["blocks"]
        or current_section["heading"]
    ):
        sections.append(
            current_section
        )

    for section in sections:
        text_parts = []

        if section["heading"]:
            text_parts.append(
                section["heading"]
            )

        for block in section["blocks"]:
            if block.get("text"):
                text_parts.append(
                    block["text"]
                )

            elif block.get("type") == "table":
                for row in block.get(
                    "rows",
                    []
                ):
                    text_parts.append(
                        " | ".join(row)
                    )

        section["text"] = "\n".join(
            text_parts
        )

    return sections


# ============================================================
# 20. TẠO FULL TEXT
# ============================================================

def build_full_text(
    title,
    summary,
    sections
):
    text_parts = []

    if title:
        text_parts.append(
            title
        )

    if summary and summary != title:
        text_parts.append(
            summary
        )

    for section in sections:
        section_text = section.get(
            "text",
            ""
        )

        if section_text:
            text_parts.append(
                section_text
            )

    final_parts = []
    seen_parts = set()

    for part in text_parts:
        normalized_part = normalize_multiline_text(
            part
        )

        part_key = normalized_part.lower()

        if (
            not normalized_part
            or part_key in seen_parts
        ):
            continue

        seen_parts.add(
            part_key
        )

        final_parts.append(
            normalized_part
        )

    return "\n\n".join(
        final_parts
    )


# ============================================================
# 21. SUY LUẬN THỜI LƯỢNG HÀNH TRÌNH
# ============================================================

def infer_duration(
    title,
    full_text
):
    search_text = (
        f"{title} {full_text[:1500]}"
    )

    patterns = [
        r"\b(\d+)\s*[-–]\s*day[s]?\s*,?\s*(\d+)\s*[-–]\s*night[s]?\b",
        r"\b(\d+)\s*day[s]?\s*(\d+)\s*night[s]?\b",
        r"\b(\d+)\s*ngày\s*(\d+)\s*đêm\b"
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            search_text,
            flags=re.IGNORECASE
        )

        if match:
            return {
                "days": int(
                    match.group(1)
                ),
                "nights": int(
                    match.group(2)
                ),
                "label": match.group(0)
            }

    return {
        "days": None,
        "nights": None,
        "label": ""
    }


# ============================================================
# 22. TÁCH LỊCH TRÌNH THEO DAY
# ============================================================

def extract_itinerary(sections):
    itinerary = []

    day_pattern = re.compile(
        r"^(?:day|ngày)\s*(\d+)",
        flags=re.IGNORECASE
    )

    for section in sections:
        heading = section.get(
            "heading",
            ""
        )

        day_match = day_pattern.search(
            heading
        )

        if not day_match:
            continue

        activities = []

        for block in section.get(
            "blocks",
            []
        ):
            if block.get("type") in [
                "paragraph",
                "list_item",
                "quote",
                "caption"
            ]:
                activity_text = block.get(
                    "text",
                    ""
                )

                if activity_text:
                    activities.append(
                        activity_text
                    )

        itinerary.append({
            "day_number": int(
                day_match.group(1)
            ),
            "heading": heading,
            "activities": activities,
            "text": section.get(
                "text",
                ""
            )
        })

    return itinerary


# ============================================================
# 23. TẠO CHUNK CHO RAG
# ============================================================

def build_rag_chunks(
    document_id,
    topic_group,
    title,
    source_url,
    sections,
    card_data,
    maximum_characters=1800
):
    rag_chunks = []

    chunk_index = 0

    metadata_parts = [
        "Destination: Nha Trang",
        f"Group: {topic_group}",
        f"Title: {title}"
    ]

    card_description = normalize_text(
        card_data.get(
            "description",
            ""
        )
    )

    if card_description:
        metadata_parts.append(
            f"Summary: {card_description}"
        )

    metadata_prefix = "\n".join(
        metadata_parts
    )

    for section in sections:
        section_text = normalize_multiline_text(
            section.get(
                "text",
                ""
            )
        )

        if not section_text:
            continue

        paragraphs = [
            normalize_text(paragraph)
            for paragraph in section_text.split(
                "\n"
            )
            if normalize_text(paragraph)
        ]

        current_paragraphs = []

        for paragraph in paragraphs:
            candidate_text = "\n".join(
                current_paragraphs
                + [paragraph]
            )

            if (
                current_paragraphs
                and len(candidate_text)
                > maximum_characters
            ):
                chunk_index += 1

                chunk_content = "\n".join(
                    current_paragraphs
                )

                rag_chunks.append({
                    "chunk_id": (
                        f"{document_id}-chunk-"
                        f"{chunk_index:03d}"
                    ),
                    "document_id": document_id,
                    "destination": "Nha Trang",
                    "topic_group": topic_group,
                    "title": title,
                    "section_id": section.get(
                        "section_id",
                        ""
                    ),
                    "section_heading": section.get(
                        "heading",
                        ""
                    ),
                    "source_url": source_url,
                    "content": chunk_content,
                    "text": (
                        f"{metadata_prefix}\n\n"
                        f"{chunk_content}"
                    )
                })

                current_paragraphs = [
                    paragraph
                ]

            else:
                current_paragraphs.append(
                    paragraph
                )

        if current_paragraphs:
            chunk_index += 1

            chunk_content = "\n".join(
                current_paragraphs
            )

            rag_chunks.append({
                "chunk_id": (
                    f"{document_id}-chunk-"
                    f"{chunk_index:03d}"
                ),
                "document_id": document_id,
                "destination": "Nha Trang",
                "topic_group": topic_group,
                "title": title,
                "section_id": section.get(
                    "section_id",
                    ""
                ),
                "section_heading": section.get(
                    "heading",
                    ""
                ),
                "source_url": source_url,
                "content": chunk_content,
                "text": (
                    f"{metadata_prefix}\n\n"
                    f"{chunk_content}"
                )
            })

    return rag_chunks


# ============================================================
# 24. TÌM CARD DATA TƯƠNG ỨNG
# ============================================================

def find_card_data(
    canonical_url,
    html_path,
    page_title
):
    canonical_key = url_match_key(
        canonical_url
    )

    if (
        canonical_key
        and canonical_key in card_data_by_url
    ):
        return card_data_by_url[
            canonical_key
        ]

    filename_slug = re.sub(
        r"^\d+-",
        "",
        html_path.stem
    )

    page_title_slug = create_slug(
        page_title
    )

    best_card_data = {}
    best_score = 0

    for candidate_card_data in card_data_by_url.values():
        card_title_slug = create_slug(
            candidate_card_data.get(
                "title",
                ""
            )
        )

        card_url_path = urlparse(
            candidate_card_data.get(
                "topic_url",
                ""
            )
        ).path

        card_url_slug = create_slug(
            card_url_path
        )

        score = 0

        if (
            filename_slug
            and filename_slug in card_url_slug
        ):
            score += 5

        if (
            card_url_slug
            and card_url_slug in filename_slug
        ):
            score += 5

        if (
            card_title_slug
            and card_title_slug in filename_slug
        ):
            score += 3

        if (
            filename_slug
            and filename_slug in card_title_slug
        ):
            score += 3

        if (
            card_title_slug
            and card_title_slug in page_title_slug
        ):
            score += 2

        if score > best_score:
            best_score = score
            best_card_data = (
                candidate_card_data
            )

    if best_score >= 3:
        return best_card_data

    return {}


# ============================================================
# 25. PARSE MỘT FILE HTML
# ============================================================

def parse_topic_html(
    html_path,
    topic_group
):
    html_content = html_path.read_text(
        encoding="utf-8-sig",
        errors="ignore"
    )

    soup = BeautifulSoup(
        html_content,
        "lxml"
    )

    page_title = get_page_title(
        soup
    )

    canonical_url = get_canonical_url(
        soup
    )

    card_data = find_card_data(
        canonical_url=canonical_url,
        html_path=html_path,
        page_title=page_title
    )

    if not canonical_url:
        canonical_url = card_data.get(
            "topic_url",
            ""
        )

    source_url = (
        canonical_url
        or card_data.get(
            "topic_url",
            ""
        )
    )

    page_type = detect_page_type(
        soup=soup,
        html_path=html_path,
        topic_group=topic_group
    )

    main_content = choose_main_content(
        soup=soup,
        page_type=page_type
    )

    cleaned_main_content = clean_main_content(
        main_content
    )

    h1_tag = cleaned_main_content.select_one(
        "h1"
    )

    h1 = normalize_text(
        h1_tag
    )

    card_title = normalize_text(
        card_data.get(
            "title",
            ""
        )
    )

    title = (
        h1
        or card_title
        or page_title
        or html_path.stem
    )

    metadata = {
        "page_title": page_title,
        "h1": h1,
        "canonical_url": canonical_url,

        "meta_description": get_meta_content(
            soup,
            'meta[name="description"]'
        ),

        "og_title": get_meta_content(
            soup,
            'meta[property="og:title"]'
        ),

        "og_description": get_meta_content(
            soup,
            'meta[property="og:description"]'
        ),

        "og_image": normalize_url(
            get_meta_content(
                soup,
                'meta[property="og:image"]'
            ),
            source_url
        ),

        "language": (
            soup.html.get("lang", "")
            if soup.html
            else ""
        )
    }

    content_blocks = extract_content_blocks(
        main_content=cleaned_main_content,
        base_url=source_url
    )

    sections = build_sections(
        content_blocks
    )

    images = extract_images(
        main_content=cleaned_main_content,
        base_url=source_url
    )

    links = extract_links(
        main_content=cleaned_main_content,
        base_url=source_url
    )

    summary = (
        normalize_text(
            card_data.get(
                "description",
                ""
            )
        )
        or metadata["meta_description"]
        or metadata["og_description"]
    )

    if not summary:
        summary = next(
            (
                block.get("text", "")
                for block in content_blocks
                if block.get("type")
                == "paragraph"
            ),
            ""
        )

    full_text = build_full_text(
        title=title,
        summary=summary,
        sections=sections
    )

    document_id = create_document_id(
        topic_group=topic_group,
        source_url=source_url,
        title=title
    )

    journey_data = None

    if topic_group == "experience_journeys":
        journey_data = {
            "duration": infer_duration(
                title=title,
                full_text=full_text
            ),
            "itinerary": extract_itinerary(
                sections
            )
        }

    rag_chunks = build_rag_chunks(
        document_id=document_id,
        topic_group=topic_group,
        title=title,
        source_url=source_url,
        sections=sections,
        card_data=card_data
    )

    return {
        "document_id": document_id,

        "document_type": (
            "experience_journey"
            if topic_group
            == "experience_journeys"
            else "unique_experience"
        ),

        "destination": {
            "name": "Nha Trang",
            "country": "Vietnam"
        },

        "topic_group": topic_group,

        "page_type": page_type,

        "source": {
            "url": source_url,
            "canonical_url": canonical_url,
            "html_file": str(html_path),
            "html_filename": html_path.name
        },

        "card_data": {
            "topic_id": card_data.get(
                "topic_id"
            ),

            "section_title": card_data.get(
                "section_title",
                ""
            ),

            "title": card_title,

            "description": normalize_text(
                card_data.get(
                    "description",
                    ""
                )
            ),

            "location": normalize_text(
                card_data.get(
                    "location",
                    ""
                )
            ),

            "image_url": normalize_url(
                card_data.get(
                    "image_url",
                    ""
                ),
                source_url
            ),

            "image_alt": normalize_text(
                card_data.get(
                    "image_alt",
                    ""
                )
            ),

            "topic_url": card_data.get(
                "topic_url",
                ""
            )
        },

        "page_data": {
            "title": title,
            "summary": summary,
            "metadata": metadata,
            "sections": sections,
            "content_blocks": content_blocks,
            "images": images,
            "links": links,
            "full_text": full_text,
            "character_count": len(
                full_text
            )
        },

        "journey_data": journey_data,

        "rag": {
            "chunk_count": len(
                rag_chunks
            ),
            "chunks": rag_chunks
        },

        "extraction": {
            "parser_version": "nha-trang-v1.1",

            "extracted_at": datetime.now(
                timezone.utc
            ).isoformat(),

            "page_type": page_type,

            "section_count": len(
                sections
            ),

            "image_count": len(
                images
            ),

            "link_count": len(
                links
            ),

            "html_length": len(
                html_content
            )
        }
    }


# ============================================================
# 26. LẤY DANH SÁCH HTML
# ============================================================

unique_html_files = sorted(
    unique_experience_html_directory.glob(
        "*.html"
    )
)

journey_html_files = sorted(
    experience_journey_html_directory.glob(
        "*.html"
    )
)

print("\n" + "=" * 80)
print("DANH SÁCH FILE HTML")
print("=" * 80)

print(
    "Số file Unique Experiences:",
    len(unique_html_files)
)

print(
    "Số file Experience Journeys:",
    len(journey_html_files)
)


# ============================================================
# 27. PARSE UNIQUE EXPERIENCES
# ============================================================

unique_experience_documents = []

parse_error_list = []

for file_index, html_path in enumerate(
    unique_html_files,
    start=1
):
    print(
        f"\n[Unique {file_index}/"
        f"{len(unique_html_files)}] "
        f"{html_path.name}"
    )

    try:
        document = parse_topic_html(
            html_path=html_path,
            topic_group="unique_experiences"
        )

        unique_experience_documents.append(
            document
        )

        output_path = (
            unique_json_directory
            / f"{html_path.stem}.json"
        )

        output_path.write_text(
            json.dumps(
                document,
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

        print(
            "Tiêu đề:",
            document["page_data"]["title"]
        )

        print(
            "Loại trang:",
            document["page_type"]
        )

        print(
            "Số section:",
            document["extraction"][
                "section_count"
            ]
        )

        print(
            "Số ký tự:",
            document["page_data"][
                "character_count"
            ]
        )

        print(
            "Số chunk:",
            document["rag"][
                "chunk_count"
            ]
        )

        print(
            "Đã lưu:",
            output_path
        )

    except Exception as error:
        parse_error_list.append({
            "topic_group": (
                "unique_experiences"
            ),
            "html_file": str(
                html_path
            ),
            "error": repr(error)
        })

        print(
            "Lỗi parse:",
            repr(error)
        )


# ============================================================
# 28. PARSE EXPERIENCE JOURNEYS
# ============================================================

experience_journey_documents = []

for file_index, html_path in enumerate(
    journey_html_files,
    start=1
):
    print(
        f"\n[Journey {file_index}/"
        f"{len(journey_html_files)}] "
        f"{html_path.name}"
    )

    try:
        document = parse_topic_html(
            html_path=html_path,
            topic_group="experience_journeys"
        )

        experience_journey_documents.append(
            document
        )

        output_path = (
            journey_json_directory
            / f"{html_path.stem}.json"
        )

        output_path.write_text(
            json.dumps(
                document,
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

        print(
            "Tiêu đề:",
            document["page_data"]["title"]
        )

        print(
            "Thời lượng:",
            document["journey_data"][
                "duration"
            ]
        )

        print(
            "Số mục lịch trình:",
            len(
                document["journey_data"][
                    "itinerary"
                ]
            )
        )

        print(
            "Số ký tự:",
            document["page_data"][
                "character_count"
            ]
        )

        print(
            "Số chunk:",
            document["rag"][
                "chunk_count"
            ]
        )

        print(
            "Đã lưu:",
            output_path
        )

    except Exception as error:
        parse_error_list.append({
            "topic_group": (
                "experience_journeys"
            ),
            "html_file": str(
                html_path
            ),
            "error": repr(error)
        })

        print(
            "Lỗi parse:",
            repr(error)
        )


# ============================================================
# 29. TẠO COLLECTION RIÊNG CHO TỪNG NHÓM
# ============================================================

unique_collection = {
    "collection_type": (
        "unique_experience_collection"
    ),
    "destination": "Nha Trang",
    "count": len(
        unique_experience_documents
    ),
    "items": unique_experience_documents
}

journey_collection = {
    "collection_type": (
        "experience_journey_collection"
    ),
    "destination": "Nha Trang",
    "count": len(
        experience_journey_documents
    ),
    "items": experience_journey_documents
}

unique_collection_path = (
    json_output_directory
    / "nha-trang-unique-experiences.json"
)

journey_collection_path = (
    json_output_directory
    / "nha-trang-experience-journeys.json"
)

unique_collection_path.write_text(
    json.dumps(
        unique_collection,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

journey_collection_path.write_text(
    json.dumps(
        journey_collection,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 30. GỘP TOÀN BỘ DỮ LIỆU NHA TRANG
# ============================================================

all_topic_documents = (
    unique_experience_documents
    + experience_journey_documents
)

all_rag_chunks = []

for document in all_topic_documents:
    all_rag_chunks.extend(
        document.get(
            "rag",
            {}
        ).get(
            "chunks",
            []
        )
    )

nha_trang_complete_data = {
    "schema_version": "1.1",

    "document_type": (
        "destination_knowledge_base"
    ),

    "destination": {
        "name": "Nha Trang",
        "country": "Vietnam",

        "language": (
            nha_trang_destination_data
            .get("language", "en")
        ),

        "source_url": (
            nha_trang_destination_data
            .get("source_url", "")
        ),

        "page_title": (
            nha_trang_destination_data
            .get("page_title", "")
        )
    },

    "destination_overview": (
        nha_trang_destination_data
    ),

    "statistics": {
        "unique_experience_count": len(
            unique_experience_documents
        ),

        "experience_journey_count": len(
            experience_journey_documents
        ),

        "total_topic_count": len(
            all_topic_documents
        ),

        "total_rag_chunk_count": len(
            all_rag_chunks
        ),

        "parse_error_count": len(
            parse_error_list
        )
    },

    "unique_experiences": (
        unique_experience_documents
    ),

    "experience_journeys": (
        experience_journey_documents
    ),

    "all_topics": (
        all_topic_documents
    ),

    "rag_chunks": (
        all_rag_chunks
    ),

    "parse_errors": (
        parse_error_list
    )
}

complete_json_path = (
    json_output_directory
    / "nha-trang-complete.json"
)

complete_json_path.write_text(
    json.dumps(
        nha_trang_complete_data,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 31. XUẤT JSONL DÙNG CHO RAG
# ============================================================

rag_jsonl_path = (
    json_output_directory
    / "nha-trang-rag-chunks.jsonl"
)

with rag_jsonl_path.open(
    "w",
    encoding="utf-8"
) as jsonl_file:
    for chunk in all_rag_chunks:
        jsonl_file.write(
            json.dumps(
                chunk,
                ensure_ascii=False
            )
            + "\n"
        )


# ============================================================
# 32. TẠO FILE INDEX NHẸ
# ============================================================

topic_index = []

for document in all_topic_documents:
    topic_index.append({
        "document_id": document[
            "document_id"
        ],

        "document_type": document[
            "document_type"
        ],

        "topic_group": document[
            "topic_group"
        ],

        "page_type": document[
            "page_type"
        ],

        "title": document[
            "page_data"
        ][
            "title"
        ],

        "summary": document[
            "page_data"
        ][
            "summary"
        ],

        "source_url": document[
            "source"
        ][
            "url"
        ],

        "html_file": document[
            "source"
        ][
            "html_file"
        ],

        "section_count": document[
            "extraction"
        ][
            "section_count"
        ],

        "character_count": document[
            "page_data"
        ][
            "character_count"
        ],

        "rag_chunk_count": document[
            "rag"
        ][
            "chunk_count"
        ]
    })

topic_index_path = (
    json_output_directory
    / "nha-trang-topic-index.json"
)

topic_index_path.write_text(
    json.dumps(
        topic_index,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 33. TỔNG KẾT
# ============================================================

print("\n" + "=" * 80)
print("HOÀN THÀNH TỔNG HỢP DỮ LIỆU NHA TRANG")
print("=" * 80)

print(
    "Unique Experiences:",
    len(unique_experience_documents)
)

print(
    "Experience Journeys:",
    len(experience_journey_documents)
)

print(
    "Tổng topic:",
    len(all_topic_documents)
)

print(
    "Tổng chunk RAG:",
    len(all_rag_chunks)
)

print(
    "Số lỗi parse:",
    len(parse_error_list)
)

print("\nJSON tổng:")
print(
    complete_json_path.resolve()
)

print("\nJSON Unique Experiences:")
print(
    unique_collection_path.resolve()
)

print("\nJSON Experience Journeys:")
print(
    journey_collection_path.resolve()
)

print("\nJSONL dùng cho RAG:")
print(
    rag_jsonl_path.resolve()
)

print("\nTopic index:")
print(
    topic_index_path.resolve()
)

Unique Experiences: D:\vinuni\T013\data_crawl\html\nha-trang-topics\unique-experiences
Experience Journeys: D:\vinuni\T013\data_crawl\html\nha-trang-topics\experience-journeys
JSON đầu ra: D:\vinuni\T013\data_crawl\json\nha-trang-complete

DANH SÁCH FILE HTML
Số file Unique Experiences: 11
Số file Experience Journeys: 3

[Unique 1/11] 01-witness-the-full-tata-show.html
Tiêu đề: Siêu phẩm thực cảnh đa phương tiện đầu tiên tại Việt Nam
Loại trang: show_landing_page
Số section: 17
Số ký tự: 5817
Số chunk: 17
Đã lưu: D:\vinuni\T013\data_crawl\json\nha-trang-complete\topics\unique-experiences\01-witness-the-full-tata-show.json

[Unique 2/11] 02-capture-moments-at-the-world-garden.html
Tiêu đề: Feed Giraffes at The King’s Garden
Loại trang: vinwonders_article
Số section: 13
Số ký tự: 7446
Số chunk: 13
Đã lưu: D:\vinuni\T013\data_crawl\json\nha-trang-complete\topics\unique-experiences\02-capture-moments-at-the-world-garden.json

[Unique 3/11] 03-immerse-yourself-in-the-underwater-world.html
T

In [2]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import json
import re
import time

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException

In [4]:
# ============================================================
# CRAWL DỮ LIỆU TRANG PHÚ QUỐC + CRAWL HTML CÁC TRANG CON
# Chạy toàn bộ trong 1 cell Jupyter Notebook
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse
import json
import re
import time

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# 1. CẤU HÌNH
# ============================================================

# Sửa lại tên file nếu file của bạn có tên khác
HTML_FILE = Path(
    r"D:\vinuni\T013\data_crawl\html\phu-quoc.html"
)

# Thư mục lưu HTML của các trang trải nghiệm
DETAIL_HTML_DIR = Path(
    r"D:\vinuni\T013\data_crawl\html\phuquoc"
)

# File JSON lưu kết quả cuối cùng
OUTPUT_JSON = Path(
    r"D:\vinuni\T013\data_crawl\phu_quoc_data.json"
)

BASE_URL = "https://vinwonders.com/en/phu-quoc-destination/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36 "
        "Edg/150.0.0.0"
    )
}

# Tạo thư mục nếu chưa tồn tại
DETAIL_HTML_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CÁC HÀM HỖ TRỢ
# ============================================================

def clean_text(element):
    """Lấy và làm sạch nội dung text của BeautifulSoup element."""
    if element is None:
        return None

    text = element.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def absolute_url(url, base_url=BASE_URL):
    """Chuyển URL tương đối thành URL tuyệt đối."""
    if not url:
        return None

    url = url.strip()

    if not url or url == "#":
        return None

    return urljoin(base_url, url)


def get_image_url(image_element, base_url=BASE_URL):
    """
    Lấy URL ảnh từ các thuộc tính thường gặp:
    src, data-src, data-lazy-src, data-original, srcset.
    """
    if image_element is None:
        return None

    image_url = (
        image_element.get("src")
        or image_element.get("data-src")
        or image_element.get("data-lazy-src")
        or image_element.get("data-original")
    )

    if not image_url:
        srcset = (
            image_element.get("srcset")
            or image_element.get("data-srcset")
        )

        if srcset:
            image_url = srcset.split(",")[0].strip().split()[0]

    return absolute_url(image_url, base_url)


def make_safe_filename(url, index):
    """Tạo tên file HTML an toàn từ URL."""
    parsed_url = urlparse(url)

    slug = parsed_url.path.strip("/").split("/")[-1]

    if not slug:
        slug = parsed_url.netloc

    slug = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "-",
        slug
    ).strip("-").lower()

    if not slug:
        slug = "page"

    return f"{index:02d}_{slug}.html"


def first_select(parent, selectors):
    """
    Thử nhiều CSS selector và trả về element đầu tiên tìm thấy.
    Giúp code chống thay đổi class nhẹ trên website.
    """
    if parent is None:
        return None

    for selector in selectors:
        element = parent.select_one(selector)

        if element is not None:
            return element

    return None


# ============================================================
# 3. ĐỌC FILE HTML GỐC
# ============================================================

if not HTML_FILE.exists():
    # Thử tìm file HTML có chữ phu-quoc trong thư mục
    html_directory = HTML_FILE.parent

    possible_files = list(
        html_directory.glob("*phu*quoc*.html")
    )

    if possible_files:
        HTML_FILE = possible_files[0]

        print(
            "Không thấy đúng tên file đã khai báo.",
            "Tự động sử dụng file:",
            HTML_FILE
        )
    else:
        raise FileNotFoundError(
            f"Không tìm thấy file HTML:\n{HTML_FILE}\n\n"
            "Hãy sửa biến HTML_FILE đúng với tên file trong máy."
        )

html = HTML_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

soup = BeautifulSoup(html, "lxml")

print("=" * 70)
print("ĐÃ ĐỌC FILE HTML GỐC")
print("File:", HTML_FILE)
print("Độ dài HTML:", len(html))
print("Tiêu đề trang:", clean_text(soup.title))
print("=" * 70)


# ============================================================
# 4. WELCOME TO PHU QUOC UNITED CENTER
# ============================================================

welcome_data = {
    "section_title": None,
    "items": []
}

welcome_section = first_select(
    soup,
    [
        ".address-having",
        "section.address-having",
        "[class*='address-having']"
    ]
)

if welcome_section:
    welcome_title_element = first_select(
        welcome_section,
        [
            ".address-having_title",
            "[class*='address-having'][class*='title']",
            "h2",
            "h3"
        ]
    )

    welcome_data["section_title"] = clean_text(
        welcome_title_element
    )

    welcome_cards = welcome_section.select(
        ".address-having_slide .swiper-slide"
    )

    if not welcome_cards:
        welcome_cards = welcome_section.select(
            ".swiper-wrapper > .swiper-slide"
        )

    if not welcome_cards:
        welcome_cards = welcome_section.select(
            ".slide-card"
        )

    welcome_seen = set()

    for card in welcome_cards:
        image_element = first_select(
            card,
            [
                "img.slide-card_img",
                ".slide-card img",
                "img"
            ]
        )

        title_element = first_select(
            card,
            [
                ".slide-card_content_title",
                ".slide-card-content-title",
                ".slide-card_content p:nth-of-type(1)",
                "p:nth-of-type(1)"
            ]
        )

        description_element = first_select(
            card,
            [
                ".slide-card_content_description",
                ".slide-card-content-description",
                ".slide-card_content p:nth-of-type(2)",
                "p:nth-of-type(2)"
            ]
        )

        item = {
            "image_url": get_image_url(image_element),
            "title": clean_text(title_element),
            "description": clean_text(
                description_element
            )
        }

        unique_key = (
            item["image_url"],
            item["title"],
            item["description"]
        )

        # Chỉ lưu item có dữ liệu và loại bỏ slide clone
        if (
            any(item.values())
            and unique_key not in welcome_seen
        ):
            welcome_seen.add(unique_key)
            welcome_data["items"].append(item)

else:
    print(
        "CẢNH BÁO: Không tìm thấy phần "
        "'Welcome to PHU QUOC UNITED CENTER!'"
    )


# ============================================================
# 5. 10 EXCLUSIVE EXPERIENCES
# ============================================================

experience_data = {
    "section_title": None,
    "items": []
}

experience_section = first_select(
    soup,
    [
        ".experience_event",
        "section.experience_event",
        "[class*='experience_event']",
        "[class*='experience-event']"
    ]
)

if experience_section:
    experience_title_element = first_select(
        experience_section,
        [
            ".experience_title",
            "[class*='experience'][class*='title']",
            "h2",
            "h3"
        ]
    )

    experience_data["section_title"] = clean_text(
        experience_title_element
    )

    experience_cards = experience_section.select(
        ".experience_slide .swiper-slide"
    )

    if not experience_cards:
        experience_cards = experience_section.select(
            ".swiper-wrapper > .swiper-slide"
        )

    if not experience_cards:
        experience_cards = experience_section.select(
            ".experience-slide"
        )

    if not experience_cards:
        experience_cards = experience_section.select(
            ".slide-card"
        )

    experience_seen = set()

    for card in experience_cards:
        link_element = first_select(
            card,
            [
                ".slide-card > a[href]",
                "a[href]"
            ]
        )

        image_element = first_select(
            card,
            [
                ".slide-card > a img.slide-card_img",
                "a img.slide-card_img",
                "img.slide-card_img",
                "a img",
                "img"
            ]
        )

        title_element = first_select(
            card,
            [
                ".slide-card_content_title",
                ".slide-card-content-title",
                ".slide-card_content p:nth-of-type(1)",
                "p:nth-of-type(1)"
            ]
        )

        location_element = first_select(
            card,
            [
                ".slide-card_content_add",
                ".slide-card_content_location",
                ".slide-card-content-add",
                ".slide-card_content div p",
                "div p"
            ]
        )

        description_element = first_select(
            card,
            [
                ".slide-card_content_description",
                ".slide-card-content-description",
                ".slide-card_content > p:nth-of-type(2)",
                "p:nth-of-type(2)"
            ]
        )

        detail_url = None

        if link_element:
            detail_url = absolute_url(
                link_element.get("href")
            )

        item = {
            "detail_url": detail_url,
            "image_url": get_image_url(image_element),
            "title": clean_text(title_element),
            "location": clean_text(location_element),
            "description": clean_text(
                description_element
            )
        }

        unique_key = (
            item["detail_url"],
            item["title"]
        )

        if (
            any(item.values())
            and unique_key not in experience_seen
        ):
            experience_seen.add(unique_key)
            experience_data["items"].append(item)

else:
    print(
        "CẢNH BÁO: Không tìm thấy phần "
        "'10 Exclusive Experiences...'"
    )


# ============================================================
# 6. SUGGESTED ITINERARY FOR THE ULTIMATE GETAWAY
# ============================================================

itinerary_data = {
    "section_title": None,
    "items": []
}

itinerary_section = soup.select_one(
    "#news-and-experiences"
)

if itinerary_section:
    itinerary_title_element = first_select(
        itinerary_section,
        [
            ".trip_title",
            "[class*='trip'][class*='title']",
            "h2",
            "h3"
        ]
    )

    itinerary_data["section_title"] = clean_text(
        itinerary_title_element
    )

    itinerary_cards = itinerary_section.select(
        ".trip_card-item"
    )

    if not itinerary_cards:
        itinerary_cards = itinerary_section.select(
            "[class*='trip_card']"
        )

    if not itinerary_cards:
        # Theo XPath:
        # //*[@id="news-and-experiences"]/div/div
        itinerary_cards = itinerary_section.select(
            ":scope > div > div"
        )

    itinerary_seen = set()

    for card in itinerary_cards:
        image_element = first_select(
            card,
            [
                "img.trip_card-item-img",
                ".trip_card-item-img",
                "div:nth-of-type(1) img",
                "img"
            ]
        )

        title_element = first_select(
            card,
            [
                ".trip-card_card-desc_title",
                ".trip_card-desc_title",
                "div:nth-of-type(2) > p:nth-of-type(1)"
            ]
        )

        description_element = first_select(
            card,
            [
                ".trip-card_card-desc_description",
                ".trip_card-desc_description",
                "div:nth-of-type(2) > p:nth-of-type(2)"
            ]
        )

        option_element = first_select(
            card,
            [
                ".trip-card_card-desc_button a[href]",
                ".trip_card-desc_button a[href]",
                "div:nth-of-type(2) > p:nth-of-type(3) a[href]",
                "a[href]"
            ]
        )

        option_url = None

        if option_element:
            option_url = absolute_url(
                option_element.get("href")
            )

        item = {
            "image_url": get_image_url(image_element),
            "title": clean_text(title_element),
            "description": clean_text(
                description_element
            ),
            "option_text": clean_text(option_element),
            "option_url": option_url
        }

        unique_key = (
            item["title"],
            item["option_url"],
            item["image_url"]
        )

        if (
            any(item.values())
            and unique_key not in itinerary_seen
        ):
            itinerary_seen.add(unique_key)
            itinerary_data["items"].append(item)

else:
    print(
        "CẢNH BÁO: Không tìm thấy phần "
        "'Suggested Itinerary...'"
    )


# ============================================================
# 7. KHỞI TẠO EDGE WEBDRIVER
# ============================================================

def create_edge_driver():
    options = Options()

    options.add_argument("--start-maximized")

    options.add_argument(
        f"user-agent={HEADERS['User-Agent']}"
    )

    options.add_argument(
        "--disable-blink-features=AutomationControlled"
    )

    options.add_experimental_option(
        "excludeSwitches",
        ["enable-automation"]
    )

    options.add_experimental_option(
        "useAutomationExtension",
        False
    )

    driver = webdriver.Edge(options=options)

    driver.set_page_load_timeout(180)

    driver.execute_script(
        """
        Object.defineProperty(
            navigator,
            'webdriver',
            {get: () => undefined}
        );
        """
    )

    return driver


def wait_until_page_ready(driver, timeout=120):
    """
    Chờ trang vượt Cloudflare và có HTML đầy đủ.
    """
    WebDriverWait(driver, timeout).until(
        lambda current_driver: (
            "just a moment"
            not in current_driver.title.lower()

            and "attention required"
            not in current_driver.title.lower()

            and len(current_driver.page_source) > 5000
        )
    )

    WebDriverWait(driver, 30).until(
        lambda current_driver:
        current_driver.execute_script(
            "return document.readyState"
        ) == "complete"
    )

    # Chờ thêm để JavaScript render
    time.sleep(2)


# ============================================================
# 8. CRAWL HTML TỪNG TRANG TRẢI NGHIỆM
# ============================================================

valid_experience_items = [
    item
    for item in experience_data["items"]
    if item.get("detail_url")
]

if valid_experience_items:
    print("\n" + "=" * 70)
    print("BẮT ĐẦU CRAWL CÁC TRANG TRẢI NGHIỆM")
    print(
        "Số trang cần crawl:",
        len(valid_experience_items)
    )
    print("=" * 70)

    driver = create_edge_driver()

    try:
        total = len(experience_data["items"])

        for index, item in enumerate(
            experience_data["items"],
            start=1
        ):
            detail_url = item.get("detail_url")

            print(
                f"\n[{index}/{total}] "
                f"{item.get('title') or 'Không có tiêu đề'}"
            )

            if not detail_url:
                print("Bỏ qua: không có link truy cập.")

                item["crawl_status"] = "skipped_no_url"
                item["saved_html"] = None

                continue

            file_name = make_safe_filename(
                detail_url,
                index
            )

            file_path = DETAIL_HTML_DIR / file_name

            crawl_success = False
            last_error = None

            # Thử tối đa 2 lần
            for attempt in range(1, 3):
                try:
                    print(
                        f"Đang truy cập lần {attempt}:",
                        detail_url
                    )

                    driver.get(detail_url)

                    wait_until_page_ready(
                        driver,
                        timeout=120
                    )

                    detail_html = driver.page_source

                    if len(detail_html) < 5000:
                        raise ValueError(
                            "HTML quá ngắn, trang có thể "
                            "chưa tải hoàn chỉnh."
                        )

                    file_path.write_text(
                        detail_html,
                        encoding="utf-8"
                    )

                    item["crawl_status"] = "success"
                    item["requested_url"] = detail_url
                    item["final_url"] = driver.current_url
                    item["page_title"] = driver.title
                    item["saved_html"] = str(file_path)
                    item["html_length"] = len(detail_html)

                    print("Crawl thành công.")
                    print("Tiêu đề:", driver.title)
                    print("URL cuối:", driver.current_url)
                    print("Đã lưu:", file_path)
                    print(
                        "Độ dài HTML:",
                        len(detail_html)
                    )

                    crawl_success = True
                    break

                except (
                    TimeoutException,
                    WebDriverException,
                    ValueError,
                    Exception
                ) as error:
                    last_error = str(error)

                    print(
                        f"Lỗi lần {attempt}:",
                        last_error
                    )

                    if attempt < 2:
                        print("Đang chờ để thử lại...")
                        time.sleep(4)

            if not crawl_success:
                item["crawl_status"] = "failed"
                item["crawl_error"] = last_error
                item["saved_html"] = None

            # Nghỉ giữa các request
            time.sleep(2)

    finally:
        driver.quit()
        print("\nĐã đóng Edge WebDriver.")

else:
    print(
        "\nKhông tìm thấy link trải nghiệm hợp lệ, "
        "không khởi động WebDriver."
    )


# ============================================================
# 9. GỘP VÀ LƯU KẾT QUẢ JSON
# ============================================================

phu_quoc_data = {
    "source_url": BASE_URL,
    "source_html": str(HTML_FILE),

    "welcome_to_phu_quoc_united_center": (
        welcome_data
    ),

    "exclusive_experiences": (
        experience_data
    ),

    "suggested_itinerary": (
        itinerary_data
    )
}

with OUTPUT_JSON.open(
    "w",
    encoding="utf-8"
) as json_file:
    json.dump(
        phu_quoc_data,
        json_file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 10. IN KẾT QUẢ
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH")
print("=" * 70)

print(
    "Welcome to Phu Quoc United Center:",
    len(welcome_data["items"]),
    "nội dung"
)

print(
    "Exclusive Experiences:",
    len(experience_data["items"]),
    "trải nghiệm"
)

print(
    "Suggested Itinerary:",
    len(itinerary_data["items"]),
    "lịch trình"
)

successful_crawls = sum(
    1
    for item in experience_data["items"]
    if item.get("crawl_status") == "success"
)

failed_crawls = sum(
    1
    for item in experience_data["items"]
    if item.get("crawl_status") == "failed"
)

print("Trang con crawl thành công:", successful_crawls)
print("Trang con crawl thất bại:", failed_crawls)

print("\nThư mục HTML trang con:")
print(DETAIL_HTML_DIR)

print("\nFile JSON:")
print(OUTPUT_JSON)


# In thử dữ liệu lấy được
print("\n" + "=" * 70)
print("DỮ LIỆU EXCLUSIVE EXPERIENCES")
print("=" * 70)

for index, item in enumerate(
    experience_data["items"],
    start=1
):
    print(f"\n--- Trải nghiệm {index} ---")
    print("Tiêu đề:", item.get("title"))
    print("Địa điểm:", item.get("location"))
    print("Mô tả:", item.get("description"))
    print("Ảnh:", item.get("image_url"))
    print("Link:", item.get("detail_url"))
    print("Trạng thái:", item.get("crawl_status"))
    print("HTML:", item.get("saved_html"))

ĐÃ ĐỌC FILE HTML GỐC
File: D:\vinuni\T013\data_crawl\html\phu-quoc.html
Độ dài HTML: 266059
Tiêu đề trang: Điểm đến Phú Quốc_EN – VinWonders

BẮT ĐẦU CRAWL CÁC TRANG TRẢI NGHIỆM
Số trang cần crawl: 9

[1/10] Experience the “Humans in Cages, Animals Roaming Free” Challenge
Đang truy cập lần 1: https://vinwonders.com/vi/wonderpedia/news/top-6-trai-nghiem-tai-vuon-thu-viet-nam/
Crawl thành công.
Tiêu đề: TOP 6 trải nghiệm hấp dẫn tại vườn thú Việt Nam – Vinpearl Safari Phú Quốc – VinWonders
URL cuối: https://vinwonders.com/vi/wonderpedia/news/top-6-trai-nghiem-tai-vuon-thu-viet-nam/
Đã lưu: D:\vinuni\T013\data_crawl\html\phuquoc\01_top-6-trai-nghiem-tai-vuon-thu-viet-nam.html
Độ dài HTML: 321349

[2/10] Explore Night Safari
Đang truy cập lần 1: https://vinwonders.com/vi/night-safari/
Crawl thành công.
Tiêu đề: Tour Night Safari Phú Quốc | Official Website
URL cuối: https://vinwonders.com/vi/night-safari/
Đã lưu: D:\vinuni\T013\data_crawl\html\phuquoc\02_night-safari.html
Độ dài HTML: 2020

In [5]:
# ============================================================
# CRAWL VINWONDERS CUA HOI -> JSON
# Chạy toàn bộ trong 1 cell Jupyter Notebook
# ============================================================

from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import json
import re


# ============================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ============================================================

HTML_FILE = Path(
    r"D:\vinuni\T013\data_crawl\html\nghe-an.html"
)

OUTPUT_JSON = Path(
    r"D:\vinuni\T013\data_crawl\vinwonders_cua_hoi.json"
)

BASE_URL = "https://vinwonders.com/en/vinwonders-cua-hoi/"


# ============================================================
# 2. HÀM HỖ TRỢ
# ============================================================

def clean_text(element):
    """
    Lấy text và loại bỏ khoảng trắng thừa.
    """
    if element is None:
        return None

    if hasattr(element, "get_text"):
        text = element.get_text(" ", strip=True)
    else:
        text = str(element)

    text = re.sub(r"\s+", " ", text).strip()

    return text or None


def absolute_url(url, base_url=BASE_URL):
    """
    Chuyển URL tương đối thành URL tuyệt đối.
    """
    if not url:
        return None

    url = url.strip()

    if not url or url == "#":
        return None

    return urljoin(base_url, url)


def get_image_url(img, base_url=BASE_URL):
    """
    Lấy URL ảnh thật.

    Website sử dụng lazy loading nên ưu tiên:
    data-lazy-src -> data-src -> src.
    """
    if img is None:
        return None

    image_url = (
        img.get("data-lazy-src")
        or img.get("data-src")
        or img.get("data-original")
        or img.get("src")
    )

    # Tránh lấy ảnh placeholder SVG
    if image_url and image_url.startswith("data:image"):
        noscript = img.find_next_sibling("noscript")

        if noscript:
            noscript_soup = BeautifulSoup(
                noscript.decode_contents(),
                "lxml"
            )

            real_img = noscript_soup.find("img")

            if real_img:
                image_url = real_img.get("src")

    return absolute_url(image_url, base_url)


def remove_duplicates(items, fields):
    """
    Loại bỏ item trùng dựa trên các field được truyền vào.
    """
    result = []
    seen = set()

    for item in items:
        key = tuple(item.get(field) for field in fields)

        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


# ============================================================
# 3. ĐỌC HTML
# ============================================================

if not HTML_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file HTML:\n{HTML_FILE}\n\n"
        "Hãy kiểm tra lại đường dẫn HTML_FILE."
    )

html = HTML_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

soup = BeautifulSoup(html, "lxml")

print("=" * 70)
print("ĐÃ ĐỌC HTML")
print("File:", HTML_FILE)
print("Độ dài HTML:", len(html))
print("Tiêu đề:", clean_text(soup.title))
print("=" * 70)


# ============================================================
# 4. REASONS YOU MUST VISIT VINWONDERS CUA HOI
# ============================================================

reasons_section = soup.select_one(
    ".content_reason_to"
)

reasons_data = {
    "title": None,
    "items": []
}

if reasons_section is not None:

    reasons_data["title"] = clean_text(
        reasons_section.select_one("h2.tit_h2")
    )

    reason_cards = reasons_section.select(
        ".address-having_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    for card in reason_cards:

        image_element = card.select_one(
            ".slide-card img.slide-card_img"
        )

        description_element = card.select_one(
            ".slide-card_content p"
        )

        item = {
            "image_url": get_image_url(
                image_element
            ),
            "description": clean_text(
                description_element
            )
        }

        if item["image_url"] or item["description"]:
            reasons_data["items"].append(item)

    # Swiper có các slide clone nên cần loại trùng
    reasons_data["items"] = remove_duplicates(
        reasons_data["items"],
        fields=["image_url", "description"]
    )

else:
    print(
        "CẢNH BÁO: Không tìm thấy phần "
        "'Reasons You Must Visit VinWonders Cua Hoi'."
    )


# ============================================================
# 5. MUST-SEE SHOWS
# ============================================================

shows_section = soup.select_one(
    ".tg_event_top-custom"
)

shows_data = {
    "title": None,
    "items": []
}

if shows_section is not None:

    shows_data["title"] = clean_text(
        shows_section.select_one("h2.tit_h2")
    )

    # --------------------------------------------------------
    # 5.1. Chương trình lớn đầu tiên
    # --------------------------------------------------------

    main_show = shows_section.select_one(
        ".list_event_top-pc .top_special"
    )

    if main_show is not None:

        main_image = main_show.select_one(
            ".wrap_figure img"
        )

        main_name = main_show.select_one(
            ".top-special__text--tittle"
        )

        info_columns = main_show.select(
            ".show-info-row__col"
        )

        show_time = None
        show_location = None

        if len(info_columns) >= 1:
            show_time = clean_text(
                info_columns[0].select_one(
                    ".show-info-row__text"
                )
            )

        if len(info_columns) >= 2:
            show_location = clean_text(
                info_columns[1].select_one(
                    ".show-info-row__text--right"
                )
            )

        item = {
            "name": clean_text(main_name),
            "time": show_time,
            "location": show_location,
            "image_url": get_image_url(main_image)
        }

        if any(item.values()):
            shows_data["items"].append(item)


    # --------------------------------------------------------
    # 5.2. Các chương trình nhỏ còn lại
    # --------------------------------------------------------

    other_show_cards = shows_section.select(
        ".list_event_top-pc "
        ".other-special__card"
    )

    for card in other_show_cards:

        image_element = card.select_one(
            ".other-special__image img"
        )

        name_element = card.select_one(
            ".other-special__title"
        )

        time_element = card.select_one(
            ".other-special__info-row--time span"
        )

        location_element = card.select_one(
            ".other-special__info-row--text span"
        )

        item = {
            "name": clean_text(name_element),
            "time": clean_text(time_element),
            "location": clean_text(
                location_element
            ),
            "image_url": get_image_url(
                image_element
            )
        }

        if any(item.values()):
            shows_data["items"].append(item)


    # --------------------------------------------------------
    # 5.3. Nếu cấu trúc desktop không lấy được,
    #      dùng dữ liệu mobile làm phương án dự phòng
    # --------------------------------------------------------

    if not shows_data["items"]:

        mobile_cards = shows_section.select(
            ".list_event_top-mobile "
            ".other-special__card"
        )

        for card in mobile_cards:

            image_element = card.select_one(
                ".other-special__image img"
            )

            name_element = card.select_one(
                ".other-special__title"
            )

            info_rows = card.select(
                ".other-special__info-row"
            )

            show_time = None
            show_location = None

            if len(info_rows) >= 1:
                show_time = clean_text(
                    info_rows[0].select_one("span")
                )

            if len(info_rows) >= 2:
                show_location = clean_text(
                    info_rows[1].select_one("span")
                )

            item = {
                "name": clean_text(name_element),
                "time": show_time,
                "location": show_location,
                "image_url": get_image_url(
                    image_element
                )
            }

            if any(item.values()):
                shows_data["items"].append(item)


    # Loại dữ liệu trùng giữa desktop và mobile
    shows_data["items"] = remove_duplicates(
        shows_data["items"],
        fields=[
            "name",
            "time",
            "location",
            "image_url"
        ]
    )

else:
    print(
        "CẢNH BÁO: Không tìm thấy phần "
        "'MUST-SEE SHOWS'."
    )


# ============================================================
# 6. THÔNG TIN TRANG
# ============================================================

canonical_element = soup.select_one(
    'link[rel="canonical"]'
)

meta_description_element = soup.select_one(
    'meta[name="description"]'
)

page_information = {
    "page_title": clean_text(soup.title),
    "source_file": str(HTML_FILE),
    "source_url": (
        canonical_element.get("href")
        if canonical_element
        else BASE_URL
    ),
    "description": (
        clean_text(
            meta_description_element.get("content")
        )
        if meta_description_element
        else None
    )
}


# ============================================================
# 7. TỔNG HỢP JSON
# ============================================================

vinwonders_cua_hoi_data = {
    "destination": {
        "name": "VinWonders Cua Hoi",
        "province": "Nghe An",
        "country": "Vietnam"
    },

    "page_information": page_information,

    "sections": {
        "reasons_you_must_visit": reasons_data,
        "must_see_shows": shows_data
    },

    "statistics": {
        "reason_count": len(
            reasons_data["items"]
        ),
        "show_count": len(
            shows_data["items"]
        )
    }
}


# ============================================================
# 8. LƯU JSON
# ============================================================

OUTPUT_JSON.parent.mkdir(
    parents=True,
    exist_ok=True
)

with OUTPUT_JSON.open(
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        vinwonders_cua_hoi_data,
        json_file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 9. IN KẾT QUẢ
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH")
print("=" * 70)

print(
    "Số lý do nên ghé thăm:",
    len(reasons_data["items"])
)

print(
    "Số chương trình biểu diễn:",
    len(shows_data["items"])
)

print("\nFile JSON:")
print(OUTPUT_JSON)


print("\n" + "=" * 70)
print("REASONS YOU MUST VISIT")
print("=" * 70)

for index, item in enumerate(
    reasons_data["items"],
    start=1
):
    print(f"\n--- Lý do {index} ---")
    print("Mô tả:", item["description"])
    print("Ảnh:", item["image_url"])


print("\n" + "=" * 70)
print("MUST-SEE SHOWS")
print("=" * 70)

for index, item in enumerate(
    shows_data["items"],
    start=1
):
    print(f"\n--- Show {index} ---")
    print("Tên:", item["name"])
    print("Thời gian:", item["time"])
    print("Địa điểm:", item["location"])
    print("Ảnh:", item["image_url"])

ĐÃ ĐỌC HTML
File: D:\vinuni\T013\data_crawl\html\nghe-an.html
Độ dài HTML: 291402
Tiêu đề: VinWonders Cua Hoi | Official Website

HOÀN THÀNH
Số lý do nên ghé thăm: 3
Số chương trình biểu diễn: 3

File JSON:
D:\vinuni\T013\data_crawl\vinwonders_cua_hoi.json

REASONS YOU MUST VISIT

--- Lý do 1 ---
Mô tả: Entertainment, Shopping & Dining: A paradise of international standards.
Ảnh: https://static.vinwonders.com/production/2025/09/check-in-vinwonders-cua-hoi.jpg

--- Lý do 2 ---
Mô tả: The first and only sea-crossing cable car in the North Central region of Vietnam
Ảnh: https://static.vinwonders.com/production/cap-treo-vuot-bien-vinwonders-cua-hoi.jpg

--- Lý do 3 ---
Mô tả: ECO-TOURISM AND SPIRITUAL ISLAND
Ảnh: https://static.vinwonders.com/production/2025/03/VWCH_Chua-Song-Ngu-scaled.jpg

MUST-SEE SHOWS

--- Show 1 ---
Tên: The Little Fish
Thời gian: 10:00 - 10:15
Địa điểm: Song Ngu Station
Ảnh: https://static.vinwonders.com/production/2026/01/ca-chep.jpg

--- Show 2 ---
Tên: Vietnamese

In [7]:
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import json
import re


# ============================================================
# 1. CẤU HÌNH
# ============================================================

HTML_FILE = Path(
    r"D:\vinuni\T013\data_crawl\html\ha-noi.html"
)

OUTPUT_JSON = Path(
    r"D:\vinuni\T013\data_crawl\grand_world_ha_noi.json"
)

BASE_URL = "https://vinwonders.com/en/grand-world/"


# ============================================================
# 2. HÀM HỖ TRỢ
# ============================================================

def clean_text(element):
    if element is None:
        return None

    if hasattr(element, "get_text"):
        text = element.get_text(" ", strip=True)
    else:
        text = str(element)

    text = re.sub(r"\s+", " ", text).strip()

    return text or None


def absolute_url(url, base_url=BASE_URL):
    if not url:
        return None

    url = str(url).strip()

    if not url or url == "#":
        return None

    return urljoin(base_url, url)


def get_image_url(img, base_url=BASE_URL):
    if img is None:
        return None

    image_url = (
        img.get("data-lazy-src")
        or img.get("data-src")
        or img.get("data-original")
        or img.get("src")
    )

    if image_url and image_url.startswith("data:image"):
        image_url = None

    if not image_url:
        srcset = (
            img.get("data-srcset")
            or img.get("srcset")
        )

        if srcset:
            image_url = (
                srcset.split(",")[0]
                .strip()
                .split()[0]
            )

    if not image_url:
        noscript = img.find_next_sibling("noscript")

        if noscript:
            noscript_soup = BeautifulSoup(
                noscript.decode_contents(),
                "lxml"
            )

            real_img = noscript_soup.find("img")

            if real_img:
                image_url = (
                    real_img.get("src")
                    or real_img.get("data-src")
                )

    return absolute_url(image_url, base_url)


def remove_duplicates(items, fields):
    result = []
    seen = set()

    for item in items:
        key = tuple(
            item.get(field)
            for field in fields
        )

        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


def get_meta_content(soup, selector):
    element = soup.select_one(selector)

    if element is None:
        return None

    return clean_text(
        element.get("content")
    )


# ============================================================
# 3. ĐỌC HTML
# ============================================================

if not HTML_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file HTML:\n{HTML_FILE}"
    )

html = HTML_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

soup = BeautifulSoup(html, "lxml")

canonical_element = soup.select_one(
    'link[rel="canonical"]'
)

source_url = (
    canonical_element.get("href")
    if canonical_element
    and canonical_element.get("href")
    else BASE_URL
)

print("=" * 70)
print("ĐÃ ĐỌC HTML")
print("Tiêu đề:", clean_text(soup.title))
print("URL:", source_url)
print("Độ dài HTML:", len(html))
print("=" * 70)


# ============================================================
# 4. REASONS TO VISIT GRAND WORLD
# ============================================================

reasons_data = {
    "title": None,
    "items": []
}

# Tìm đúng section có heading REASONS TO VISIT GRAND WORLD
reason_heading = soup.find(
    ["h1", "h2", "h3"],
    string=lambda text: (
        text
        and "REASONS TO VISIT GRAND WORLD"
        in re.sub(r"\s+", " ", text).upper()
    )
)

reasons_section = None

if reason_heading:
    reasons_section = reason_heading.find_parent(
        class_="content_reason_to"
    )

    if reasons_section is None:
        reasons_section = reason_heading.find_parent(
            ["section", "div"]
        )

if reasons_section is not None:

    reasons_data["title"] = clean_text(
        reason_heading
    )

    reason_cards = reasons_section.select(
        ".address-having_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    if not reason_cards:
        reason_cards = reasons_section.select(
            ".swiper-wrapper > .swiper-slide"
        )

    for card in reason_cards:

        image_element = card.select_one(
            ".slide-card img.slide-card_img, "
            "img.slide-card_img, "
            ".slide-card img, "
            "img"
        )

        description_element = card.select_one(
            ".slide-card_content_title, "
            ".slide-card_content p, "
            ".slide-card_content"
        )

        item = {
            "image_url": get_image_url(
                image_element,
                source_url
            ),
            "description": clean_text(
                description_element
            )
        }

        if item["image_url"] or item["description"]:
            reasons_data["items"].append(item)

    reasons_data["items"] = remove_duplicates(
        reasons_data["items"],
        fields=[
            "image_url",
            "description"
        ]
    )

else:
    print(
        "Không tìm thấy phần "
        "'REASONS TO VISIT GRAND WORLD'."
    )


# ============================================================
# 5. MUST-SEE EVENTS
# ============================================================

events_data = {
    "title": None,
    "items": []
}

events_section = soup.select_one(
    ".tg_event_top-custom"
)

if events_section is not None:

    events_data["title"] = clean_text(
        events_section.select_one("h2.tit_h2")
    )

    # --------------------------------------------------------
    # 5.1. EVENT CHÍNH
    # --------------------------------------------------------

    main_event = events_section.select_one(
        ".list_event_top-pc .top_special"
    )

    if main_event is not None:

        item = {
            "name": clean_text(
                main_event.select_one(
                    ".top-special__text--tittle"
                )
            ),
            "time": clean_text(
                main_event.select_one(
                    ".show-info-row__text"
                )
            ),
            "location": clean_text(
                main_event.select_one(
                    ".show-info-row__text--right"
                )
            ),
            "image_url": get_image_url(
                main_event.select_one(
                    ".wrap_figure img"
                ),
                source_url
            )
        }

        if any(item.values()):
            events_data["items"].append(item)


    # --------------------------------------------------------
    # 5.2. CÁC EVENT NHỎ
    # --------------------------------------------------------

    other_cards = events_section.select(
        ".list_event_top-pc "
        ".other-special__card"
    )

    for card in other_cards:

        item = {
            "name": clean_text(
                card.select_one(
                    ".other-special__title"
                )
            ),
            "time": clean_text(
                card.select_one(
                    ".other-special__info-row--time span"
                )
            ),
            "location": clean_text(
                card.select_one(
                    ".other-special__info-row--text span"
                )
            ),
            "image_url": get_image_url(
                card.select_one(
                    ".other-special__image img"
                ),
                source_url
            )
        }

        if any(item.values()):
            events_data["items"].append(item)


    # --------------------------------------------------------
    # 5.3. MOBILE DỰ PHÒNG
    # --------------------------------------------------------

    mobile_cards = events_section.select(
        ".list_event_top-mobile "
        ".other-special__card"
    )

    for card in mobile_cards:

        item = {
            "name": clean_text(
                card.select_one(
                    ".other-special__title"
                )
            ),
            "time": clean_text(
                card.select_one(
                    ".other-special__info-row--time span"
                )
            ),
            "location": clean_text(
                card.select_one(
                    ".other-special__info-row--text span"
                )
            ),
            "image_url": get_image_url(
                card.select_one(
                    ".other-special__image img"
                ),
                source_url
            )
        }

        if any(item.values()):
            events_data["items"].append(item)

    events_data["items"] = remove_duplicates(
        events_data["items"],
        fields=[
            "name",
            "time",
            "location"
        ]
    )

else:
    print(
        "Không tìm thấy phần Must-see Events."
    )


# ============================================================
# 6. PAGE INFORMATION
# ============================================================

page_information = {
    "page_title": clean_text(
        soup.title
    ),

    "source_file": str(
        HTML_FILE
    ),

    "source_url": source_url,

    "language": (
        soup.html.get("lang")
        if soup.html
        else None
    ),

    "description": get_meta_content(
        soup,
        'meta[name="description"]'
    ),

    "og_title": get_meta_content(
        soup,
        'meta[property="og:title"]'
    ),

    "og_description": get_meta_content(
        soup,
        'meta[property="og:description"]'
    )
}


# ============================================================
# 7. TỔNG HỢP JSON
# ============================================================

hanoi_data = {
    "destination": {
        "name": "Grand World Ocean City",
        "city": "Hanoi",
        "country": "Vietnam"
    },

    "page_information": page_information,

    "sections": {
        "reasons_to_visit_grand_world": {
            "title": reasons_data["title"],
            "items": reasons_data["items"]
        },

        "must_see_events": {
            "title": events_data["title"],
            "items": events_data["items"]
        }
    },

    "statistics": {
        "reason_count": len(
            reasons_data["items"]
        ),

        "event_count": len(
            events_data["items"]
        )
    }
}


# ============================================================
# 8. LƯU JSON
# ============================================================

OUTPUT_JSON.parent.mkdir(
    parents=True,
    exist_ok=True
)

with OUTPUT_JSON.open(
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        hanoi_data,
        json_file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 9. IN KẾT QUẢ
# ============================================================

print("\n" + "=" * 70)
print("HOÀN THÀNH")
print("=" * 70)

print(
    "Reasons:",
    len(reasons_data["items"])
)

print(
    "Events:",
    len(events_data["items"])
)

print("\nFile JSON:")
print(OUTPUT_JSON)


print("\n" + "=" * 70)
print("REASONS TO VISIT GRAND WORLD")
print("=" * 70)

for index, item in enumerate(
    reasons_data["items"],
    start=1
):
    print(f"\n--- Reason {index} ---")
    print(
        "Description:",
        item["description"]
    )
    print(
        "Image:",
        item["image_url"]
    )


print("\n" + "=" * 70)
print("MUST-SEE EVENTS")
print("=" * 70)

for index, item in enumerate(
    events_data["items"],
    start=1
):
    print(f"\n--- Event {index} ---")
    print("Name:", item["name"])
    print("Time:", item["time"])
    print("Location:", item["location"])
    print("Image:", item["image_url"])

ĐÃ ĐỌC HTML
Tiêu đề: Grand World Ocean City East of Hanoi | Official Website
URL: https://vinwonders.com/en/grand-world/
Độ dài HTML: 278223

HOÀN THÀNH
Reasons: 3
Events: 9

File JSON:
D:\vinuni\T013\data_crawl\grand_world_ha_noi.json

REASONS TO VISIT GRAND WORLD

--- Reason 1 ---
Description: A destination for grand festivals, combining the essence of Asian-European culture, art, and architecture
Image: https://static.vinwonders.com/production/su-kien-scaled-1.jpg

--- Reason 2 ---
Description: The Venice - Riverside Western Market
Image: None

--- Reason 3 ---
Description: The K-Town - Coastal Eastern Street
Image: https://static.vinwonders.com/production/k-town.jpg

MUST-SEE EVENTS

--- Event 1 ---
Name: The Grand Voyage
Time: 21:00 - 22:00
Location: Venice river
Image: https://static.vinwonders.com/production/grand-voyage-2.jpg

--- Event 2 ---
Name: Gondola ride
Time: 10:00 - 20:00
Location: Venice river
Image: https://static.vinwonders.com/production/Thuyen-Gondola-scaled-1.jpg

In [8]:
# ============================================================
# CRAWL HTML NAM HỘI AN + HẢI PHÒNG BẰNG SELENIUM EDGE
# Chạy toàn bộ trong 1 cell Jupyter Notebook
# ============================================================

from pathlib import Path
import time

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# 1. CẤU HÌNH
# ============================================================

OUTPUT_DIR = Path(
    r"D:\vinuni\T013\data_crawl"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36 "
        "Edg/150.0.0.0"
    )
}

PAGES = [
    {
        "url": (
            "https://vinwonders.com/en/"
            "nam-hoi-an-destination/"
        ),
        "file_name": "nam_hoi_an.html"
    },
    {
        "url": (
            "https://vinwonders.com/en/"
            "vu-yen-royal-island/"
        ),
        "file_name": "hai_phong.html"
    }
]


# ============================================================
# 2. KHỞI TẠO EDGE WEBDRIVER
# ============================================================

options = Options()

options.add_argument("--start-maximized")

options.add_argument(
    f"user-agent={HEADERS['User-Agent']}"
)

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)

options.add_experimental_option(
    "useAutomationExtension",
    False
)

driver = webdriver.Edge(
    options=options
)

driver.set_page_load_timeout(180)

driver.execute_script(
    """
    Object.defineProperty(
        navigator,
        'webdriver',
        {get: () => undefined}
    );
    """
)


# ============================================================
# 3. HÀM CHỜ TRANG TẢI
# ============================================================

def wait_until_page_ready(
    driver,
    timeout=120
):
    """
    Chờ trang vượt Cloudflare và có HTML đầy đủ.
    """

    WebDriverWait(
        driver,
        timeout
    ).until(
        lambda current_driver: (
            "just a moment"
            not in current_driver.title.lower()

            and "attention required"
            not in current_driver.title.lower()

            and "checking your browser"
            not in current_driver.title.lower()

            and len(
                current_driver.page_source
            ) > 5000
        )
    )

    WebDriverWait(
        driver,
        30
    ).until(
        lambda current_driver:
        current_driver.execute_script(
            "return document.readyState"
        ) == "complete"
    )

    # Chờ thêm để JavaScript và lazy-load render
    time.sleep(3)


# ============================================================
# 4. CRAWL VÀ LƯU HTML
# ============================================================

crawl_results = []

try:

    total_pages = len(PAGES)

    for index, page in enumerate(
        PAGES,
        start=1
    ):

        url = page["url"]
        file_name = page["file_name"]
        output_file = OUTPUT_DIR / file_name

        print("\n" + "=" * 70)
        print(
            f"[{index}/{total_pages}] "
            f"ĐANG CRAWL"
        )
        print("URL:", url)
        print("=" * 70)

        success = False
        last_error = None

        # Thử tối đa 2 lần nếu lỗi
        for attempt in range(1, 3):

            try:

                print(
                    f"Lần thử {attempt}..."
                )

                driver.get(url)

                wait_until_page_ready(
                    driver,
                    timeout=120
                )

                html = driver.page_source

                if len(html) < 5000:
                    raise ValueError(
                        "HTML quá ngắn, trang có thể "
                        "chưa tải hoàn chỉnh."
                    )

                output_file.write_text(
                    html,
                    encoding="utf-8"
                )

                result = {
                    "requested_url": url,
                    "final_url": driver.current_url,
                    "page_title": driver.title,
                    "file_path": str(output_file),
                    "html_length": len(html),
                    "status": "success"
                }

                crawl_results.append(
                    result
                )

                print("Crawl thành công.")
                print(
                    "Tiêu đề:",
                    driver.title
                )
                print(
                    "URL cuối:",
                    driver.current_url
                )
                print(
                    "Độ dài HTML:",
                    len(html)
                )
                print(
                    "Đã lưu tại:",
                    output_file
                )

                success = True
                break

            except (
                TimeoutException,
                WebDriverException,
                ValueError,
                Exception
            ) as error:

                last_error = str(error)

                print(
                    f"Lỗi lần {attempt}:",
                    last_error
                )

                if attempt < 2:
                    print(
                        "Đang chờ để thử lại..."
                    )
                    time.sleep(5)

        if not success:

            crawl_results.append(
                {
                    "requested_url": url,
                    "file_path": str(
                        output_file
                    ),
                    "status": "failed",
                    "error": last_error
                }
            )

            print(
                "Không crawl được trang:",
                url
            )

        # Nghỉ giữa hai lần truy cập
        time.sleep(3)

finally:

    driver.quit()

    print("\nĐã đóng Edge WebDriver.")


# ============================================================
# 5. IN KẾT QUẢ TỔNG HỢP
# ============================================================

print("\n" + "=" * 70)
print("KẾT QUẢ CRAWL")
print("=" * 70)

for result in crawl_results:

    print(
        "\nTrạng thái:",
        result.get("status")
    )

    print(
        "URL:",
        result.get("requested_url")
    )

    print(
        "File:",
        result.get("file_path")
    )

    if result.get("status") == "success":

        print(
            "Tiêu đề:",
            result.get("page_title")
        )

        print(
            "Độ dài HTML:",
            result.get("html_length")
        )

    else:

        print(
            "Lỗi:",
            result.get("error")
        )


[1/2] ĐANG CRAWL
URL: https://vinwonders.com/en/nam-hoi-an-destination/
Lần thử 1...
Crawl thành công.
Tiêu đề: Nam Hoi An Destination – VinWonders
URL cuối: https://vinwonders.com/en/nam-hoi-an-destination/
Độ dài HTML: 281302
Đã lưu tại: D:\vinuni\T013\data_crawl\nam_hoi_an.html

[2/2] ĐANG CRAWL
URL: https://vinwonders.com/en/vu-yen-royal-island/
Lần thử 1...
Crawl thành công.
Tiêu đề: Vu Yen Royal Island | Golf, Safari & Entertainment in Hai Phong
URL cuối: https://vinwonders.com/en/vu-yen-royal-island/
Độ dài HTML: 205167
Đã lưu tại: D:\vinuni\T013\data_crawl\hai_phong.html

Đã đóng Edge WebDriver.

KẾT QUẢ CRAWL

Trạng thái: success
URL: https://vinwonders.com/en/nam-hoi-an-destination/
File: D:\vinuni\T013\data_crawl\nam_hoi_an.html
Tiêu đề: Nam Hoi An Destination – VinWonders
Độ dài HTML: 281302

Trạng thái: success
URL: https://vinwonders.com/en/vu-yen-royal-island/
File: D:\vinuni\T013\data_crawl\hai_phong.html
Tiêu đề: Vu Yen Royal Island | Golf, Safari & Entertainment in H

In [11]:
# ============================================================
# NAM HOI AN:
# LẤY DỮ LIỆU 3 SECTION + CRAWL TỪNG LINK
#
# MỖI URL:
# - MỞ 1 WEBDRIVER MỚI
# - LẤY VÀ LƯU HTML
# - ĐÓNG WEBDRIVER NGAY
#
# KHÔNG TẠO JSON
# KHÔNG TIME.SLEEP GIỮA CÁC URL
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import re

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# 1. CẤU HÌNH
# ============================================================

HTML_FILE = Path(
    r"D:\vinuni\T013\data_crawl\nam_hoi_an.html"
)

DETAIL_HTML_DIR = Path(
    r"D:\vinuni\T013\data_crawl\html\nam_hoi_an"
)

BASE_URL = (
    "https://vinwonders.com/en/"
    "nam-hoi-an-destination/"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36 "
        "Edg/150.0.0.0"
    )
}

DETAIL_HTML_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. HÀM HỖ TRỢ BEAUTIFULSOUP
# ============================================================

def clean_text(element):
    if element is None:
        return None

    if hasattr(element, "get_text"):
        text = element.get_text(
            " ",
            strip=True
        )
    else:
        text = str(element)

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text or None


def absolute_url(
    url,
    base_url=BASE_URL
):
    if not url:
        return None

    url = str(url).strip()

    if (
        not url
        or url == "#"
        or url.lower().startswith(
            (
                "javascript:",
                "mailto:",
                "tel:"
            )
        )
    ):
        return None

    return urljoin(
        base_url,
        url
    )


def get_image_url(
    image_element,
    base_url=BASE_URL
):
    if image_element is None:
        return None

    image_url = (
        image_element.get("data-lazy-src")
        or image_element.get("data-src")
        or image_element.get("data-original")
        or image_element.get("src")
    )

    if (
        image_url
        and image_url.startswith("data:image")
    ):
        image_url = None

    if not image_url:
        srcset = (
            image_element.get("data-srcset")
            or image_element.get("srcset")
        )

        if srcset:
            image_url = (
                srcset
                .split(",")[0]
                .strip()
                .split()[0]
            )

    if not image_url:
        noscript = (
            image_element.find_next_sibling(
                "noscript"
            )
        )

        if noscript:
            noscript_soup = BeautifulSoup(
                noscript.decode_contents(),
                "lxml"
            )

            real_image = noscript_soup.find(
                "img"
            )

            if real_image:
                image_url = (
                    real_image.get("src")
                    or real_image.get(
                        "data-src"
                    )
                )

    return absolute_url(
        image_url,
        base_url
    )


def remove_duplicates(
    items,
    fields
):
    result = []
    seen = set()

    for item in items:
        key = tuple(
            item.get(field)
            for field in fields
        )

        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


def make_safe_filename(
    url,
    index,
    prefix
):
    parsed = urlparse(url)

    slug = (
        parsed.path
        .strip("/")
        .split("/")[-1]
    )

    if not slug:
        slug = parsed.netloc

    slug = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "-",
        slug
    ).strip("-").lower()

    if not slug:
        slug = "page"

    return (
        f"{prefix}_{index:02d}_"
        f"{slug}.html"
    )


# ============================================================
# 3. ĐỌC HTML GỐC
# ============================================================

if not HTML_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file:\n"
        f"{HTML_FILE}"
    )

html = HTML_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

soup = BeautifulSoup(
    html,
    "lxml"
)

canonical = soup.select_one(
    'link[rel="canonical"]'
)

source_url = (
    canonical.get("href")
    if (
        canonical
        and canonical.get("href")
    )
    else BASE_URL
)

print("=" * 75)
print("ĐÃ ĐỌC HTML NAM HỘI AN")
print("Tiêu đề:", clean_text(soup.title))
print("URL:", source_url)
print("Độ dài HTML:", len(html))
print("=" * 75)


# ============================================================
# 4. WELCOME TO THE LAND OF HERITAGE
# Chỉ lấy dữ liệu, không có link trang con
# ============================================================

welcome_items = []

welcome_section = soup.select_one(
    ".address-having"
)

if welcome_section:

    welcome_cards = welcome_section.select(
        ".address-having_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    for card in welcome_cards:

        item = {
            "image_url": get_image_url(
                card.select_one(
                    "img.slide-card_img"
                ),
                source_url
            ),

            "title": clean_text(
                card.select_one(
                    ".slide-card_content_title"
                )
            ),

            "description": clean_text(
                card.select_one(
                    ".slide-card_content_description"
                )
            )
        }

        if any(item.values()):
            welcome_items.append(item)

    welcome_items = remove_duplicates(
        welcome_items,
        fields=[
            "title",
            "image_url"
        ]
    )

else:
    print(
        "Không tìm thấy phần "
        "Welcome to the LAND OF HERITAGE."
    )


# ============================================================
# 5. CHALLENGE YOURSELF TO 13 UNMISSABLE EXPERIENCES
# ============================================================

experience_items = []

experience_section = soup.select_one(
    ".experience_event"
)

if experience_section:

    experience_cards = experience_section.select(
        ".experience_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    if not experience_cards:
        experience_cards = (
            experience_section.select(
                ".swiper-slide.experience-slide"
            )
        )

    for card in experience_cards:

        anchor = card.select_one(
            ".slide-card > a[href]"
        )

        detail_url = None

        if anchor:
            detail_url = absolute_url(
                anchor.get("href"),
                source_url
            )

        item = {
            "image_url": get_image_url(
                card.select_one(
                    ".slide-card > a "
                    "img.slide-card_img"
                ),
                source_url
            ),

            "address": clean_text(
                card.select_one(
                    ".slide-card_content_add"
                )
            ),

            "title": clean_text(
                card.select_one(
                    ".slide-card_content_title"
                )
            ),

            "description": clean_text(
                card.select_one(
                    ".slide-card_content_description"
                )
            ),

            "detail_url": detail_url
        }

        if any(item.values()):
            experience_items.append(item)

    experience_items = remove_duplicates(
        experience_items,
        fields=[
            "title",
            "detail_url"
        ]
    )

else:
    print(
        "Không tìm thấy phần "
        "Challenge Yourself to "
        "13 Unmissable Experiences."
    )


# ============================================================
# 6. JOURNEY OF EXPERIENCES
# ============================================================

journey_items = []

journey_section = soup.select_one(
    "#news-and-experiences"
)

if journey_section:

    journey_cards = journey_section.select(
        ".trip_card-item"
    )

    for card in journey_cards:

        anchor = card.select_one(
            ".trip-card_card-desc_button "
            "a[href]"
        )

        detail_url = None

        if anchor:
            detail_url = absolute_url(
                anchor.get("href"),
                source_url
            )

        item = {
            "image_url": get_image_url(
                card.select_one(
                    "img.trip_card-item-img"
                ),
                source_url
            ),

            "title": clean_text(
                card.select_one(
                    ".trip-card_card-desc_title"
                )
            ),

            "description": clean_text(
                card.select_one(
                    ".trip-card_card-desc_description"
                )
            ),

            "detail_url": detail_url
        }

        if any(item.values()):
            journey_items.append(item)

    journey_items = remove_duplicates(
        journey_items,
        fields=[
            "title",
            "detail_url"
        ]
    )

else:
    print(
        "Không tìm thấy phần "
        "Journey of Experiences."
    )


# ============================================================
# 7. GỘP CÁC URL CẦN CRAWL
# ============================================================

crawl_targets = []


for index, item in enumerate(
    experience_items,
    start=1
):
    if item.get("detail_url"):

        crawl_targets.append({
            "section": "experience",
            "index": index,
            "title": item.get("title"),
            "url": item["detail_url"]
        })


for index, item in enumerate(
    journey_items,
    start=1
):
    if item.get("detail_url"):

        crawl_targets.append({
            "section": "journey",
            "index": index,
            "title": item.get("title"),
            "url": item["detail_url"]
        })


# Loại URL trùng
unique_targets = []
seen_urls = set()

for target in crawl_targets:

    normalized_url = (
        target["url"]
        .split("#")[0]
        .rstrip("/")
    )

    if normalized_url not in seen_urls:
        seen_urls.add(normalized_url)
        unique_targets.append(target)

crawl_targets = unique_targets


# ============================================================
# 8. TẠO WEBDRIVER MỚI
# ============================================================

def create_edge_driver():
    options = Options()

    options.add_argument(
        "--start-maximized"
    )

    options.add_argument(
        f"user-agent={HEADERS['User-Agent']}"
    )

    options.add_argument(
        "--disable-blink-features="
        "AutomationControlled"
    )

    options.add_experimental_option(
        "excludeSwitches",
        ["enable-automation"]
    )

    options.add_experimental_option(
        "useAutomationExtension",
        False
    )

    driver = webdriver.Edge(
        options=options
    )

    driver.set_page_load_timeout(
        180
    )

    return driver


# ============================================================
# 9. KIỂM TRA HTML THẬT
# ============================================================

def is_block_page(driver):
    title = (
        driver.title or ""
    ).lower()

    page_html = (
        driver.page_source or ""
    ).lower()

    signals = [
        "just a moment",
        "performing security verification",
        "verify you are human",
        "attention required",
        "checking your browser"
    ]

    return any(
        signal in title
        or signal in page_html
        for signal in signals
    )


def wait_until_loaded(
    driver,
    timeout=120
):
    """
    Không dùng time.sleep().
    Chỉ dùng explicit wait cho đến khi
    HTML thật được tải.
    """

    WebDriverWait(
        driver,
        timeout
    ).until(
        lambda current_driver: (
            current_driver.execute_script(
                "return document.readyState"
            ) == "complete"

            and not is_block_page(
                current_driver
            )

            and len(
                current_driver.page_source
            ) > 5000
        )
    )


# ============================================================
# 10. CRAWL MỘT URL
#
# QUY TRÌNH:
# - TẠO DRIVER
# - MỞ URL
# - LƯU HTML
# - QUIT NGAY
# ============================================================

def crawl_one_url(
    url,
    output_file
):
    driver = None

    try:
        driver = create_edge_driver()

        driver.get(url)

        wait_until_loaded(
            driver,
            timeout=120
        )

        detail_html = driver.page_source

        if is_block_page(driver):
            raise ValueError(
                "Trang đang bị chặn bởi "
                "security verification."
            )

        if len(detail_html) < 5000:
            raise ValueError(
                "HTML quá ngắn hoặc chưa "
                "tải hoàn chỉnh."
            )

        output_file.write_text(
            detail_html,
            encoding="utf-8"
        )

        return {
            "status": "success",
            "requested_url": url,
            "final_url": driver.current_url,
            "page_title": driver.title,
            "saved_html": str(output_file),
            "html_length": len(detail_html)
        }

    except (
        TimeoutException,
        WebDriverException,
        ValueError,
        Exception
    ) as error:

        return {
            "status": "failed",
            "requested_url": url,
            "saved_html": None,
            "error": str(error)
        }

    finally:
        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass


# ============================================================
# 11. CRAWL TỪNG URL
#
# MỖI VÒNG LẶP GỌI crawl_one_url()
# -> TẠO DRIVER MỚI
# -> LƯU HTML
# -> QUIT
# ============================================================

crawl_results = []

print("\n" + "=" * 75)
print("BẮT ĐẦU CRAWL")
print(
    "Mỗi URL dùng một WebDriver mới."
)
print(
    "Không có thời gian nghỉ giữa các URL."
)
print(
    "Số URL:",
    len(crawl_targets)
)
print("=" * 75)


for crawl_number, target in enumerate(
    crawl_targets,
    start=1
):
    section = target["section"]
    item_index = target["index"]
    title = target.get("title")
    url = target["url"]

    prefix = (
        "experience"
        if section == "experience"
        else "journey"
    )

    file_name = make_safe_filename(
        url=url,
        index=item_index,
        prefix=prefix
    )

    output_file = (
        DETAIL_HTML_DIR
        / file_name
    )

    print("\n" + "-" * 75)
    print(
        f"[{crawl_number}/"
        f"{len(crawl_targets)}]"
    )
    print("Section:", section)
    print("Title:", title)
    print("URL:", url)
    print("Output:", output_file)
    print("-" * 75)

    result = crawl_one_url(
        url=url,
        output_file=output_file
    )

    result["section"] = section
    result["title"] = title

    crawl_results.append(result)

    print(
        "Status:",
        result["status"]
    )

    if result["status"] == "success":
        print(
            "Page title:",
            result["page_title"]
        )
        print(
            "HTML length:",
            result["html_length"]
        )
        print(
            "Saved:",
            result["saved_html"]
        )
        print(
            "WebDriver đã đóng."
        )
    else:
        print(
            "Error:",
            result["error"]
        )
        print(
            "WebDriver đã đóng."
        )


# ============================================================
# 12. IN KẾT QUẢ
# ============================================================

success_count = sum(
    result["status"] == "success"
    for result in crawl_results
)

failed_count = sum(
    result["status"] == "failed"
    for result in crawl_results
)

print("\n" + "=" * 75)
print("HOÀN THÀNH")
print("=" * 75)

print(
    "Welcome items:",
    len(welcome_items)
)

print(
    "Experience items:",
    len(experience_items)
)

print(
    "Journey items:",
    len(journey_items)
)

print(
    "Tổng URL:",
    len(crawl_results)
)

print(
    "Thành công:",
    success_count
)

print(
    "Thất bại:",
    failed_count
)

print("\nThư mục lưu HTML:")
print(DETAIL_HTML_DIR)

print("\nChưa tạo JSON.")

ĐÃ ĐỌC HTML NAM HỘI AN
Tiêu đề: Nam Hoi An Destination – VinWonders
URL: https://vinwonders.com/en/nam-hoi-an-destination/
Độ dài HTML: 281302

BẮT ĐẦU CRAWL
Mỗi URL dùng một WebDriver mới.
Không có thời gian nghỉ giữa các URL.
Số URL: 12

---------------------------------------------------------------------------
[1/12]
Section: experience
Title: Take on the “River Safari Adventure” Challenge
URL: https://vinwonders.com/en/wonderpedia/news/lich-trinh-kham-pha-river-safari-nam-hoi-an/
Output: D:\vinuni\T013\data_crawl\html\nam_hoi_an\experience_01_lich-trinh-kham-pha-river-safari-nam-hoi-an.html
---------------------------------------------------------------------------
Status: success
Page title: Khám phá River Safari Nam Hội An – vườn thú trên sông có 1-0-2 – VinWonders
HTML length: 231200
Saved: D:\vinuni\T013\data_crawl\html\nam_hoi_an\experience_01_lich-trinh-kham-pha-river-safari-nam-hoi-an.html
WebDriver đã đóng.

-----------------------------------------------------------------

In [13]:
# ============================================================
# TỔNG HỢP VU YEN ROYAL ISLAND THÀNH JSON
#
# Bao gồm:
# 1. Welcome to Vu Yen Royal Island
#    - image_url
#    - title
#    - description
#
# 2. 11 Trendy Experiences at Vu Yen Royal Island
#    - image_url
#    - title
#    - location
#    - description
#
# Không crawl trang con
# Không dùng Selenium
# ============================================================

from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import json
import re


# ============================================================
# 1. CẤU HÌNH
# ============================================================

HTML_FILE = Path(
    r"D:\vinuni\T013\data_crawl\hai_phong.html"
)

OUTPUT_JSON = Path(
    r"D:\vinuni\T013\data_crawl\hai_phong_data.json"
)

BASE_URL = (
    "https://vinwonders.com/en/"
    "vu-yen-royal-island/"
)


# ============================================================
# 2. HÀM HỖ TRỢ
# ============================================================

def clean_text(value):
    """
    Lấy nội dung text và loại bỏ khoảng trắng thừa.
    """
    if value is None:
        return None

    if hasattr(value, "get_text"):
        value = value.get_text(
            " ",
            strip=True
        )

    value = re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

    return value or None


def absolute_url(
    url,
    base_url=BASE_URL
):
    """
    Chuyển URL tương đối thành URL tuyệt đối.
    """
    if not url:
        return None

    url = str(url).strip()

    if (
        not url
        or url == "#"
        or url.startswith("data:image")
    ):
        return None

    return urljoin(
        base_url,
        url
    )


def get_image_url(
    image_element,
    base_url=BASE_URL
):
    """
    Lấy URL ảnh thật, hỗ trợ lazy loading.
    """
    if image_element is None:
        return None

    image_url = (
        image_element.get("data-lazy-src")
        or image_element.get("data-src")
        or image_element.get("data-original")
        or image_element.get("src")
    )

    if (
        image_url
        and image_url.startswith("data:image")
    ):
        image_url = None

    if not image_url:
        srcset = (
            image_element.get("data-srcset")
            or image_element.get("srcset")
        )

        if srcset:
            image_url = (
                srcset
                .split(",")[0]
                .strip()
                .split()[0]
            )

    if not image_url:
        noscript = (
            image_element.find_next_sibling(
                "noscript"
            )
        )

        if noscript:
            noscript_soup = BeautifulSoup(
                noscript.decode_contents(),
                "lxml"
            )

            real_image = noscript_soup.find(
                "img"
            )

            if real_image:
                image_url = (
                    real_image.get("src")
                    or real_image.get(
                        "data-src"
                    )
                )

    return absolute_url(
        image_url,
        base_url
    )


def remove_duplicates(
    items,
    fields
):
    """
    Loại bỏ dữ liệu trùng do Swiper clone.
    """
    result = []
    seen = set()

    for item in items:
        key = tuple(
            item.get(field)
            for field in fields
        )

        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


def get_meta_content(
    soup,
    selector
):
    """
    Lấy thuộc tính content của thẻ meta.
    """
    element = soup.select_one(
        selector
    )

    if element is None:
        return None

    return clean_text(
        element.get("content")
    )


# ============================================================
# 3. ĐỌC HTML
# ============================================================

if not HTML_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file HTML:\n"
        f"{HTML_FILE}"
    )

html = HTML_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

soup = BeautifulSoup(
    html,
    "lxml"
)

canonical_element = soup.select_one(
    'link[rel="canonical"]'
)

source_url = (
    canonical_element.get("href")
    if (
        canonical_element
        and canonical_element.get("href")
    )
    else BASE_URL
)

print("=" * 75)
print("ĐÃ ĐỌC HTML VU YEN ROYAL ISLAND")
print("File:", HTML_FILE)
print("Tiêu đề:", clean_text(soup.title))
print("URL:", source_url)
print("Độ dài HTML:", len(html))
print("=" * 75)


# ============================================================
# 4. WELCOME TO VU YEN ROYAL ISLAND
# ============================================================

welcome_data = {
    "title": None,
    "items": []
}

welcome_section = soup.select_one(
    ".address-having"
)

if welcome_section is not None:

    welcome_data["title"] = clean_text(
        welcome_section.select_one(
            ".address-having_title"
        )
    )

    welcome_cards = welcome_section.select(
        ".address-having_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    if not welcome_cards:
        welcome_cards = (
            welcome_section.select(
                ".swiper-wrapper > .swiper-slide"
            )
        )

    for index, card in enumerate(
        welcome_cards,
        start=1
    ):

        item = {
            "id": index,

            "image_url": get_image_url(
                card.select_one(
                    ".slide-card img.slide-card_img, "
                    "img.slide-card_img"
                ),
                source_url
            ),

            "title": clean_text(
                card.select_one(
                    ".slide-card_content_title"
                )
            ),

            "description": clean_text(
                card.select_one(
                    ".slide-card_content_description"
                )
            )
        }

        if any(
            value
            for key, value in item.items()
            if key != "id"
        ):
            welcome_data["items"].append(
                item
            )

    welcome_data["items"] = (
        remove_duplicates(
            welcome_data["items"],
            fields=[
                "title",
                "image_url"
            ]
        )
    )

    # Đánh lại ID sau khi loại trùng
    for index, item in enumerate(
        welcome_data["items"],
        start=1
    ):
        item["id"] = index

else:
    print(
        "Không tìm thấy phần "
        "'Welcome to Vu Yen Royal Island'."
    )


# ============================================================
# 5. 11 TRENDY EXPERIENCES AT VU YEN ROYAL ISLAND
# ============================================================

experiences_data = {
    "title": None,
    "items": []
}

experience_section = soup.select_one(
    ".experience_event"
)

if experience_section is not None:

    experiences_data["title"] = clean_text(
        experience_section.select_one(
            ".experience_title"
        )
    )

    experience_cards = experience_section.select(
        ".experience_slide "
        ".swiper-wrapper > .swiper-slide"
    )

    if not experience_cards:
        experience_cards = (
            experience_section.select(
                ".swiper-slide.experience-slide"
            )
        )

    for index, card in enumerate(
        experience_cards,
        start=1
    ):

        item = {
            "id": index,

            "image_url": get_image_url(
                card.select_one(
                    ".slide-card img.slide-card_img, "
                    "img.slide-card_img"
                ),
                source_url
            ),

            "title": clean_text(
                card.select_one(
                    ".slide-card_content_title"
                )
            ),

            "location": clean_text(
                card.select_one(
                    ".slide-card_content_add, "
                    ".slide-card-add-container p, "
                    ".slide-card_content_location"
                )
            ),

            "description": clean_text(
                card.select_one(
                    ".slide-card_content_description"
                )
            )
        }

        if any(
            value
            for key, value in item.items()
            if key != "id"
        ):
            experiences_data[
                "items"
            ].append(item)

    experiences_data["items"] = (
        remove_duplicates(
            experiences_data["items"],
            fields=[
                "title",
                "image_url"
            ]
        )
    )

    # Đánh lại ID sau khi loại trùng
    for index, item in enumerate(
        experiences_data["items"],
        start=1
    ):
        item["id"] = index

else:
    print(
        "Không tìm thấy phần "
        "'11 Trendy Experiences at "
        "Vu Yen Royal Island'."
    )


# ============================================================
# 6. THÔNG TIN TRANG
# ============================================================

page_information = {
    "page_title": clean_text(
        soup.title
    ),

    "source_file": str(
        HTML_FILE
    ),

    "source_url": source_url,

    "language": (
        soup.html.get("lang")
        if soup.html
        else None
    ),

    "meta_description": get_meta_content(
        soup,
        'meta[name="description"]'
    ),

    "og_title": get_meta_content(
        soup,
        'meta[property="og:title"]'
    ),

    "og_description": get_meta_content(
        soup,
        'meta[property="og:description"]'
    ),

    "og_image": absolute_url(
        get_meta_content(
            soup,
            'meta[property="og:image"]'
        ),
        source_url
    )
}


# ============================================================
# 7. TỔNG HỢP JSON
# ============================================================

hai_phong_data = {
    "dataset_name": (
        "Vu Yen Royal Island "
        "Destination Dataset"
    ),

    "destination": {
        "name": "Vu Yen Royal Island",
        "city": "Hai Phong",
        "country": "Vietnam"
    },

    "page_information": (
        page_information
    ),

    "sections": {
        "welcome_to_vu_yen_royal_island": {
            "title": welcome_data["title"],
            "items": welcome_data["items"]
        },

        "trendy_experiences": {
            "title": experiences_data["title"],
            "items": experiences_data["items"]
        }
    },

    "statistics": {
        "welcome_item_count": len(
            welcome_data["items"]
        ),

        "trendy_experience_count": len(
            experiences_data["items"]
        )
    }
}


# ============================================================
# 8. LƯU JSON
# ============================================================

OUTPUT_JSON.parent.mkdir(
    parents=True,
    exist_ok=True
)

with OUTPUT_JSON.open(
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        hai_phong_data,
        json_file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 9. IN KẾT QUẢ KIỂM TRA
# ============================================================

print("\n" + "=" * 75)
print("HOÀN THÀNH")
print("=" * 75)

print(
    "Welcome items:",
    len(welcome_data["items"])
)

print(
    "Trendy experiences:",
    len(experiences_data["items"])
)

print("\nFile JSON:")
print(OUTPUT_JSON)


print("\n" + "=" * 75)
print("WELCOME TO VU YEN ROYAL ISLAND")
print("=" * 75)

for item in welcome_data["items"]:

    print(
        f"\n--- Welcome {item['id']} ---"
    )

    print(
        "Title:",
        item["title"]
    )

    print(
        "Description:",
        item["description"]
    )

    print(
        "Image:",
        item["image_url"]
    )


print("\n" + "=" * 75)
print("11 TRENDY EXPERIENCES")
print("=" * 75)

for item in experiences_data["items"]:

    print(
        f"\n--- Experience {item['id']} ---"
    )

    print(
        "Title:",
        item["title"]
    )

    print(
        "Location:",
        item["location"]
    )

    print(
        "Description:",
        item["description"]
    )

    print(
        "Image:",
        item["image_url"]
    )

ĐÃ ĐỌC HTML VU YEN ROYAL ISLAND
File: D:\vinuni\T013\data_crawl\hai_phong.html
Tiêu đề: Vu Yen Royal Island | Golf, Safari & Entertainment in Hai Phong
URL: https://vinwonders.com/en/vu-yen-royal-island/
Độ dài HTML: 205167

HOÀN THÀNH
Welcome items: 5
Trendy experiences: 11

File JSON:
D:\vinuni\T013\data_crawl\hai_phong_data.json

WELCOME TO VU YEN ROYAL ISLAND

--- Welcome 1 ---
Title: Step into enchantopia
Description: Embark on a magical journey with countless games and thousands of animals at VinWonders Vu Yen
Image: https://static.vinwonders.com/production/2025/08/Untitled-design-2.png

--- Welcome 2 ---
Title: Mastering the sport of the privileged
Description: Perfecting your swing in expansive natural surroundings, with every hole offering a new challenge.
Image: https://static.vinwonders.com/production/2026/06/4h_1781597373.png

--- Welcome 3 ---
Title: Experience aristocratic living
Description: Savor the leisurely art of horseback riding, enjoying elite pursuits at Vinpearl

In [14]:
# ============================================================
# CRAWL HTML GRAND PARK BẰNG SELENIUM EDGE
# Chạy toàn bộ trong 1 cell Jupyter Notebook
# ============================================================

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# 1. CẤU HÌNH
# ============================================================

URL = "https://vinwonders.com/en/grand-park/"

OUTPUT_FILE = Path(
    r"D:\vinuni\T013\data_crawl\grand_park.html"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36 "
        "Edg/150.0.0.0"
    )
}

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. KHỞI TẠO EDGE WEBDRIVER
# ============================================================

options = Options()

options.add_argument(
    "--start-maximized"
)

options.add_argument(
    f"user-agent={HEADERS['User-Agent']}"
)

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)

options.add_experimental_option(
    "useAutomationExtension",
    False
)

driver = webdriver.Edge(
    options=options
)

driver.set_page_load_timeout(
    180
)


# ============================================================
# 3. HÀM KIỂM TRA TRANG BLOCK
# ============================================================

def is_block_page(driver):
    title = (
        driver.title or ""
    ).lower()

    page_html = (
        driver.page_source or ""
    ).lower()

    signals = [
        "just a moment",
        "performing security verification",
        "verify you are human",
        "attention required",
        "checking your browser"
    ]

    return any(
        signal in title
        or signal in page_html
        for signal in signals
    )


# ============================================================
# 4. CRAWL VÀ LƯU HTML
# ============================================================

try:
    print("=" * 70)
    print("ĐANG CRAWL GRAND PARK")
    print("URL:", URL)
    print("=" * 70)

    driver.get(URL)

    WebDriverWait(
        driver,
        120
    ).until(
        lambda current_driver: (
            current_driver.execute_script(
                "return document.readyState"
            ) == "complete"

            and not is_block_page(
                current_driver
            )

            and len(
                current_driver.page_source
            ) > 5000
        )
    )

    html = driver.page_source

    if is_block_page(driver):
        raise ValueError(
            "Trang vẫn đang ở màn hình "
            "security verification."
        )

    if len(html) < 5000:
        raise ValueError(
            "HTML quá ngắn hoặc trang "
            "chưa tải hoàn chỉnh."
        )

    OUTPUT_FILE.write_text(
        html,
        encoding="utf-8"
    )

    print("\nCrawl thành công.")
    print("Tiêu đề:", driver.title)
    print("URL cuối:", driver.current_url)
    print("Độ dài HTML:", len(html))
    print("Đã lưu tại:", OUTPUT_FILE)

except (
    TimeoutException,
    WebDriverException,
    ValueError,
    Exception
) as error:

    print("\nCrawl thất bại.")
    print("Lỗi:", str(error))

finally:
    driver.quit()
    print("\nĐã đóng Edge WebDriver.")

ĐANG CRAWL GRAND PARK
URL: https://vinwonders.com/en/grand-park/

Crawl thành công.
Tiêu đề: Grand Park Ho Chi Minh City | Official Website
URL cuối: https://vinwonders.com/en/grand-park/
Độ dài HTML: 295341
Đã lưu tại: D:\vinuni\T013\data_crawl\grand_park.html

Đã đóng Edge WebDriver.


In [15]:
# ============================================================
# CRAWL HTML CÔNG VIÊN NƯỚC HÀ TĨNH BẰNG SELENIUM EDGE
# Chạy toàn bộ trong 1 cell Jupyter Notebook
# ============================================================

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# 1. CẤU HÌNH
# ============================================================

URL = "https://vinwonders.com/vi/cong-vien-nuoc-ha-tinh/"

OUTPUT_FILE = Path(
    r"D:\vinuni\T013\data_crawl\ha_tinh.html"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36 "
        "Edg/150.0.0.0"
    )
}

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. KHỞI TẠO EDGE WEBDRIVER
# ============================================================

options = Options()

options.add_argument(
    "--start-maximized"
)

options.add_argument(
    f"user-agent={HEADERS['User-Agent']}"
)

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)

options.add_experimental_option(
    "useAutomationExtension",
    False
)

driver = webdriver.Edge(
    options=options
)

driver.set_page_load_timeout(
    180
)


# ============================================================
# 3. KIỂM TRA TRANG BỊ BLOCK
# ============================================================

def is_block_page(driver):
    title = (
        driver.title or ""
    ).lower()

    page_html = (
        driver.page_source or ""
    ).lower()

    block_signals = [
        "just a moment",
        "performing security verification",
        "verify you are human",
        "attention required",
        "checking your browser"
    ]

    return any(
        signal in title
        or signal in page_html
        for signal in block_signals
    )


# ============================================================
# 4. CRAWL VÀ LƯU HTML
# ============================================================

try:
    print("=" * 70)
    print("ĐANG CRAWL CÔNG VIÊN NƯỚC HÀ TĨNH")
    print("URL:", URL)
    print("=" * 70)

    driver.get(URL)

    WebDriverWait(
        driver,
        120
    ).until(
        lambda current_driver: (
            current_driver.execute_script(
                "return document.readyState"
            ) == "complete"

            and not is_block_page(
                current_driver
            )

            and len(
                current_driver.page_source
            ) > 5000
        )
    )

    html = driver.page_source

    if is_block_page(driver):
        raise ValueError(
            "Trang vẫn đang ở màn hình "
            "security verification."
        )

    if len(html) < 5000:
        raise ValueError(
            "HTML quá ngắn hoặc trang "
            "chưa tải hoàn chỉnh."
        )

    OUTPUT_FILE.write_text(
        html,
        encoding="utf-8"
    )

    print("\nCrawl thành công.")
    print("Tiêu đề:", driver.title)
    print("URL cuối:", driver.current_url)
    print("Độ dài HTML:", len(html))
    print("Đã lưu tại:", OUTPUT_FILE)

except (
    TimeoutException,
    WebDriverException,
    ValueError,
    Exception
) as error:

    print("\nCrawl thất bại.")
    print("Lỗi:", str(error))

finally:
    driver.quit()
    print("\nĐã đóng Edge WebDriver.")

ĐANG CRAWL CÔNG VIÊN NƯỚC HÀ TĨNH
URL: https://vinwonders.com/vi/cong-vien-nuoc-ha-tinh/

Crawl thành công.
Tiêu đề: Công viên nước Hà Tĩnh | Official Website
URL cuối: https://vinwonders.com/vi/cong-vien-nuoc-ha-tinh/
Độ dài HTML: 176383
Đã lưu tại: D:\vinuni\T013\data_crawl\ha_tinh.html

Đã đóng Edge WebDriver.
